# LegalQA (UIT DSC 2026 — Task 2) · Kaggle T4×2 · **v6_2**

Dẫn xuất từ `v6_1_result/legalqa_kaggle_v6_1.ipynb`. Một phiên, ba option, **mỗi option một
cổng split-half trên cùng 300 câu dev** (chọn trên nửa A, chỉ nhận khi cả hai nửa cùng dương).
Cấu hình nào không qua cổng thì bài nộp giữ nguyên hành vi v6_1.

| Option | Thay đổi | Ở đâu | Cách đo trong phiên |
|---|---|---|---|
| 1 · reranker | Fine-tune lại theo **đúng công thức Task 1** (lr 1e-5 từ gốc, tích luỹ 8 nhóm, warmup 10%, gradient checkpointing, clip 1,0). Giữ **cả** zero-shot lẫn bản fine-tune | Cell 2, 10 | Cổng 1: `zs` vs `ft2` (zs chọn văn bản, ft chọn Điều) vs `ft` |
| 2 · câu dẫn | 4 biến thể câu dẫn Điều; đo trước 0 GPU trên 3.427 Điều gold: tốt nhất T3 **+0,0040** | Cell 2, 11 | Cổng 2: trên ứng viên hạng 1 của nhánh thắng cổng 1 |
| 3 · encoder | **Không đổi sang model mới** (`e5-base-v2` là model tiếng Anh và không nằm trong danh sách đăng ký). Thay vào đó đo từng kênh encoder đang đóng góp gì | Cell 11, 12 | Cổng 3: đủ kênh vs bỏ A vs bỏ B, dùng lại xếp hạng + điểm CE đã chấm |

Thay đổi hạ tầng: encode corpus fp16 có cổng cosine so với fp32 (Cell 9) để lấy lại thời gian
cho fine-tune reranker. Hai file log mới: `v6_2_decisions.json` (số đo từng cổng) và
`v6_2_dev_arms.jsonl` (từng câu dev × từng nhánh: văn bản tầng 1, ứng viên, điểm CE, METEOR).

⚠️ Dev chỉ 300 câu (150/nửa): hiệu ứng dưới ~1 điểm có thể không phân định được — log in cảnh
báo `|Δ| < 2·SE` cạnh từng dòng. Cổng từ chối không có nghĩa option vô dụng, chỉ là chưa đủ
bằng chứng để đổi bài nộp.


---

# LegalQA (UIT DSC 2026 — Task 2) · Kaggle T4×2 · **v6_1**

Dẫn xuất từ `legalqa_kaggle_v6_multistage.ipynb`. Tài liệu kỹ thuật đầy đủ:
`legalqa/V6_1_TECHNICAL.md`.

v6 đặt cược vào **bề rộng** — grid search BM25, query expansion PRF, ba biến thể loss cho
reranker, combo-testing ba chiều. Cộng lại ~770 phút, vượt xa mọi phiên thật. v6_1 đi ngược
lại: **cắt mọi nhánh đã đo là âm, dồn toàn bộ phần tiết kiệm được vào đúng một chỗ còn dư
địa đã đo được.**

---

## 1. Ba con số quyết định toàn bộ thiết kế

Tất cả lấy từ lượt Kaggle thật `2026-09-09` (`v6_log.txt`, `experiment_log.txt`), không phải
ước lượng.

| | |
|---|---|
| **+7,7 điểm** đang bỏ trống ở tầng xếp hạng | oracle chọn tốt nhất trong 5 ứng viên = 0,6401 · thực tế = 0,5630 (`result.md` §7). Reranker chọn đúng 61,3%, khoảng cách điểm hạng 1 vs hạng 2 trung vị **0,0023** — gần như tung đồng xu. |
| **138 phút** tiêu cho một nhánh thua | fine-tune reranker 110,4 phút + nhánh dev-eval thứ ba ~28 phút. Kết quả: METEOR 0,5382 so với zero-shot 0,5709 → dev-eval tự chọn zero-shot, checkpoint bị vứt. |
| **4,9 giây/câu** khi chạy 2 tầng trên 2 GPU | ⇒ eval 7.000 câu = **572 phút**, nhiều hơn cả fine-tune + encode corpus cộng lại (307 phút). Con số 7.000 không nằm trong bất kỳ phiên nào. |

---

## 2. Hai lỗi của v6 đã sửa

**(a) Chốt chặn false-negative chưa bao giờ chạy.** `mine_family_negatives()` gọi
`load_reranker_on(device, "zero-shot")` — truyền chuỗi mô tả vào tham số đợi tên model.
HuggingFace không có repo tên `zero-shot` → trả `None` → in *"bỏ chốt chặn, RỦI RO CAO"* →
toàn bộ negative vào train không qua lọc. Đúng cái chốt mà `result.md` §8.3 gọi là **bắt
buộc**: không lọc tức là dạy model dìm chính đáp án đúng xuống. Hệ quả dây chuyền: reranker-ft
OOM-backoff 8→4→2→**1** rồi chạy 7.494 step ở batch 1.

**(b) `NameError` ở cell cuối cùng.** Cell sổ thí nghiệm đọc `USE_TASK1_LABELS` và
`n_task1_added`, nhưng v6 đã xoá sạch code Task 1 khỏi Cell 7 mà quên cell này — crash sau
khi đã tiêu gần hết phiên GPU.

---

## 3. Cắt gì, vì sao

| Bỏ | Bằng chứng | Lấy lại |
|---|---|---|
| Fine-tune reranker (+ 3 biến thể loss) | thua zero-shot 3,3 điểm METEOR trên cùng một lượt | **~138 phút** |
| Nhánh dev-eval "không rerank" | Recall@1 0,154 vs 0,580 — không phải chuyện gần ngưỡng nhiễu | ~28 phút |
| Quét LSE + cổng split-half | đo âm **hai lần độc lập**: −3,2 đến −3,9 điểm (`result.md` §8.1) và cổng tự loại ở lượt Kaggle | vài phút + một trục rủi ro |
| BM25 grid search, query expansion PRF | chưa từng đo, thêm trục chưa kiểm chứng vào lượt vốn đã chật | ~40 phút |
| Lượt build `rows_clean` thứ hai | chỉ phục vụ fine-tune reranker, giờ đã tắt | 3,4 phút |

## 4. Thêm gì

**HARVEST — một lượt GPU, ba mục đích.** Chạy `retrieve_two_tier()` trên câu train rồi cache
lại top-5 ứng viên + điểm CE + vector đặc trưng + **METEOR của từng ứng viên**. Từ đúng cache
đó, ba việc sau không tốn thêm giây GPU nào: chọn `TOP_N_ANSWER`, train bộ chọn LTR, và dựng
file phân tích lỗi.

**Bộ chọn Điều học được (LightGBM LambdaRank)** — hướng ưu tiên #1 của cả `result.md` §7 lẫn
`RESEARCH_PLAN_BREAKTHROUGH.md` §3.1. 13 đặc trưng, không cái nào nhìn thấy gold. Qua **cổng
split-half** (chọn nửa A, đọc điểm nửa B, chỉ nhận khi cả hai cùng dương) — quy trình đã ba
lần bắt được ảo giác "đỉnh trên toàn dev" trong repo này.

**Ba tập dữ liệu rời nhau**, không tập nào chồng lên tập nào:

```
fine-tune encoder :  câu CÓ citation, trừ dev_ids
train LTR         :  câu KHÔNG citation, trừ dev_ids   <- chưa bao giờ vào fine-tune
gate LTR          :  dev_ids                            <- đã loại khỏi fine-tune từ Cell 7
```

Nhờ vậy METEOR đo trên harvest **không bị thổi phồng bởi contamination**. Nó vẫn cao hơn
public, nhưng vì template `render_answer` khớp văn phong `train.json` (`result.md` §5) — đó
là lệch phân phối, không phải rò rỉ, và không sửa được bằng cách chọn mẫu khác.

---

## 5. Thứ tự chạy — bài nộp được bảo vệ trước

Trần 8 giờ là **tự đặt, không phải giới hạn của Kaggle**: lượt trước chạy 586,6 phút (9,8 h)
và hoàn thành bình thường; trần thật của notebook GPU là 12 h. v6_1 nới ngân sách lên 11 h
nhưng tách riêng một mốc cứng cho bài nộp, và **đảo thứ tự** để `submission.zip` lên đĩa
trước khi làm phần chẩn đoán.

| # | Bước | Ước tính | Ghi chú |
|---|---|---:|---|
| 1 | chunk + BM25 2 tầng | 3 | |
| 2 | sinh nhãn + build rows | 5 | bỏ lượt `rows_clean` thứ hai |
| 3 | fine-tune 2 dense encoder | 104 | song song thật 2 GPU |
| 4 | encode corpus | 203 | phần tốn nhất, không cắt được |
| 5 | nạp reranker zero-shot | 2 | |
| 6 | **harvest pha A** + dev-eval | ~74 | tự cắt ngắn nếu chạm ngân sách Bước 7 |
| 7 | train LTR + cổng split-half | 3 | CPU |
| 8 | **sinh public + đóng gói** | 81 | ✅ **bài nộp an toàn từ đây** |
| 9 | harvest pha C + phân tích lỗi | phần dư | mất cũng không sao |

Mốc 8 rơi vào khoảng phút **475**. Nếu Kaggle ngắt ở giờ thứ 9, bài nộp vẫn nguyên.

---

## 6. File sinh ra

| File | Dùng để |
|---|---|
| `submission.zip` | nộp |
| `eval_harvest_summary.json` | **quyết định hướng đi** — phân bố lỗi, dư địa oracle, tỉ lệ chọn đúng |
| `eval_harvest_full.json` | soi từng ca lỗi cụ thể, có đủ 5 ứng viên kèm `meteor_if_chosen` |
| `ltr_report.json` | bộ chọn được nhận hay bị cổng loại, kèm delta từng nửa |
| `experiment_log.jsonl` | sổ thí nghiệm, có cả `resource_log` (VRAM/RAM đỉnh mỗi checkpoint) |

⚠️ **Điểm trong các file này KHÔNG dùng để dò tham số.** Chúng trả lời *"sai ở đâu"*, không
phải *"cấu hình nào tốt hơn"*. Dùng để chọn tham số là quay lại đúng cái bẫy `result.md` §14
đã ghi ba lần: dev tăng, public giảm.


In [ ]:
# Cell 1: Cài đặt thư viện
!pip install -q -U sentence-transformers datasets "accelerate>=1.1.0" nltk rouge_score sentencepiece peft
# bitsandbytes (optimizer 8-bit) — cài riêng, KHÔNG chặn nếu lỗi (hay gặp trên Windows/GPU cũ):
!pip install -q -U bitsandbytes || echo "bitsandbytes cai khong duoc - se tu dung AdamW thuong, khong crash"
# lightgbm (LTR — STAGE 2B): thường đã có sẵn trên image Kaggle chuẩn; cài phòng hờ, KHÔNG chặn nếu lỗi (LTR sẽ tự bỏ qua, xem Cell mới "STAGE 2A+2B").
!pip install -q -U lightgbm || echo "lightgbm cai khong duoc - STAGE 2B (LTR) se tu bo qua"


In [ ]:
# Cell 2: Đường dẫn và tham số
import os

# BẢN SỬA (log lỗi thật: chạy trên máy cá nhân RTX 2050 4GB nhưng CONTEXT_DIR trước đây cứng
# đường dẫn Kaggle /kaggle/input/... -> FileNotFoundError): tự nhận diện MÔI TRƯỜNG thay vì
# cứng 1 kiểu đường dẫn — /kaggle/input CHỈ tồn tại thật trên Kaggle, nên dùng chính nó làm
# điều kiện phát hiện. Nhờ vậy CÙNG MỘT nguồn code chạy đúng trên cả 2 nơi (notebook Kaggle
# tự lấy đường dẫn Kaggle, .py trên máy cá nhân tự lấy đường dẫn CẠNH SCRIPT — đúng quy ước
# HERE-relative của legalqa_local.py, dữ liệu đặt cùng thư mục file .py).
IS_KAGGLE = os.path.isdir("/kaggle/input")

if IS_KAGGLE:
    DATA_DIR = "/kaggle/input/datasets/anhnguyen7508/uit-data-science-dataset/"
    CONTEXT_DIR = os.path.join(DATA_DIR, "selected-contexts/selected-contexts/")
    TRAIN_PATH = os.path.join(DATA_DIR, "train.json")
    WARMUP_PATH = os.path.join(DATA_DIR, "warmup.json")
    PUBLIC_PATH = os.path.join(DATA_DIR, "public-official.json")
    # /kaggle/input CHỈ ĐỌC. Mọi thứ ghi ra phải nằm ở /kaggle/working (được lưu khi
    # "Save Version" — đây là nơi submission.zip phải nằm) hoặc /kaggle/temp (KHÔNG được
    # lưu, mất khi session kết thúc — dùng cho cache/model tải về, đỡ tốn quota output).
    OUT_DIR = "/kaggle/working"
    CACHE_DIR = "/kaggle/temp/legalqa_cache"
else:
    # Máy cá nhân (hoặc bất kỳ máy nào không phải Kaggle): đặt file .py CẠNH train.json,
    # public-official.json, selected-contexts/ — đúng quy ước của legalqa_local.py, không có
    # trần thời gian phiên nào để lo (khác Kaggle) nên OUT_DIR/CACHE_DIR cũng nằm ngay cạnh
    # script, dễ tìm dễ dọn.
    HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    DATA_DIR = HERE
    CONTEXT_DIR = os.path.join(HERE, "selected-contexts")
    TRAIN_PATH = os.path.join(HERE, "train.json")
    WARMUP_PATH = os.path.join(HERE, "warmup.json")
    PUBLIC_PATH = os.path.join(HERE, "public-official.json")
    OUT_DIR = HERE
    CACHE_DIR = os.path.join(HERE, "cache")

HF_CACHE_DIR = os.path.join(CACHE_DIR, "hf")
NLTK_CACHE_DIR = os.path.join(CACHE_DIR, "nltk_data")
TRAINER_TMP_DIR = os.path.join(CACHE_DIR, "trainer_tmp")
for _d in (OUT_DIR, HF_CACHE_DIR, NLTK_CACHE_DIR, TRAINER_TMP_DIR):
    os.makedirs(_d, exist_ok=True)
os.environ.setdefault("HF_HOME", HF_CACHE_DIR)
os.environ.setdefault("HF_HUB_CACHE", os.path.join(HF_CACHE_DIR, "hub"))
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")

# Tham số
CHUNK_SIZE = 512      # không sử dụng — chunk theo Điều (xem Bước 1), giữ lại đúng như đề bài
TOP_K_RETRIEVE = 100  # số ứng viên lấy ra sau RRF fusion (BM25 + dense)
TOP_K_RERANK = 5      # trần trên cho số Điều đưa vào 1 câu trả lời (top_n tĩnh VÀ trần adaptive-k)

# BẢN SỬA v4: KIẾN TRÚC HAI TẦNG (result.md §3) — v3 trở về trước cắt theo Điều NGAY Ở
# TẦNG RETRIEVAL, đúng cách Task 1 đã đo là làm document recall TỆ ĐI 1,19 điểm (p=0,024).
# v4 tách: tầng 1 truy xuất trên corpus 450 TỪ (bm25_t1 + dense_channels mã hoá
# all_chunks_t1) để CHỌN VĂN BẢN, tầng 2 mới cắt Điều — CHỈ trong DOC_K văn bản đã chọn —
# rồi rerank lại để chọn Điều cuối cùng. Xem chunk_passage_words()/retrieve_two_tier() ở
# Cell 5/11.
DOC_K = 5                 # số văn bản mở ra ở tầng 1 để cắt Điều tầng 2 (mirror run_qa.py)
MAX_DIEU_CANDIDATES = 100 # trần tổng số Điều đưa vào rerank tầng 2 (giữ THẤP HƠN
                          # run_qa.py's MAX_CANDS=150: v4 giờ rerank 2 LƯỢT/câu thay vì 1,
                          # nên tổng chi phí cross-encoder cao hơn v3 đáng kể — CHƯA đo
                          # được tác động lên thời lượng phiên Kaggle, xem cảnh báo ở Cell 0)
USE_FINETUNE = True   # bật mặc định — có đủ VRAM (16GB/thẻ Kaggle, hoặc OOM-backoff tự lùi
                       # trên máy yếu hơn) thì fine-tune, không cần đắn đo trước. Đặt False nếu
                       # muốn chạy thử nhanh hoặc đang tiết kiệm quota GPU trên Kaggle.

# Kiến trúc "mạnh nhất" — 2 dense encoder khác họ, fine-tune SONG SONG trên 2 GPU riêng (nếu
# có; tự lùi về tuần tự nếu chỉ 1 GPU — xem Bước 4), fusion RRF 3 kênh với BM25, thay cho 1
# bi-encoder 135M tự train trước đây. Không có nhãn document-level của Task 1 nên train HOÀN
# TOÀN từ nhãn citation của Task 2 — chỉ đổi SỐ LƯỢNG và ĐỘ MẠNH encoder, không cần dữ liệu ngoài.
# =============================================================================
# CHUYỂN GIAO TỪ TASK 1 (LegalIR) — hai thứ, mỗi thứ một cờ riêng để đo tách bạch
# =============================================================================
# Nguồn: result.md §31. Bài nộp LegalIR tốt nhất là cấu hình #18 + gộp LSE T=2:
#
#     #18 với max      public recall@5 = 0,9471
#     #18 với LSE T=2  public recall@5 = 0,9519      (+0,48)
#
# (1) HÀM GỘP Ở MỨC VĂN BẢN. Task 1 gộp điểm cross-encoder của nhiều chunk về một
#     document rồi xếp hạng document. `max` cho document thắng nhờ MỘT chunk tốt nhất;
#     `logsumexp` thưởng thêm cho document có NHIỀU chunk cùng tốt. Ở đây chunk là Điều
#     thay vì đoạn 450 từ, nhưng phép toán y hệt: gộp điểm các Điều cùng một văn bản,
#     chọn văn bản, rồi lấy Điều tốt nhất TRONG văn bản đó.
#
#     CẢNH BÁO VỀ T: nhiệt độ phụ thuộc THANG ĐIỂM của cross-encoder. Task 1 đo T=2 trên
#     logit thô; reranker ở notebook này có thể trả về thang khác, nên T=2 KHÔNG bê thẳng
#     được. Bước 6 quét T và chọn bằng SPLIT-HALF (chỉnh trên nửa này, đo nửa kia) — đúng
#     quy trình đã bắt được ảo giác "đỉnh trên toàn dev" ba lần trong repo này.
#
# (2) CẶP ENCODER. Task 1 thay bge-m3+e5-large bằng vnembv2+harrier, cả hai fine-tune:
#     0,9439 -> 0,9471. Cả hai đều KHÔNG dùng tiền tố query:/passage: (job Task 1 chạy
#     harrier với cờ --e5_no_prefix). Đặt False để quay về cặp cũ đã cho 0,5528.
# Negative "cùng họ" cho reranker (chuyển giao PHƯƠNG PHÁP Task 1 — xem Cell 10).
# Task 1 dùng 7 negative/nhóm + listwise softmax; giữ nguyên con số đó.
N_NEG_RERANK = 7
RERANK_FALSE_NEG_GATE = True   # bỏ ứng viên bị reranker zero-shot chấm CAO HƠN gold —
                                # gần như chắc chắn là positive chưa gán nhãn, không phải
                                # negative. Tắt cái này là dạy model dìm đáp án đúng.

# BẢN SỬA v6_1 — LSE ĐÃ ĐO ÂM HAI LẦN ĐỘC LẬP, ĐÓNG HƯỚNG:
#   (1) result.md §8.1, 501 câu dev server: max 0,5630 · lse T=0,5 0,5306 · T=1 0,5249 ·
#       T=2 0,5240 -> âm 3,2 đến 3,9 điểm.
#   (2) Lượt Kaggle thật 2026-09-09 (experiment_log.txt): cổng split-half tự BỎ lse,
#       "agg_mode_chosen": "max" — nửa B âm ở cả 3 giá trị T.
# Lý do cơ chế (không phải nhiễu): Task 1 tối ưu recall@5, văn bản đúng chỉ cần lọt 1 trong
# 5 slot nên thưởng "bằng chứng trải rộng" là đúng hướng. Task 2 trích ĐÚNG MỘT Điều, nên
# đẩy văn bản có vài Điều tầm tầm lên trên văn bản có một Điều xuất sắc là đánh đổi NGƯỢC
# dấu. Ép cứng "max", xoá luôn Bước 6b (quét T) để lấy lại thời gian cho harvest.
AGG_MODE = "max"
AGG_T_GRID = []             # rỗng = không quét T nữa (xem khối lý do ngay trên)
USE_NEW_ENCODERS = True     # False -> giữ nguyên bge-m3 + e5-large của bản 0.5528


BASE_DENSE_MODEL_A = ("AITeamVN/Vietnamese_Embedding_v2" if USE_NEW_ENCODERS
                       else "BAAI/bge-m3")          # họ bge-m3: CLS, KHÔNG tiền tố
BASE_DENSE_MODEL_B = ("mainguyen9/vietlegal-harrier-0.6b" if USE_NEW_ENCODERS
                       else "intfloat/multilingual-e5-large")
# Tiền tố query của TỪNG kênh. Sai tiền tố = embedding lệch hệ toạ độ, recall tụt mà
# KHÔNG lỗi nào bắn ra — bẫy kinh điển, đã ghi ở nhiều chỗ trong repo.
# e5-large là model DUY NHẤT trong bốn cái trên cần tiền tố.
QUERY_PREFIX_A = ""
QUERY_PREFIX_B = "" if USE_NEW_ENCODERS else "query: "
PASSAGE_PREFIX_B = "" if USE_NEW_ENCODERS else "passage: "
                                                            # — bẫy kinh điển: quên tiền tố thì
                                                            # recall tụt mà KHÔNG có lỗi nào bắn ra
                                                            # (embedding vẫn ra số, chỉ lệch hệ toạ độ).
# v6_2 — CHỈ model đã có trong danh sách đăng ký (rule.md §2.3; hạn gửi đề xuất model mới là
# 13/09/2026). Cổng ở cuối cell này chặn cứng nếu ai đổi sang model ngoài danh sách.
DENSE_MAX_SEQ_LEN = 256
CHECKPOINT_DIR = os.path.join(OUT_DIR, "checkpoints")   # 1 thư mục duy nhất — Kaggle: giữ lại
                                                          # khi Save Version, tự tải về; máy cá
                                                          # nhân: nằm cạnh script như mọi output khác.
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
MIN_TRAIN_PAIRS = 50
MAX_TRAIN_EXAMPLES = 9000   # trần số cặp train dùng để fine-tune encoder (Bước 4) —
                             # không sợ vượt giờ: Bước 4 time-box theo FINETUNE_TIME_BUDGET_SEC
                             # (đo tốc độ vài step đầu rồi tự tính max_steps).
N_NEG_PER_ROW = 2

# BẢN SỬA (theo yêu cầu — chạy trên máy cá nhân, không cần đắn đo ngân sách còn lại): batch
# khởi điểm GIỮ NGUYÊN mức đã tối ưu cho GPU nhiều VRAM (Kaggle T4 16GB); trên máy VRAM nhỏ
# hơn (vd RTX 2050 4GB), OOM-backoff đã có sẵn ở mọi bước (Bước 4/5/6/7/5b) tự lùi batch khi
# OOM thật xảy ra — KHÔNG cần hạ tay trước, đúng triết lý "dùng tối đa tài nguyên, chỉ lùi khi
# thật sự hết" đã áp dụng xuyên suốt từ legalqa_local.py.
TRAIN_BATCH_SIZE = 64        # batch HIỆU DỤNG (số in-batch negative)
TRAIN_MINI_BATCH_SIZE = 32
# BẢN SỬA (log ver9): encoder thứ hai kết thúc ở mini_batch_final=4 sau 176 PHÚT cho chỉ
# 334 step — tức OOM-backoff phải chia đôi ba lần (32->16->8->4), mỗi lần OOM là một lượt
# forward vứt đi. Với USE_NEW_ENCODERS=True kênh B là harrier-0.6b (596M, lớn hơn
# e5-large), nên xuất phát thấp ngay thay vì dò xuống. So sánh: kênh A xong 440 step
# trong 82 phút ở mini_batch 16.
TRAIN_MINI_BATCH_SIZE_B = 8 if USE_NEW_ENCODERS else 32   # batch THẬT mỗi forward — OOM-backoff tự giảm nếu máy yếu hơn Kaggle.
ENCODE_BATCH_SIZE = 256      # batch encode corpus mỗi tiến trình GPU.
# v6_2 — encode corpus fp16 có cổng (Cell 9): chỉ dùng fp16 khi không NaN và cosine nhỏ nhất
# so với fp32 trên mẫu >= FP16_MIN_COS. Lượt v6_1 encode fp32 hết 185 phút.
ENCODE_FP16 = True
FP16_GUARD_N = 256
FP16_MIN_COS = 0.995
RERANK_SUBBATCH = 64         # batch xử lý mỗi lần forward reranker (568M) — xử lý theo lô thay
                              # vì lô vừa phải luôn nhanh hơn 1 lô khổng lồ (padding ít hơn,
                              # không nghẽn băng thông bộ nhớ) — xem Cell 11.

# BẢN SỬA: TIME_BUDGET chỉ thật sự cần trên Kaggle (trần phiên GPU ~9-12h). Máy cá nhân KHÔNG
# có giới hạn phiên nào — đặt trần RẤT RỘNG (không phải vô hạn, để tránh treo vĩnh viễn nếu có
# bug logic nào đó) thay vì ép chạy nhanh/cắt ngắn không cần thiết.
# BẢN SỬA v6_1 — TRẦN 8 GIỜ LÀ TỰ ĐẶT, KHÔNG PHẢI GIỚI HẠN CỦA KAGGLE.
# Bằng chứng: lượt 2026-09-09 (v6_log.txt) chạy 586,6 phút = 9,8 GIỜ và HOÀN THÀNH bình
# thường — checkpoint() in "còn lại ~-106,6 phút" suốt 80 phút cuối mà không có gì dừng nó.
# Trần thật của notebook GPU Kaggle là 12h/phiên. Nới TIME_BUDGET lên 11h để phần chẩn đoán
# có chỗ chạy, NHƯNG tách riêng một mốc cứng cho bài nộp:
#
#   SUBMISSION_DEADLINE_SEC = 8h — mốc mà submission.zip PHẢI nằm trên đĩa.
#
# Mọi việc sau mốc đó (harvest mở rộng, error analysis) là phần ăn thêm: mất cũng không ảnh
# hưởng bài nộp. Đây là lý do Bước 7 (sinh public) được đẩy lên TRƯỚC harvest mở rộng trong
# v6_1 — xem Cell 0. Nếu Kaggle bị kill ở giờ thứ 9, ta vẫn có bài nộp hoàn chỉnh.
TIME_BUDGET_SEC = (11 * 3600) if IS_KAGGLE else (48 * 3600)
# v6_2: +1,5h cho fine-tune reranker và so nhánh trên dev (xem Cell 10/12).
SUBMISSION_DEADLINE_SEC = (int(9.5 * 3600)) if IS_KAGGLE else (24 * 3600)
# BẢN SỬA (log ver9): hai encoder ngốn 259/490 phút = 53% phiên, mà METEOR gần như không
# nhúc nhích. Trong khi nút thắt đo được nằm ở TẦNG CHỌN ĐIỀU: oracle chọn tốt nhất trong
# 5 ứng viên đã lấy ra cho 0,6401 so với 0,5630 thực tế — +7,7 điểm bỏ trống. Rót thời
# gian sang reranker, nơi vừa đổi sang công thức Task 1.
FINETUNE_TIME_BUDGET_SEC = (100 * 60) if IS_KAGGLE else (16 * 3600)
DEV_EVAL_SAMPLE_SIZE = 300

# BẢN SỬA (tái lập được kết quả + ablation warmup + sổ thí nghiệm — theo phân tích
# chênh lệch điểm .py-vs-Kaggle, xem PHAN_TICH_KY_THUAT.md): trước đây random.seed(42) chỉ
# đặt ngay trước lúc lấy mẫu dev-eval (Bước 6) — random.sample/random.choice ở Bước 4 (chọn
# tập con để fine-tune, chọn hard-negative dự phòng) chạy TRƯỚC đó với random state KHÔNG
# seed, nên 2 lần chạy cùng code vẫn fine-tune trên 2 tập con khác nhau. SEED được set_all_seeds()
# NGAY SAU khi import xong (Cell 4) — trước Bước 3/4 — để tái lập được. USE_WARMUP cho phép
# ablation có/không warmup.json ở CÙNG seed. EXPERIMENT_LOG_PATH nằm trong OUT_DIR — trên
# Kaggle được giữ lại khi "Save Version" (khác cache/ ở /kaggle/temp, mất khi session kết
# thúc); trên máy cá nhân nằm cạnh script như submission.zip.
# CÂU KẾT — xem docstring render_answer() ở Cell 11 để biết số đo đầy đủ (+4,8 điểm METEOR
# so với không có câu kết, đo trên 501 câu dev, xác nhận bằng split-half).
CONCL = "echo2"          # none | echo | echo2
# v6_2 — OPTION 2: biến thể CÂU DẪN (chỉ đơn vị Điều; câu kết giữ echo2). Đo trước 0 GPU trên
# 3.427 Điều gold của train.json, METEOR thật, Δ so với T0 (hai nửa cùng dấu, SE ~0,0001):
#   T1_theo_qd_tai +0,0007 · T2_title +0,0029 · T3_theo_qd_tai_title +0,0040
# Hiệu ứng THẬT nhưng NHỎ (+0,4 điểm). Cell 12 đo lại trên ứng viên pipeline thật chọn và chỉ
# đổi khỏi T0 khi cả hai nửa dev cùng dương.
ANSWER_TEMPLATES = ["T0_current", "T1_theo_qd_tai", "T2_title", "T3_theo_qd_tai_title"]
ANSWER_TEMPLATE = "T0_current"   # mốc; cổng 2 ở Cell 12 có thể ghi đè

# NHÃN HUẤN LUYỆN — CHỈ Task 2 (v6: đã xoá hoàn toàn code Task 1, xem Cell 0).
# Nhãn citation của Task 2 (build_train_pairs, Cell 7) phân giải được ~47,8% câu (3.349/
# 7.000) — phần còn lại KHÔNG có nhãn document-level thay thế trong notebook này.
SEED = 42
USE_WARMUP = True   # đặt False để ablation: chỉ dùng train.json, không gộp warmup.json
EXPERIMENT_LOG_PATH = os.path.join(OUT_DIR, "experiment_log.jsonl")

# BẢN SỬA (kết quả thật: dual-encoder 0.5215/0.4829 chỉ nhích rất ít so với single-encoder
# 0.5199/0.4806 dù retrieval mạnh hơn nhiều -> retrieval không còn là nút thắt chính, reranker
# ZERO-SHOT giờ nhiều khả năng là trần chặn điểm. Fine-tune reranker trên chính nhãn citation
# Task 2 -- tái dùng `rows` đã build cho Bước 4, KHÔNG cần dữ liệu thêm.
# =============================================================================
# BẢN SỬA v6_1 — TẮT FINE-TUNE RERANKER. Đây là thay đổi lấy lại NHIỀU THỜI GIAN NHẤT.
# =============================================================================
# Số đo thật, lượt Kaggle 2026-09-09 (v6_log.txt + experiment_log.txt), CÙNG MỘT lượt chạy
# nên so sánh là paired, không phải so giữa hai phiên khác nhau:
#
#     Recall@1 dev (n=143)   zero-shot 58,0%   fine-tuned 55,2%
#     METEOR  dev (n=300)    zero-shot 0,5709  fine-tuned 0,5382      -> âm 3,3 điểm
#     "reranker_source": "zeroshot"   <- dev-eval TỰ CHỌN zero-shot
#
# Chi phí của nhánh thua đó: 110,4 phút fine-tune + ~28 phút chạy thêm một nhánh dev-eval
# = 138 phút (23,5% phiên) cho một checkpoint bị vứt đi ngay sau khi đo.
#
# VÀ nó thua vì một BUG CỤ THỂ, không phải vì phương pháp sai — xem Cell 10:
# `mine_family_negatives()` gọi `load_reranker_on(device, "zero-shot")`, truyền CHUỖI MÔ TẢ
# "zero-shot" vào chỗ đợi TÊN MODEL. HuggingFace không có repo tên "zero-shot" -> trả None ->
# in "[mine] không tải được reranker để lọc -> bỏ chốt chặn, RỦI RO CAO" -> toàn bộ negative
# vào thẳng không qua lọc false-negative. Đúng cái chốt chặn mà result.md §8.3 gọi là BẮT
# BUỘC: không lọc tức là dạy model DÌM đáp án đúng xuống. Hệ quả dây chuyền: reranker-ft
# OOM-backoff tụt batch 8->4->2->1 rồi chạy 7.494 step ở batch 1.
#
# Vì sao v6_1 vẫn TẮT thay vì sửa bug rồi chạy lại: sửa một dòng thì dễ, nhưng để BIẾT bản
# sửa có thắng zero-shot hay không vẫn phải trả đủ 138 phút mỗi lượt, và đó là 138 phút lấy
# thẳng từ chỗ duy nhất còn dư địa lớn đã đo được (§7: +7,7 điểm ở tầng xếp hạng). Bug đã
# được sửa sẵn trong Cell 10 và cờ này để lại nguyên — bật lên là chạy được ngay khi nào có
# một phiên GPU rảnh để dành riêng cho việc đó.
#
# v6_2 — BẬT LẠI, với hai thay đổi so với lượt thua 3,3 điểm:
#   (1) công thức train chép đúng Task 1 (Cell 10: lr 1e-5 từ gốc, tích luỹ 8 nhóm, warmup,
#       gradient checkpointing) — bản cũ tụt về batch 1 và dùng lr của vòng train tiếp;
#   (2) KHÔNG thay thế zero-shot nữa: Cell 12 chấm cả hai trên cùng dev, qua cổng split-half.
USE_RERANKER_FINETUNE = True
RERANKER_BASE = "AITeamVN/Vietnamese_Reranker"
RERANKER_FT_TIME_BUDGET_SEC = (75 * 60) if IS_KAGGLE else (6 * 3600)
RERANKER_FT_ACCUM = 8                   # nhóm (câu hỏi) mỗi bước cập nhật — Task 1 --grad_accum 8
RERANKER_FT_EPOCHS = 2                  # Task 1 --epochs 2; bị cắt sớm hơn nếu hết ngân sách
RERANKER_FT_WARMUP = 0.1
RERANKER_FT_MAX_LEN = 512
# BẢN SỬA (log ver9): 1e-5 -> 3e-6. Đây là lr Task 1 đo được cho ĐÚNG công thức
# listwise + negative cùng họ mà Cell 10 vừa chuyển sang. Giữ 1e-5 với loss listwise là
# trộn hai công thức khác nhau.
# v6_2: 3e-6 là lr Task 1 dùng khi train TIẾP từ checkpoint vòng 2 (eval/job_family.sh);
# train từ model GỐC Task 1 dùng 1e-5 (mặc định eval/train_reranker.py).
RERANKER_FT_LR = 1e-5

# BẢN SỬA (log lỗi thật: torch.AcceleratorError OOM ngay ở lần optimizer.step() ĐẦU TIÊN,
# trước khi kịp chạy batch nào — full fine-tune AdamW cho model ~568M cần khoảng 6-7GB CHỈ
# RIÊNG optimizer state (2 buffer fp32/tham số), không phụ thuộc batch size. Trên GPU 4GB,
# KHÔNG batch nào nhỏ tới đâu cũng không đủ — đây là giới hạn vật lý, không phải cấu hình sai.
# LoRA (chỉ train 1 phần rất nhỏ tham số, đóng băng phần còn lại) không phải "cho nhanh hơn"
# mà là ĐIỀU KIỆN BẮT BUỘC để fine-tune được model cỡ này trên 4GB — đồng thời PHÙ HỢP HƠN
# về phương pháp với lượng dữ liệu nhỏ hiện có (3.500-9.000 câu là rất ít so với 568M tham
# số, full fine-tune có rủi ro overfit/quên kiến thức gốc thật sự). Trên Kaggle (nhiều VRAM),
# GIỮ NGUYÊN full fine-tune như cũ — không đổi kết quả 0.5526/0.4817 đã có kiểm chứng.
_total_vram_gb = 0.0
try:
    import torch as _torch_probe
    if _torch_probe.cuda.is_available():
        _total_vram_gb = _torch_probe.cuda.get_device_properties(0).total_memory / (1024 ** 3)
except Exception:
    pass
LOW_VRAM_MODE = (not IS_KAGGLE) and (_total_vram_gb > 0) and (_total_vram_gb < 10)
USE_LORA = LOW_VRAM_MODE        # LoRA cho CẢ dense encoder LẪN reranker khi VRAM thấp
USE_8BIT_OPTIM = LOW_VRAM_MODE  # optimizer AdamW 8-bit (bitsandbytes) khi VRAM thấp — cộng
                                 # dồn với LoRA, không thay thế; tự lùi về AdamW thường êm ái
                                 # nếu bitsandbytes không cài được (xem Cell 1).
LORA_R = 16          # rank — 16 là điểm cân bằng phổ biến, đủ biểu đạt cho fine-tune domain
LORA_ALPHA = 32      # thường đặt = 2 * LORA_R
LORA_DROPOUT = 0.05

print(f"Môi trường: {'Kaggle' if IS_KAGGLE else 'máy cá nhân (không phải Kaggle)'}")
print(f"DATA_DIR  = {DATA_DIR}")
print(f"OUT_DIR   = {OUT_DIR}")
print(f"CACHE_DIR = {CACHE_DIR}" + ("  (tạm, mất khi session kết thúc)" if IS_KAGGLE else ""))
print(f"LOW_VRAM_MODE={LOW_VRAM_MODE} (VRAM={_total_vram_gb:.1f}GB) -> USE_LORA={USE_LORA}, USE_8BIT_OPTIM={USE_8BIT_OPTIM}")

# =============================================================================
# v6_1 — HARVEST + BỘ CHỌN ĐIỀU HỌC ĐƯỢC (LTR). Đây là toàn bộ nội dung mới của v6_1.
# =============================================================================
# BỐI CẢNH (result.md §7, đo trên 501 câu dev):
#
#     reranker chọn hạng 1 (hiện tại)                    METEOR 0,5630
#     oracle: chọn tốt nhất trong ĐÚNG 5 ứng viên đã có  METEOR 0,6401
#     ------------------------------------------------------------------
#     dư địa nằm sẵn trong danh sách, chỉ bị xếp sai thứ tự       +7,7 điểm
#
# Reranker chọn đúng Điều tốt nhất chỉ 61,3% số lần, và khoảng cách điểm giữa hạng 1 và
# hạng 2 có TRUNG VỊ 0,0023 — ở rất nhiều câu nó gần như tung đồng xu. Bốn quy tắc rẻ tiền
# (dài nhất / ngắn nhất / gần độ dài trung vị) đều TỆ HƠN giữ nguyên hạng 1, và tương quan
# (độ dài, METEOR) chỉ −0,078 -> độ dài không phải tín hiệu. Cần một bộ chọn HỌC ĐƯỢC.
#
# HARVEST — một lượt GPU, ba mục đích. Chạy retrieve_two_tier() trên câu train và CACHE lại
# top-K ứng viên kèm điểm CE + đặc trưng + METEOR của từng ứng viên. Từ đúng cache đó:
#   (1) train LTR          (2) dev-eval chọn cấu hình     (3) error analysis cho đồng đội
# Trước đây (1) và (3) sẽ là hai lượt GPU riêng; gộp lại tiết kiệm nguyên một lượt.
#
# ⏱️ VÌ SAO KHÔNG PHẢI 7.000 CÂU. Đo từ lượt thật: dev-eval 300 câu / 2 GPU song song hết
# 1.479s, Bước 7 1.000 câu hết 4.862s -> ~4,9 giây/câu. 7.000 câu = 34.300s = **572 phút**,
# tức nhiều hơn cả phần fine-tune + encode corpus cộng lại. Con số đó không nằm trong bất kỳ
# phiên nào. HARVEST_TARGET_TOTAL đặt theo thời gian thật còn lại, không theo con số 7.000.
HARVEST_PHASE_A_N = 900       # PHA A (bắt buộc, TRƯỚC bài nộp): đủ để train LTR + gate.
                               # ~74 phút. Gồm cả DEV_EVAL_SAMPLE_SIZE câu dev.
HARVEST_TARGET_TOTAL = 3500   # PHA C (tuỳ chọn, SAU khi submission.zip đã nằm trên đĩa):
                               # harvest thêm tới mốc này HOẶC tới khi hết giờ, tuỳ cái nào
                               # đến trước. Chỉ phục vụ error analysis.
HARVEST_BATCH = 100           # kiểm tra ngân sách sau mỗi lô này -> dừng sạch, không cụt file
PUBLIC_RESERVE_SEC = 100 * 60 # thời gian GIỮ LẠI cho Bước 7 + đóng gói. Pha A tự cắt ngắn
                               # nếu chạm vào phần này. Đo thật: Bước 7 hết 81 phút.

USE_LTR = True                # bộ chọn Điều học được (LightGBM LambdaRank)
LTR_TOP_K_CANDIDATES = 5      # số ứng viên đầu bảng đưa vào LTR — khớp TOP_K_RERANK, và
                               # khớp đúng tập mà oracle +7,7 điểm được đo trên đó
LTR_MIN_GROUPS = 300          # dưới ngưỡng này thì không train LTR (quá ít nhóm để tin)
LTR_NUM_LEAVES = 15           # giữ NHỎ: đặc trưng chỉ ~12 chiều, vài nghìn nhóm — cây to là
LTR_N_ESTIMATORS = 200        # mời overfit. Gate split-half ở Cell 13 vẫn là chốt chặn cuối.
LTR_LEARNING_RATE = 0.05

# Error analysis — ngưỡng phân loại lỗi, xem classify_error() ở Cell 15.
ERR_OK_METEOR = 0.60          # >= ngưỡng này coi như câu đã tốt
ERR_RANKING_GAP = 0.05        # oracle - thực tế >= ngưỡng này -> lỗi XẾP HẠNG (ứng viên tốt
                               # đã nằm trong tay mà không chọn) — đây là loại lỗi LTR nhắm tới
ERR_RETRIEVAL_CEIL = 0.40     # oracle_best < ngưỡng này -> lỗi TRUY XUẤT (không ứng viên nào
                               # cứu được câu này, LTR bó tay)

print(f"v6_1: harvest pha A={HARVEST_PHASE_A_N} câu, mục tiêu tổng={HARVEST_TARGET_TOTAL}, "
      f"USE_LTR={USE_LTR}, USE_RERANKER_FINETUNE={USE_RERANKER_FINETUNE}, AGG_MODE={AGG_MODE}")
print(f"     ngân sách {TIME_BUDGET_SEC/3600:.0f}h · mốc cứng cho submission "
      f"{SUBMISSION_DEADLINE_SEC/3600:.0f}h · giữ lại cho Bước 7 {PUBLIC_RESERVE_SEC/60:.0f} phút")

# =============================================================================
# v6_2 — CỜ BA CỔNG (Cell 12) + CHẶN MODEL NGOÀI DANH SÁCH
# =============================================================================
RUN_TEMPLATE_GATE = True        # Option 2
RUN_CHANNEL_ABLATION = True     # Option 3: đo từng kênh encoder đang đóng góp gì, 0 encode thêm
REGISTERED_MODELS = {
    "AITeamVN/Vietnamese_Embedding_v2", "mainguyen9/vietlegal-harrier-0.6b",
    "BAAI/bge-m3", "intfloat/multilingual-e5-large",
    "AITeamVN/Vietnamese_Reranker", "Qwen/Qwen3-Reranker-0.6B",
}
for _m in (BASE_DENSE_MODEL_A, BASE_DENSE_MODEL_B, RERANKER_BASE):
    if _m not in REGISTERED_MODELS:
        raise SystemExit(f"⛔ {_m} không có trong danh sách model đã đăng ký (rule.md §2.3).")
print(f"v6_2: reranker-ft={USE_RERANKER_FINETUNE} (lr {RERANKER_FT_LR}, accum {RERANKER_FT_ACCUM}, "
      f"{RERANKER_FT_TIME_BUDGET_SEC/60:.0f} phút) · template gate={RUN_TEMPLATE_GATE} "
      f"· channel ablation={RUN_CHANNEL_ABLATION} · encode fp16={ENCODE_FP16}")


In [ ]:
# Cell 3: Kiểm tra GPU (mong đợi 2x Tesla T4) + tiện ích thời gian + theo dõi tài nguyên
import time
import torch

N_GPU = torch.cuda.device_count()
print(f"Số GPU thấy được: {N_GPU}")
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} — {p.name}, {p.total_memory/1024**3:.1f} GB")

if N_GPU == 0:
    DEVICES = ["cpu"]
    print("[CẢNH BÁO] Không thấy GPU nào — kiểm tra Settings > Accelerator = GPU T4 x2. "
          "Sẽ chạy CPU, RẤT chậm cho Bước 4/5/6/7.")
elif N_GPU == 1:
    DEVICES = ["cuda:0"]
    print("[CẢNH BÁO] Chỉ thấy 1 GPU — vẫn chạy được nhưng KHÔNG tận dụng song song 2 thẻ "
          "ở Bước 5/6/7. Kiểm tra Settings > Accelerator = GPU T4 x2 nếu muốn đủ 2 thẻ.")
else:
    DEVICES = [f"cuda:{i}" for i in range(N_GPU)]
    print(f"OK — sẽ dùng song song {DEVICES} ở Bước 5 (encode corpus) và Bước 6/7 (rerank).")

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

_START_TIME = time.time()

def elapsed() -> float:
    return time.time() - _START_TIME

def remaining() -> float:
    return TIME_BUDGET_SEC - elapsed()


# =============================================================================
# v6_1 — THEO DÕI TÀI NGUYÊN. Gắn vào checkpoint() nên KHÔNG tốn gì thêm.
# =============================================================================
# Chỉ chạy ~12 lần mỗi phiên (mỗi checkpoint một lần), đọc con số torch đã giữ sẵn trong bộ
# nhớ — không gọi nvidia-smi, không tạo tiến trình con, không thread nền. Chi phí thật sự
# bằng 0, khác hẳn phương án lấy mẫu định kỳ vốn phải giành GIL với luồng GPU.
#
# Vì sao đáng ghi: hai sự cố tốn nhiều giờ nhất của lượt trước đều là sự cố BỘ NHỚ mà log
# không hề ghi lại — encoder B tụt mini-batch 32->4 sau OOM-backoff (99 phút cho 149 step),
# và reranker-ft tụt batch 8->1. Khi đọc log sau đó, không có cách nào biết VRAM đã ở đâu
# lúc chuyện xảy ra. `max_memory_allocated` là con số đúng cho việc đó: nó nhớ ĐỈNH, không
# phải giá trị tức thời — nên vẫn bắt được cú vọt dù ta chỉ đọc mỗi checkpoint một lần.
_RESOURCE_LOG = []

def resource_snapshot() -> dict:
    snap = {"t_min": round(elapsed() / 60, 1), "gpu": []}
    for i in range(N_GPU):
        try:
            snap["gpu"].append({
                "dev": f"cuda:{i}",
                "alloc_gb": round(torch.cuda.memory_allocated(i) / 1024**3, 2),
                "peak_gb": round(torch.cuda.max_memory_allocated(i) / 1024**3, 2),
                "reserved_gb": round(torch.cuda.memory_reserved(i) / 1024**3, 2),
            })
        except Exception:
            pass
    try:
        import resource as _res
        # ru_maxrss: KB trên Linux. Đây là ĐỈNH RSS của tiến trình, không phải hiện tại —
        # đúng thứ cần để biết có suýt chạm trần RAM 30GB của Kaggle hay không.
        snap["host_peak_rss_gb"] = round(
            _res.getrusage(_res.RUSAGE_SELF).ru_maxrss / 1024**2, 2)
    except Exception:
        pass
    return snap


def checkpoint(label: str) -> None:
    snap = resource_snapshot()
    snap["label"] = label
    _RESOURCE_LOG.append(snap)
    gpu_txt = " · ".join(f"{g['dev']} {g['alloc_gb']:.1f}/{g['peak_gb']:.1f}GB"
                          for g in snap["gpu"])
    ram_txt = (f" · RAM đỉnh {snap['host_peak_rss_gb']:.1f}GB"
               if "host_peak_rss_gb" in snap else "")
    print(f"[{elapsed()/60:6.1f} phút] {label}  (còn ~{remaining()/60:.1f} phút)"
          + (f"\n           VRAM (đang dùng/đỉnh): {gpu_txt}{ram_txt}" if gpu_txt else ram_txt))


checkpoint("Bắt đầu")


In [ ]:
# Cell 4: Import chung + hằng số regex cho chunk theo Điều
import re
import json
import math
import random
import zipfile
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np

DIEU_RE = re.compile(r"^[ \t]*Điều\s+(\d+)[a-zđA-ZĐ]?[\.\s]", re.MULTILINE)
# BẢN SỬA v5 (Batch 1 A1+A2, xem EDA_ANALYSIS_REPORT.md §4-5): tầng bậc regex mở rộng
# cho 15,5% document không có Điều (Mục/Phần dùng ở Quyết định hành chính, tiết số thập
# phân dùng ở QCVN/TCVN, Phụ lục dùng ở biểu mẫu) + chặn trên cho đơn vị bất thường dài.
MUC_RE = re.compile(r"^[ \t]*Mục\s+([0-9]+|[IVXLCDM]+)\s*[\.\s:]", re.MULTILINE)
PHU_LUC_RE = re.compile(r"^[ \t]*Phụ\s+lục\s+([0-9IVXLCDM]+)\b", re.MULTILINE | re.IGNORECASE)
TIET_RE = re.compile(r"^[ \t]*(\d+\.\d+(?:\.\d+)?)\s*[\.\s]", re.MULTILINE)
MAX_UNIT_WORDS = 2000  # A2: chặn trên 1 đơn vị (Điều/Mục/...) — EDA đo outlier tới 189.366
                        # từ/đơn vị khi không chặn (regex bỏ sót heading do lỗi OCR/định dạng)
_MUC_PREFIX_STRIP_RE = re.compile(r"^\s*Mục\s+[0-9IVXLCDM]+\s*[\.\s:]*\s*", re.IGNORECASE)
_PHU_LUC_PREFIX_STRIP_RE = re.compile(r"^\s*Phụ\s+lục\s+[0-9IVXLCDM]+\s*[\.\s:]*\s*", re.IGNORECASE)
_TIET_PREFIX_STRIP_RE = re.compile(r"^\s*\d+\.\d+(?:\.\d+)?\s*[\.\s]*\s*")
SO_HEADER_RE = re.compile(r"Số\s*[:：]\s*([0-9A-Za-zĐđ/\-]+)")
SO_HIEU_RE = re.compile(r"\d{1,6}[A-Za-z]{0,3}/(?:\d{4}/)?[A-Za-zĐđ]{2,10}(?:-[A-Za-zĐđ]{2,10})?")
LOAI_VB_CANON = ["Thông tư liên tịch", "Nghị định", "Luật", "Thông tư", "Quyết định",
                 "Pháp lệnh", "Nghị quyết", "Bộ luật", "Chỉ thị"]
LOAI_PATTERN = re.compile("(" + "|".join(re.escape(x) for x in LOAI_VB_CANON) + ")", re.IGNORECASE)
DIEU_CITATION_RE = re.compile(r"Điều\s+(\d+)\s*[a-zđA-ZĐ]?\b")
_TOKEN_RE = re.compile(r"[^\W\d_]+|\d+", re.UNICODE)
_DIEU_PREFIX_STRIP_RE = re.compile(r"^\s*Điều\s+\d+[a-zđA-ZĐ]?\.?\s*", re.IGNORECASE)


def extract_vb_info(passage: str):
    m = SO_HEADER_RE.search(passage[:1500])
    so_hieu = m.group(1).strip("., ") if m else ""
    if not (so_hieu and SO_HIEU_RE.fullmatch(so_hieu)):
        m2 = SO_HIEU_RE.search(passage[:1500])
        so_hieu = m2.group(0) if m2 else ""
    m3 = LOAI_PATTERN.search(passage[:200]) or LOAI_PATTERN.search(passage[:800])
    loai_vb = ""
    if m3:
        low = m3.group(1).lower()
        for canon in LOAI_VB_CANON:
            if canon.lower() == low:
                loai_vb = canon
                break
    return loai_vb, so_hieu


def tokenize_simple(text: str) -> list:
    return _TOKEN_RE.findall(text.lower())


def norm_so_hieu(s: str) -> str:
    return s.strip().upper()

# BẢN SỬA: seed toàn cục NGAY SAU khi import xong (trước Bước 3 ở Cell 7, trước Bước 4 ở
# Cell 8 — hai nơi duy nhất gọi random.sample/random.choice) — xem giải thích đầy đủ ở Cell 2.
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_all_seeds(SEED)
print(f"  Đã seed toàn cục với SEED={SEED} (random/numpy/torch) — trước mọi lời gọi random "
      f"ở Bước 3/4, để nhiều lần chạy cùng code fine-tune trên cùng 1 tập con, tái lập được.")

In [ ]:
# Cell 5: Bước 1 — Chunk corpus, HAI TẦNG (BẢN SỬA v4, xem Cell 2/result.md §3):
#   chunk_passage()       tầng 2 — theo Điều (neo đầu dòng, tránh rách nội dung khi 1 Điều
#                          trích dẫn Điều khác trong thân bài) — CHỈ dùng cho sinh nhãn
#                          (Cell 7), fine-tune (Cell 8/10), và làm ứng viên cuối trong
#                          retrieve_two_tier() (Cell 11)
#   chunk_passage_words()  tầng 1 (MỚI) — 450 từ liên tục, dùng cho BM25+dense retrieval
#                          để CHỌN VĂN BẢN trước — biến `all_chunks` (Điều) và
#                          `all_chunks_t1` (450 từ) TÁCH BIỆT hoàn toàn từ đây, không
#                          trộn lẫn ở bất kỳ cell nào phía sau.
# Cả hai đều trích so_hieu/loai_vb từ NỘI DUNG (không phải tên file).
def _split_words_raw(text: str, n: int) -> list:
    """Cắt PHẲNG theo từ, không quan tâm ranh giới câu/cấu trúc — dùng làm fallback cuối
    (không khớp bất kỳ tầng bậc nào) hoặc chặn trên (A2) cho chunk_passage()."""
    words = text.split()
    if not words:
        return []
    return [" ".join(words[i:i + n]) for i in range(0, len(words), n)]


def chunk_passage(passage: str, doc_id) -> list:
    """Tầng 2 — TẦNG BẬC (BẢN SỬA v5, Batch 1 A1+A2 — xem EDA_ANALYSIS_REPORT.md §4-5):
    thử Điều -> Mục -> Phụ lục -> tiết (TCVN dạng "2.2.1.2") theo thứ tự, dùng tầng bậc ĐẦU
    TIÊN khớp được ít nhất 1 lần. Nếu không tầng nào khớp -> cắt 450 từ (KHÔNG còn trả
    nguyên văn bản như v4 — oracle đo trả nguyên văn bản chỉ 0,195 METEOR, result.md §3).

    A2: mỗi đơn vị tách được bị chặn trên MAX_UNIT_WORDS — EDA đo outlier tới 189.366
    từ/đơn vị (11,5% document có >=1 đơn vị >5.000 từ) do lỗi OCR/định dạng khiến heading
    nằm giữa dòng, regex (neo ^ đầu dòng) bỏ sót. Đơn vị vượt trần bị cắt lại bằng
    _split_words_raw() thay vì giữ nguyên — tránh chunk khổng lồ làm hại precision của
    render_answer() hoặc bị cross-encoder cắt cụt ở max_length.

    `dieu_so` giữ nguyên ý nghĩa cũ (số Điều, "0" nếu không phải đơn vị Điều) để KHÔNG phá
    logic đếm/citation hiện có ở load_corpus()/build_train_pairs(). `unit_type`/`unit_no` là
    trường MỚI, dùng ở render_answer() (Cell 11) để dựng câu dẫn đúng loại đơn vị."""
    for regex, unit_type in ((DIEU_RE, "dieu"), (MUC_RE, "muc"),
                              (PHU_LUC_RE, "phu_luc"), (TIET_RE, "tiet")):
        matches = list(regex.finditer(passage))
        if matches:
            break
    else:
        matches, unit_type = [], None

    if not matches:
        words_chunks = _split_words_raw(passage, 450)
        if not words_chunks:
            return [{"id": f"{doc_id}_0", "dieu_so": "0", "unit_type": "raw", "unit_no": "",
                     "loai_vb": "", "so_hieu": "", "text": passage.strip()}]
        return [{"id": f"{doc_id}_w{i}", "dieu_so": "0", "unit_type": "raw450", "unit_no": "",
                 "loai_vb": "", "so_hieu": "", "text": t}
                for i, t in enumerate(words_chunks)]

    chunks = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(passage)
        unit_no = m.group(1)
        unit_text = passage[start:end].strip()
        if len(unit_text.split()) > MAX_UNIT_WORDS:
            for j, sub in enumerate(_split_words_raw(unit_text, 450)):
                chunks.append({"id": f"{doc_id}_{unit_type}{unit_no}_{i}_{j}",
                               "dieu_so": unit_no if unit_type == "dieu" else "0",
                               "unit_type": f"{unit_type}_capped", "unit_no": unit_no,
                               "loai_vb": "", "so_hieu": "", "text": sub})
        else:
            chunks.append({"id": f"{doc_id}_{unit_type}{unit_no}_{i}",
                           "dieu_so": unit_no if unit_type == "dieu" else "0",
                           "unit_type": unit_type, "unit_no": unit_no,
                           "loai_vb": "", "so_hieu": "", "text": unit_text})
    return chunks


def chunk_passage_words(passage: str, doc_id, n: int = 450) -> list:
    """Tầng 1: cắt 450 từ liên tục, KHÔNG quan tâm ranh giới Điều. Task 1 đo trực tiếp:
    450 từ cho document recall TỐT HƠN cắt theo Điều 1,19 điểm (p=0,024, result.md §3) —
    lý do là chunk cắt theo cấu trúc pháp lý làm mất ngữ cảnh liên Điều mà câu hỏi cần."""
    words = passage.split()
    if not words:
        return [{"id": f"{doc_id}_w0", "text": passage.strip()}]
    return [{"id": f"{doc_id}_w{i // n}", "text": " ".join(words[i:i + n])}
            for i in range(0, len(words), n)]


def load_corpus(contexts_dir) -> list:
    contexts_dir = Path(contexts_dir)
    if not contexts_dir.exists():
        raise FileNotFoundError(f"Không tìm thấy {contexts_dir} — kiểm tra lại CONTEXT_DIR ở Cell 2.")
    files = sorted(contexts_dir.glob("context_*.json"))
    if not files:
        nested = contexts_dir / "selected-contexts"
        if nested.exists():
            files = sorted(nested.glob("context_*.json"))
    if not files:
        raise FileNotFoundError(f"Không tìm thấy context_*.json trong {contexts_dir}")

    all_chunks, all_chunks_t1, n_no_dieu = [], [], 0
    dieu_by_doc = {}
    for fp in files:
        try:
            with fp.open(encoding="utf-8") as f:
                doc = json.load(f)
        except Exception:
            continue
        passage = doc.get("passage")
        if not passage:
            continue
        doc_id = doc["id"]
        loai_vb, so_hieu = extract_vb_info(passage)

        # tầng 2 (Điều) — GIỮ NGUYÊN như v3, tên biến `all_chunks` không đổi để Cell 7/8/10
        # (sinh nhãn, fine-tune encoder/reranker, đào negative cùng họ) không cần sửa gì.
        chunks = chunk_passage(passage, doc_id)
        if len(chunks) == 1 and chunks[0]["dieu_so"] == "0":
            n_no_dieu += 1
        for c in chunks:
            c["loai_vb"], c["so_hieu"] = loai_vb, so_hieu
        all_chunks.extend(chunks)
        dieu_by_doc[str(doc_id)] = chunks   # tra cứu O(1) ở tầng 2 của retrieve_two_tier()

        # tầng 1 (450 từ, MỚI) — dùng cho BM25/dense retrieval, xem Cell 6/9/11.
        for c in chunk_passage_words(passage, doc_id):
            c["loai_vb"], c["so_hieu"] = loai_vb, so_hieu
            all_chunks_t1.append(c)

    pct = round(100 * (1 - n_no_dieu / len(files)), 2) if files else 0.0
    print(f"  {len(files)} văn bản -> {len(all_chunks)} chunk tầng 2 · "
          f"{len(all_chunks_t1)} chunk 450 từ (tầng 1). {pct}% văn bản có cấu trúc Điều.")
    # BẢN SỬA v5: phân bố unit_type — soi Batch 1 (A1+A2) có hiệu lực thật hay không, đặc
    # biệt bao nhiêu document rơi vào muc/phu_luc/tiet (A1) và *_capped (A2, outlier).
    unit_type_counts = Counter(c.get("unit_type", "dieu") for c in all_chunks)
    print(f"  Phân bố unit_type (tầng 2): "
          + ", ".join(f"{k}={v}" for k, v in unit_type_counts.most_common()))
    n_capped = sum(v for k, v in unit_type_counts.items() if k.endswith("_capped"))
    if n_capped:
        print(f"  [A2] {n_capped} chunk bị chặn trên MAX_UNIT_WORDS={MAX_UNIT_WORDS} rồi cắt lại 450 từ "
              f"(outlier catastrophic — xem EDA_ANALYSIS_REPORT.md §5.1).")
    n_fallback_raw = unit_type_counts.get("raw450", 0) + unit_type_counts.get("raw", 0)
    if n_fallback_raw:
        print(f"  [A1] {n_fallback_raw} chunk fallback cắt 450 từ (không khớp Điều/Mục/Phụ lục/tiết).")
    return all_chunks, all_chunks_t1, dieu_by_doc


print("=== Bước 1: Chunk corpus (2 tầng — BẢN SỬA v4, xem Cell 2/result.md §3) ===")
all_chunks, all_chunks_t1, dieu_by_doc = load_corpus(CONTEXT_DIR)
checkpoint("Xong chunking (2 tầng)")

In [ ]:
# Cell 6: Bước 2 — BM25 tự viết bằng numpy (inverted index vector hoá — nhanh trên corpus lớn)
class BM25:
    def __init__(self, tokenized_docs, k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b
        self.N = len(tokenized_docs)
        self.doc_len = np.array([len(d) for d in tokenized_docs], dtype=np.float64)
        self.avgdl = self.doc_len.mean() if self.N else 0.0

        raw_postings = defaultdict(list)
        for i, doc in enumerate(tokenized_docs):
            for term, f in Counter(doc).items():
                raw_postings[term].append((i, f))

        self.inverted: dict = {}
        for term, postings in raw_postings.items():
            idxs = np.fromiter((p[0] for p in postings), dtype=np.int32, count=len(postings))
            freqs = np.fromiter((p[1] for p in postings), dtype=np.float64, count=len(postings))
            self.inverted[term] = (idxs, freqs)

        df = {t: len(idxs) for t, (idxs, _f) in self.inverted.items()}
        idf_raw = {t: math.log((self.N - n + 0.5) / (n + 0.5) + 1) for t, n in df.items()}
        avg_idf = sum(idf_raw.values()) / len(idf_raw) if idf_raw else 0.0
        eps = 0.25 * avg_idf
        self.idf = {t: (v if v > 0 else eps) for t, v in idf_raw.items()}

    def get_scores(self, query_tokens) -> np.ndarray:
        scores = np.zeros(self.N, dtype=np.float64)
        for term in set(query_tokens):
            posting = self.inverted.get(term)
            if posting is None:
                continue
            idxs, freqs = posting
            idf = self.idf[term]
            denom = freqs + self.k1 * (1 - self.b + self.b * self.doc_len[idxs] / self.avgdl)
            contrib = idf * freqs * (self.k1 + 1) / denom
            scores[idxs] += contrib
        return scores

    def top_k(self, query_tokens, k: int) -> list:
        scores = self.get_scores(query_tokens)
        return list(np.argsort(-scores)[:k])


print("=== Bước 2: BM25 index (2 tầng — BẢN SỬA v4) ===")
# tầng 2 (Điều) — GIỮ NGUYÊN như v3: dùng cho negative sampling ở Cell 8 (_build_training_rows)
# và Cell 10 (mine_family_negatives), CẢ HAI đều lấy positional index vào `all_chunks` nên
# `bm25` và `all_chunks` PHẢI luôn là một cặp cùng corpus — không được lẫn với bm25_t1.
tokenized = [tokenize_simple(f"{c.get('loai_vb','')} {c['text']}") for c in all_chunks]
bm25 = BM25(tokenized)
# tầng 1 (450 từ, MỚI) — dùng cho retrieve_two_tier() ở Cell 11, cặp với all_chunks_t1.
tokenized_t1 = [tokenize_simple(f"{c.get('loai_vb','')} {c['text']}") for c in all_chunks_t1]
bm25_t1 = BM25(tokenized_t1)
checkpoint("Xong BM25 index (2 tầng)")

In [ ]:
# Cell 7: Bước 3 — Sinh nhãn (question -> chunk) từ citation trong train.json (+ warmup.json
# nếu USE_WARMUP=True, cùng schema — gộp thêm dữ liệu train, KHÔNG gộp vào mẫu dev-eval để
# tránh lẫn chất lượng nhãn chưa kiểm chứng vào lúc CHỌN cấu hình cuối cùng)
def extract_citations(answer) -> list:
    # answer có thể KHÔNG phải string (đã gặp thật: warmup.json có answer kiểu list ở một số
    # câu, khác train.json toàn string) — bỏ qua câu đó thay vì crash cả pipeline.
    if not isinstance(answer, str):
        return []
    out = []
    for m in DIEU_CITATION_RE.finditer(answer):
        window = answer[m.end(): m.end() + 60]
        so_m = SO_HIEU_RE.search(window)
        if so_m and so_m.start() <= 40:
            out.append((m.group(1), so_m.group(0)))
    return out


def build_train_pairs(train_data: dict, all_chunks: list):
    so_hieu_index = {}
    for c in all_chunks:
        if c["so_hieu"] and c["dieu_so"] != "0":
            so_hieu_index.setdefault((c["dieu_so"], norm_so_hieu(c["so_hieu"])), c["id"])

    positive = {}
    n_skipped_type = 0
    for qid, item in train_data.items():
        if not isinstance(item.get("answer"), str):
            n_skipped_type += 1
            continue
        for dieu, so_hieu in extract_citations(item["answer"]):
            key = (dieu, norm_so_hieu(so_hieu))
            if key in so_hieu_index:
                positive[qid] = so_hieu_index[key]
                break
    if n_skipped_type:
        print(f"  [CẢNH BÁO] {n_skipped_type} câu có answer KHÔNG phải string -> bỏ qua khi "
              f"sinh nhãn, không tính vào positive pairs.")
    chunk_by_id = {c["id"]: c for c in all_chunks}
    return positive, chunk_by_id


print("=== Bước 3: Sinh nhãn từ train.json" + (" + warmup.json" if USE_WARMUP else "") + " ===")
with open(TRAIN_PATH, encoding="utf-8") as f:
    train_data = json.load(f)
print(f"  train.json: {len(train_data)} câu")

# warmup.json: gộp thêm CHỈ KHI USE_WARMUP=True VÀ file tồn tại VÀ đúng schema
# {qid: {"question": str, "answer": str}} — lọc chặt cả kiểu dữ liệu. KHÔNG gộp vào mẫu
# dev-eval ở Bước 6 — dev-eval chỉ dùng train.json gốc để giữ tín hiệu chọn cấu hình đáng tin
# (xem phần 2.2 trong PHAN_TICH_KY_THUAT.md). USE_WARMUP=False cho phép ablation có kiểm soát.
train_data_for_pairs = dict(train_data)
n_warmup_used = 0
if USE_WARMUP and os.path.exists(WARMUP_PATH):
    try:
        with open(WARMUP_PATH, encoding="utf-8") as f:
            warmup_data = json.load(f)
        n_bad_type = 0
        for qid, item in warmup_data.items():
            if not isinstance(item, dict):
                n_bad_type += 1
                continue
            q, a = item.get("question"), item.get("answer")
            if isinstance(q, str) and isinstance(a, str):
                train_data_for_pairs[f"warmup_{qid}"] = {"question": q, "answer": a}
                n_warmup_used += 1
            else:
                n_bad_type += 1
        print(f"  warmup.json: {len(warmup_data)} câu, {n_warmup_used} câu đúng schema -> gộp thêm"
              + (f", {n_bad_type} câu sai kiểu -> bỏ qua." if n_bad_type else ".")
              + " KHÔNG gộp vào mẫu dev-eval/Recall@k ở Bước 6.")
    except Exception as e:
        print(f"  [CẢNH BÁO] Có WARMUP_PATH nhưng đọc lỗi ({e}) -> bỏ qua, chỉ dùng train.json.")
elif USE_WARMUP:
    print(f"  USE_WARMUP=True nhưng không thấy {WARMUP_PATH} -> chỉ dùng train.json.")
else:
    print(f"  USE_WARMUP=False -> chỉ dùng train.json (bỏ qua warmup.json dù có tồn tại).")

train_positive, chunk_by_id = build_train_pairs(train_data_for_pairs, all_chunks)
print(f"  Positive pairs: {len(train_positive)}/{len(train_data_for_pairs)}")
# v6: KHÔNG còn nhãn Task 1 — chỉ nhãn citation của chính Task 2 (build_train_pairs trên).
# =============================================================================
# BẢN SỬA v4 — RÒ RỈ TRAIN/DEV-EVAL phát hiện khi rà lại pipeline (result.md, mục lỗi mới)
# =============================================================================
# Trước v4: Bước 4/5b lấy TOÀN BỘ train_positive làm dữ liệu fine-tune, còn Bước 6 (dev-eval,
# Cell 12) lấy 300 câu NGẪU NHIÊN từ TOÀN BỘ train_data — KHÔNG có bước nào loại các câu
# dev-eval khỏi tập train trước đó. Với ~3.579 câu có nhãn citation / 7.000 câu train_data,
# MỘT CÂU DEV-EVAL BẤT KỲ CÓ ~51% XÁC SUẤT ĐÃ ĐƯỢC DÙNG ĐỂ FINE-TUNE reranker/encoder — tức
# METEOR ở Bước 6 (và mọi lựa chọn suy ra từ nó: TOP_N_ANSWER, reranker nào thắng, T của hàm
# gộp) MỘT PHẦN đang đo trí nhớ, không phải khả năng tổng quát hoá. Cố định dev_ids Ở ĐÂY,
# TRƯỚC khi Bước 4 build training rows, rồi loại khỏi train_positive.
#
# train_positive_all GIỮ NGUYÊN (không loại gì) — Cell 12 dùng nó để đo Recall@k, vì đó là
# ĐÁNH GIÁ bằng nhãn thật, không phải huấn luyện, nên không có gì phải loại trừ ở đó.
random.seed(SEED)
dev_ids = random.sample(list(train_data.keys()), min(DEV_EVAL_SAMPLE_SIZE, len(train_data)))
dev_ids_set = set(dev_ids)
train_positive_all = dict(train_positive)
n_dev_had_label = sum(1 for q in dev_ids if q in train_positive)
train_positive = {qid: cid for qid, cid in train_positive.items() if qid not in dev_ids_set}
print(f"  Cố định {len(dev_ids)} câu dev-eval NGAY TỪ ĐÂY (trước Bước 4). {n_dev_had_label} "
      f"câu trong số đó có nhãn citation -> LOẠI khỏi tập fine-tune "
      f"({len(train_positive_all)} -> {len(train_positive)} positive pair khả dụng để "
      f"train), nhưng VẪN dùng nhãn gốc (train_positive_all) để đo Recall@k ở Bước 6.")

# =============================================================================
# v6_1 — CHỐT TẬP HARVEST NGAY TẠI ĐÂY, CÙNG CHỖ VÀ CÙNG SEED VỚI dev_ids
# =============================================================================
# Tập harvest dùng cho: (1) train bộ chọn LTR (Cell 13), (2) error analysis (Cell 15).
# Nó PHẢI sạch với fine-tune encoder, nếu không mọi số đo trên đó là đo trí nhớ — đúng lỗi
# mà v4 đã phải sửa một lần (khối ngay trên: 51% câu dev từng nằm trong tập fine-tune).
#
# Chỗ khéo: nhãn của LTR là **METEOR của từng ứng viên so với gold answer**, KHÔNG phải nhãn
# citation. Nên LTR train được trên chính những câu KHÔNG phân giải được citation — mà đó
# đúng là nhóm câu chưa từng đi vào fine-tune encoder (fine-tune chỉ ăn train_positive, tức
# chỉ câu CÓ citation). Hai ràng buộc gặp nhau ở đúng một chỗ:
#
#     ltr_pool = (câu KHÔNG có nhãn citation) \ dev_ids
#              = câu chưa từng vào fine-tune, và cũng không phải câu dùng để chấm
#
# Nhờ vậy điểm đo trên toàn bộ harvest KHÔNG bị thổi phồng bởi contamination encoder — khác
# hẳn con số dev 0,5709 của lượt trước, vốn lẫn cả câu model đã học. Phần lệch còn lại chỉ
# là lệch phân phối train-vs-public, không phải rò rỉ.
_no_citation = [q for q in train_data.keys()
                 if q not in train_positive_all and q not in dev_ids_set]
random.shuffle(_no_citation)          # đã seed ở Cell 4 -> tái lập được
ltr_pool = _no_citation
# dev_ids đi TRƯỚC: pha A phải chấm xong dev thì Cell 13 mới có gì để gate.
harvest_ids_all = list(dev_ids) + ltr_pool
print(f"  [v6_1] Tập harvest: {len(dev_ids)} câu dev (gate) + {len(ltr_pool)} câu không có "
      f"nhãn citation = {len(harvest_ids_all)} câu khả dụng.")
print(f"         Cả hai nhóm đều CHƯA từng vào fine-tune encoder -> METEOR đo trên đây "
      f"không bị thổi phồng bởi contamination.")
if len(ltr_pool) < LTR_MIN_GROUPS:
    print(f"  [CẢNH BÁO] chỉ {len(ltr_pool)} câu sạch cho LTR (< LTR_MIN_GROUPS="
          f"{LTR_MIN_GROUPS}) -> Cell 13 nhiều khả năng bỏ qua LTR.")

checkpoint("Xong sinh nhãn")


In [ ]:
# Cell 8: Bước 4 — Fine-tune 2 dense encoder SONG SONG THẬT trên 2 GPU riêng (subprocess)
#
# CHỦ Ý dùng subprocess (không phải threading/multiprocessing.Process kiểu fork): Cell 3 đã
# init CUDA context trong tiến trình notebook (gọi torch.cuda.get_device_properties) — fork
# SAU khi CUDA đã init là lỗi kinh điển ("Cannot re-initialize CUDA in forked subprocess").
# subprocess.Popen luôn khởi động tiến trình Python HOÀN TOÀN MỚI (tương đương spawn), mỗi
# tiến trình con tự import torch riêng, tự nhận CUDA_VISIBLE_DEVICES riêng — an toàn tuyệt
# đối, đúng pattern `run_shards` của bản gốc `run_qa.py` đầu dự án.
#
# Checkpoint ĐƯỢC LƯU lần này (khác các bản trước) — bạn cần dùng lại qua nhiều phiên Kaggle,
# và vì tiến trình con/cha là 2 process riêng, cách DUY NHẤT đưa model đã train về tiến trình
# cha là qua đĩa (`model.save_pretrained()` rồi `SentenceTransformer(path)` load lại).
import subprocess
import sys  # SỬA: sys.executable dùng để gọi WORKER_SCRIPT bên dưới — thiếu import này\n# là bug thật (Python vẫn cho phép dùng module chưa import NẾU nó tình cờ đã có trong\n# builtins/đã import ở cell khác cùng kernel session — dễ chạy "trót lọt" trong notebook\n# rồi lỗi khó hiểu khi chạy .py độc lập; luôn import tường minh module mình dùng).

print("=== Bước 4: Fine-tune 2 dense encoder song song (bge-m3 @ cuda:0, e5-large @ cuda:1) ===")

WORKER_SCRIPT = os.path.join(CACHE_DIR, "_train_encoder_worker.py")
# Worker con — fine-tune MOT SentenceTransformer tren MOT GPU, chay qua subprocess.Popen,
# nhan tham so qua argv, khong phu thuoc bien toan cuc cua notebook.
worker_code = '''
import argparse, json, os, sys, time


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--base-model", required=True)
    p.add_argument("--gpu-index", required=True)
    p.add_argument("--rows-path", required=True)
    p.add_argument("--output-dir", required=True)
    p.add_argument("--max-seq-len", type=int, default=256)
    p.add_argument("--batch-size", type=int, default=64)
    p.add_argument("--mini-batch-size", type=int, default=16)
    p.add_argument("--time-budget-sec", type=float, required=True)
    p.add_argument("--seed", type=int, required=True)
    p.add_argument("--query-prefix", default="")
    p.add_argument("--passage-prefix", default="")
    p.add_argument("--use-lora", action="store_true")
    p.add_argument("--use-8bit-optim", action="store_true")
    p.add_argument("--lora-r", type=int, default=16)
    p.add_argument("--lora-alpha", type=int, default=32)
    p.add_argument("--lora-dropout", type=float, default=0.05)
    args = p.parse_args()

    os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu_index
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    import random
    import numpy as np
    import torch
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    try:
        from datasets import Dataset
        from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainer,
                                            SentenceTransformerTrainingArguments)
        from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
        import sentence_transformers as _st
        import accelerate as _acc
    except ImportError as e:
        print(f"[LOI IMPORT] {e}", flush=True)
        print(f"[LOI IMPORT] Ban co the dang dung sentence-transformers/accelerate qua cu -- "
              f"CachedMultipleNegativesRankingLoss can sentence-transformers >= 3.0, "
              f"accelerate >= 1.1.0. Chay:", flush=True)
        print(f'    pip install -U "sentence-transformers>=3.0" "accelerate>=1.1.0"', flush=True)
        sys.exit(1)
    st_ver = tuple(int(x) for x in _st.__version__.split(".")[:2] if x.isdigit())
    if st_ver < (3, 0):
        print(f"[PHIEN BAN CU] sentence-transformers={_st.__version__} (can >= 3.0). Chay: "
              f'pip install -U "sentence-transformers>=3.0"', flush=True)
        sys.exit(1)

    # BAN SUA (log loi that: torch.AcceleratorError OOM ngay o optimizer.step() DAU TIEN --
    # AdamW full fine-tune cho model ~568M can ~6-7GB CHI RIENG optimizer state, khong phu
    # thuoc batch size -- khong batch nao du tren GPU 4GB). LoRA: chi train 1 phan rat nho
    # tham so (dong bang phan con lai) -> optimizer state nho lai theo dung ty le do, GIAI
    # QUYET DUOC loai OOM nay ma batch-backoff khong the giai quyet.
    optim_name = "adamw_torch"
    if args.use_8bit_optim:
        try:
            import bitsandbytes  # noqa: F401
            optim_name = "adamw_bnb_8bit"
            print(f"[{args.base_model}] Dung optimizer AdamW 8-bit (bitsandbytes).", flush=True)
        except ImportError:
            print(f"[{args.base_model}] bitsandbytes khong cai duoc -> dung AdamW thuong.", flush=True)

    with open(args.rows_path, encoding="utf-8") as f:
        rows = json.load(f)
    if args.query_prefix or args.passage_prefix:
        fixed = []
        for r in rows:
            r2 = dict(r)
            r2["anchor"] = args.query_prefix + r["anchor"]
            for k in r:
                if k.startswith("positive") or k.startswith("negative"):
                    r2[k] = args.passage_prefix + r[k]
            fixed.append(r2)
        rows = fixed
    dataset = Dataset.from_list(rows)

    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer(args.base_model, device=device)
    model.max_seq_length = args.max_seq_len

    lora_used = False
    if args.use_lora:
        try:
            from peft import LoraConfig, get_peft_model
            lora_cfg = LoraConfig(r=args.lora_r, lora_alpha=args.lora_alpha,
                                   lora_dropout=args.lora_dropout, bias="none",
                                   target_modules="all-linear")
            model[0].auto_model = get_peft_model(model[0].auto_model, lora_cfg)
            lora_used = True
            n_trainable = sum(pp.numel() for pp in model[0].auto_model.parameters() if pp.requires_grad)
            n_total = sum(pp.numel() for pp in model[0].auto_model.parameters())
            print(f"[{args.base_model}] LoRA bat: {n_trainable}/{n_total} tham so co the train "
                  f"({100*n_trainable/max(n_total,1):.2f}%)", flush=True)
        except Exception as e:
            print(f"[{args.base_model}] LoRA loi ({e}) -> full fine-tune (can nhieu VRAM hon).", flush=True)

    batch_size, mini_batch_size = args.batch_size, args.mini_batch_size
    max_steps, calib_time = 0, None
    t0 = time.time()
    for attempt in range(4):
        try:
            loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=mini_batch_size)
            calib_steps = min(10, max(1, len(dataset) // batch_size))
            calib_args = SentenceTransformerTrainingArguments(
                output_dir=args.output_dir + "_tmp", max_steps=calib_steps,
                per_device_train_batch_size=batch_size, logging_steps=calib_steps + 1,
                save_strategy="no", report_to=[], disable_tqdm=True, fp16=(device == "cuda:0"),
                optim=optim_name)
            c0 = time.time()
            print(f"[{args.base_model}] calib training (batch={batch_size}, mini_batch={mini_batch_size})...", flush=True)
            SentenceTransformerTrainer(model=model, args=calib_args, train_dataset=dataset, loss=loss).train()
            calib_time = (time.time() - c0) / calib_steps

            budget_left = args.time_budget_sec - (time.time() - t0) - 60
            max_steps = max(0, int(budget_left / max(calib_time, 1e-6)))
            max_steps = min(max_steps, (len(dataset) // batch_size) * 8)
            print(f"[{args.base_model}] calib {calib_time:.2f}s/step, ngan sach con "
                  f"{budget_left/60:.1f} phut -> {max_steps} step", flush=True)

            if max_steps > 0:
                targs = SentenceTransformerTrainingArguments(
                    output_dir=args.output_dir + "_tmp", max_steps=max_steps,
                    per_device_train_batch_size=batch_size, learning_rate=2e-5,
                    warmup_steps=0.05, lr_scheduler_type="cosine",
                    logging_steps=max(1, max_steps // 20), save_strategy="no", report_to=[],
                    fp16=(device == "cuda:0"), optim=optim_name)
                SentenceTransformerTrainer(model=model, args=targs, train_dataset=dataset, loss=loss).train()
            break
        except Exception as e:
            if "out of memory" in str(e).lower() and mini_batch_size > 1:
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass  # cache da can kiet toi muc khong con gi de don -- bo qua, cu lui batch
                mini_batch_size = max(1, mini_batch_size // 2)
                print(f"[{args.base_model}] OOM -> mini_batch_size={mini_batch_size}", flush=True)
                continue
            raise

    if lora_used:
        try:
            model[0].auto_model = model[0].auto_model.merge_and_unload()
            print(f"[{args.base_model}] Da merge LoRA vao model goc.", flush=True)
        except Exception as e:
            print(f"[{args.base_model}] Merge LoRA loi ({e}) -> luu adapter rieng.", flush=True)

    model.save_pretrained(args.output_dir)
    meta = {"max_steps": max_steps, "mini_batch_final": mini_batch_size,
            "calib_time_s": calib_time, "elapsed_s": time.time() - t0,
            "lora_used": lora_used, "optim": optim_name}
    with open(args.output_dir + "_meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f)
    print(f"[{args.base_model}] DONE -> {args.output_dir}", flush=True)


if __name__ == "__main__":
    main()
'''
with open(WORKER_SCRIPT, "w", encoding="utf-8") as f:
    f.write(worker_code)

# ---- Tạo training rows 1 LẦN trong tiến trình cha (dùng chung cho cả 2 encoder) ----
def _build_training_rows(train_positive, train_data, chunk_by_id, all_chunks, bm25, n_neg=N_NEG_PER_ROW):
    rows = []
    n = len(train_positive)
    for i, (qid, pos_id) in enumerate(train_positive.items()):
        question = train_data[qid]["question"]
        pos_text = chunk_by_id[pos_id]["text"]
        token_q = tokenize_simple(question)
        ranked = bm25.top_k(token_q, 60)
        neg_ids = [all_chunks[i2]["id"] for i2 in ranked[5:60] if all_chunks[i2]["id"] != pos_id][:n_neg]
        if len(neg_ids) < n_neg:
            pool = [c["id"] for c in all_chunks if c["id"] != pos_id]
            while len(neg_ids) < n_neg and pool:
                neg_ids.append(random.choice(pool))
        # _pos_id/_qid: metadata cho tầng đào negative "cùng họ" ở Cell 10. Tiền tố "_"
        # để phân biệt với các khoá anchor/positive/negative_* mà trainer thật sự đọc.
        row = {"anchor": question, "positive": pos_text, "_pos_id": pos_id, "_qid": qid}
        for j, nid in enumerate(neg_ids[:n_neg]):
            row[f"negative_{j+1}"] = chunk_by_id[nid]["text"]
        rows.append(row)
        if (i + 1) % 500 == 0 or (i + 1) == n:
            print(f"    _build_training_rows: {i+1}/{n}  ({elapsed()/60:.1f} phút)")
    return rows


finetune_info = {"used_finetune": False, "reason": None, "n_pairs_available": len(train_positive),
                  "n_pairs_used": 0, "models": {}}
DENSE_CHANNELS = []  # điền ở cuối cell; embeddings điền ở Cell 9

# BẢN SỬA: build `rows` (anchor/positive/negative) MỘT LẦN, KHÔNG PHỤ THUỘC USE_FINETUNE của
# dense encoder — Cell 10 (fine-tune reranker) cần dùng lại đúng `rows` này. Trước đây rows chỉ
# được build bên trong nhánh "if use_finetune" của dense encoder, nên nếu USE_FINETUNE=False thì
# Cell 10 không có gì để fine-tune reranker dù USE_RERANKER_FINETUNE=True.
#
# BẢN SỬA THÊM: tách riêng `rows_clean` (CHỈ nhãn citation, chính xác tới từng Điều) khỏi
# `rows` (citation + Task 1) — nhãn Task 1 là GIÁM SÁT YẾU ở mức Điều (chọn Điều điểm BM25 cao
# nhất TRONG đúng văn bản gold — có thể sai Điều dù đúng văn bản, xem docstring
# build_task1_pairs() ở Cell 7). Dense encoder train bằng contrastive loss với nhiều negative,
# chịu nhiễu nhãn tốt — dùng cả 2 nguồn (`rows`) là hợp lý. Reranker train bằng margin ranking
# loss trên 1 cặp positive/negative mỗi lần, KHÔNG có gì làm mềm nhiễu — lỡ học "Điều sai nhưng
# đúng văn bản" thành positive thật sẽ kéo NGƯỢC độ chính xác, đúng kiểu lỗi khớp với quan sát
# thật: thêm nhãn Task 1 làm METEOR tăng mạnh (tìm đúng NHIỀU câu hỏi hơn) nhưng ROUGE-L gần
# như đứng yên (không chính xác hơn ở mức từng chữ) — nghi vấn hợp lý là do một phần nhãn Điều
# không hoàn toàn đúng đang lẫn vào huấn luyện. Reranker vì vậy CHỈ train trên `rows_clean`.
rows_needed = (USE_FINETUNE or USE_RERANKER_FINETUNE) and len(train_positive) >= MIN_TRAIN_PAIRS \
              and remaining() > 10 * 60
rows, rows_path = [], None
rows_clean = []
if rows_needed:
    train_positive_used = train_positive
    if len(train_positive) > MAX_TRAIN_EXAMPLES:
        sampled_qids = random.sample(list(train_positive.keys()), MAX_TRAIN_EXAMPLES)
        train_positive_used = {qid: train_positive[qid] for qid in sampled_qids}
        print(f"  Có {len(train_positive)} positive pairs, lấy mẫu {MAX_TRAIN_EXAMPLES} "
              f"(tái lập được nhờ SEED={SEED}).")
    finetune_info["n_pairs_used"] = len(train_positive_used)

    print(f"  Đang tạo training rows (dense encoder — citation + Task 1)...")
    rows = _build_training_rows(train_positive_used, train_data_for_pairs, chunk_by_id, all_chunks, bm25)
    rows_path = os.path.join(CACHE_DIR, "train_rows.json")
    with open(rows_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False)
    print(f"  {len(rows)} rows -> {rows_path}")

    # BẢN SỬA v6_1: `rows_clean` CHỈ phục vụ fine-tune reranker, mà v6_1 đã tắt hẳn nhánh đó
    # (xem khối lý do ở Cell 2). Lượt build thứ hai này tốn 3,4 phút trong lượt trước và cho
    # ra một mảng không ai đọc. Chỉ build khi cờ thật sự bật.
    if USE_RERANKER_FINETUNE:
        clean_positive = {qid: cid for qid, cid in train_positive_used.items()
                           if not str(qid).startswith("task1_")}
        if clean_positive:
            print(f"  Đang tạo training rows (reranker — CHỈ citation, {len(clean_positive)} câu)...")
            rows_clean = _build_training_rows(clean_positive, train_data_for_pairs, chunk_by_id,
                                               all_chunks, bm25)
            print(f"  {len(rows_clean)} rows_clean (reranker)")
    else:
        print(f"  USE_RERANKER_FINETUNE=False -> BỎ QUA lượt build rows_clean thứ hai "
              f"(tiết kiệm ~3,4 phút, đo từ lượt trước).")

use_finetune = USE_FINETUNE and bool(rows)
if not use_finetune:
    reason = ("USE_FINETUNE=False" if not USE_FINETUNE else
               ("chưa có rows (xem lý do rows_needed=False ở trên)" if not rows else "?"))
    print(f"  {reason} -> dùng zero-shot cho cả 2 dense encoder, không fine-tune.")
    finetune_info["reason"] = reason
    from sentence_transformers import SentenceTransformer
    # Nạp có ĐƯỜNG LÙI. Cặp encoder mới (vnembv2 + harrier) là chuyển giao từ Task 1 và
    # CHƯA từng chạy trên Kaggle: có thể thiếu quyền tải, thiếu trust_remote_code, hoặc
    # kiến trúc sentence-transformers không nhận. Hỏng ở đây mà không có đường lùi là mất
    # trắng một phiên GPU 6 tiếng, nên thử -> lỗi thì quay về cặp cũ đã cho 0,5528 và NÓI TO.
    def _load(name, dev):
        return SentenceTransformer(name, device=dev, trust_remote_code=True)

    try:
        m_a = _load(BASE_DENSE_MODEL_A, DEVICES[0])
        m_b = _load(BASE_DENSE_MODEL_B, DEVICES[-1])
    except Exception as e:
        print(f"  ⛔ Không nạp được cặp encoder mới ({type(e).__name__}: {e})")
        print(f"     -> LÙI VỀ bge-m3 + e5-large (cấu hình đã cho 0,5528). "
              f"Mọi so sánh với bản mới KHÔNG còn hiệu lực.")
        BASE_DENSE_MODEL_A, BASE_DENSE_MODEL_B = "BAAI/bge-m3", "intfloat/multilingual-e5-large"
        QUERY_PREFIX_A, QUERY_PREFIX_B, PASSAGE_PREFIX_B = "", "query: ", "passage: "
        m_a = _load(BASE_DENSE_MODEL_A, DEVICES[0])
        m_b = _load(BASE_DENSE_MODEL_B, DEVICES[-1])
    m_a.max_seq_length = DENSE_MAX_SEQ_LEN
    m_b.max_seq_length = DENSE_MAX_SEQ_LEN
    DENSE_CHANNELS = [
        {"name": "A", "model": m_a, "embeddings": None,
     "query_prefix": QUERY_PREFIX_A, "passage_prefix": ""},
        {"name": "B", "model": m_b, "embeddings": None,
     "query_prefix": QUERY_PREFIX_B, "passage_prefix": PASSAGE_PREFIX_B},
    ]
else:
    specs = [
        # BẢN SỬA: tên slot suy TỪ MODEL THẬT, không viết cứng. Bản cũ luôn ghi
        # "bge-m3"/"e5-large" vào log dù USE_NEW_ENCODERS=True đang nạp model khác —
        # khiến log ver9 không thể quy kết, và chính tôi đã đọc sai log vì lý do đó.
        {"name": BASE_DENSE_MODEL_A.split("/")[-1], "base_model": BASE_DENSE_MODEL_A,
         "gpu": DEVICES[0].split(":")[-1],
         "out": os.path.join(CHECKPOINT_DIR, "A-ft"), "mini_batch": TRAIN_MINI_BATCH_SIZE,
     "query_prefix": QUERY_PREFIX_A, "passage_prefix": ""},
        {"name": BASE_DENSE_MODEL_B.split("/")[-1], "base_model": BASE_DENSE_MODEL_B,
         "gpu": DEVICES[-1].split(":")[-1],
         "out": os.path.join(CHECKPOINT_DIR, "B-ft"), "mini_batch": TRAIN_MINI_BATCH_SIZE_B,
     "query_prefix": QUERY_PREFIX_B, "passage_prefix": PASSAGE_PREFIX_B},
    ]
    # Chỉ chạy THẬT SỰ song song nếu có >= 2 GPU riêng biệt cho 2 spec — nếu chỉ 1 GPU, cả 2
    # subprocess sẽ tranh cùng 1 thẻ nếu phóng cùng lúc (dễ OOM cả hai) -> chạy TUẦN TỰ.
    run_parallel = len(DEVICES) > 1 and specs[0]["gpu"] != specs[1]["gpu"]
    time_budget_each = max(600.0, min(remaining() - 5 * 60, FINETUNE_TIME_BUDGET_SEC)
                            / (1.0 if run_parallel else 2.0))
    print(f"  Chạy {'SONG SONG (2 GPU riêng)' if run_parallel else 'TUẦN TỰ (chỉ 1 GPU khả dụng)'} "
          f"— ngân sách mỗi encoder ~{time_budget_each/60:.0f} phút.")

    def _launch(spec):
        log_path = os.path.join(CACHE_DIR, f"train_{spec['name']}.log")
        cmd = [sys.executable, WORKER_SCRIPT,
               "--base-model", spec["base_model"], "--gpu-index", spec["gpu"],
               "--rows-path", rows_path, "--output-dir", spec["out"],
               "--max-seq-len", str(DENSE_MAX_SEQ_LEN), "--batch-size", str(TRAIN_BATCH_SIZE),
               "--mini-batch-size", str(spec.get("mini_batch", TRAIN_MINI_BATCH_SIZE)), "--time-budget-sec", str(time_budget_each),
               "--seed", str(SEED), "--query-prefix", spec["query_prefix"], "--passage-prefix", spec["passage_prefix"],
               "--lora-r", str(LORA_R), "--lora-alpha", str(LORA_ALPHA), "--lora-dropout", str(LORA_DROPOUT)]
        if USE_LORA:
            cmd.append("--use-lora")
        if USE_8BIT_OPTIM:
            cmd.append("--use-8bit-optim")
        lf = open(log_path, "w")
        print(f"  Khởi động fine-tune {spec['name']} trên GPU {spec['gpu']} -> log: {log_path}")
        return subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT), lf

    failed = []
    if run_parallel:
        procs = [(spec, *_launch(spec)) for spec in specs]
        print(f"  Đang chờ {len(procs)} tiến trình fine-tune song song...", flush=True)
        for spec, proc, lf in procs:
            rc = proc.wait()
            print(f"  {spec['name']}: xong, mã thoát {rc}")
            if rc != 0:
                failed.append(spec["name"])
            lf.close()
    else:
        for spec in specs:
            proc, lf = _launch(spec)
            rc = proc.wait()
            print(f"  {spec['name']}: xong, mã thoát {rc}")
            if rc != 0:
                failed.append(spec["name"])
            lf.close()
    if failed:
        raise SystemExit(f"Fine-tune lỗi: {failed} — xem log trong {CACHE_DIR}/train_<tên>.log")

    from sentence_transformers import SentenceTransformer
    for spec in specs:
        meta_path = spec["out"] + "_meta.json"
        with open(meta_path, encoding="utf-8") as f:
            m = json.load(f)
        finetune_info["models"][spec["name"]] = m
        print(f"  {spec['name']}: {m['max_steps']} step, mini_batch cuối={m['mini_batch_final']}, "
              f"{m['elapsed_s']/60:.1f} phút")

    m_a = SentenceTransformer(specs[0]["out"], device=DEVICES[0])
    m_b = SentenceTransformer(specs[1]["out"], device=DEVICES[-1])
    DENSE_CHANNELS = [
        {"name": "A", "model": m_a, "embeddings": None,
     "query_prefix": QUERY_PREFIX_A, "passage_prefix": ""},
        {"name": "B", "model": m_b, "embeddings": None,
     "query_prefix": QUERY_PREFIX_B, "passage_prefix": PASSAGE_PREFIX_B},
    ]
    finetune_info["used_finetune"] = True
    print(f"  Checkpoint đã lưu trong {CHECKPOINT_DIR} — tự tải về nếu muốn dùng lại phiên sau.")

checkpoint("Xong Bước 4 (2 dense encoder)")


In [ ]:
# Cell 9 [v6_2]: Bước 5 — Encode toàn bộ corpus CHO CẢ 2 ENCODER
#
# v6_2 — ENCODE FP16 CÓ CỔNG KIỂM TRA. Lượt v6_1 encode fp32 mất 185 phút (29% phiên), trong
# khi nhánh 1-GPU của chính cell này vốn đã .half(). Nhánh 2-GPU (multi-process pool) chưa
# bao giờ ép fp16. Tiết kiệm thời gian ở đây là thứ trả cho fine-tune reranker ở Cell 10.
#
# Rủi ro có thật: harrier là họ Qwen3, mà Qwen dễ tràn số (inf/NaN) ở fp16. Nên KHÔNG ép mù:
# encode một mẫu cả fp32 lẫn fp16, chỉ dùng fp16 khi không có NaN VÀ cosine nhỏ nhất giữa hai
# bản >= FP16_MIN_COS. Trượt cổng -> kênh đó quay về fp32 như v6_1, không có gì hỏng.
print(f"=== Bước 5 [v6_2]: Encode toàn bộ corpus (tầng 1, 450 từ) cho {len(DENSE_CHANNELS)} encoder ===")
texts_raw = [c["text"] for c in all_chunks_t1]
encode_info = {}


def _fp16_guard(model, sample_texts, device):
    """-> (dùng_fp16, thông_tin). Để model ở fp16 nếu qua cổng, fp32 nếu trượt."""
    dtype0 = next(model.parameters()).dtype
    if dtype0 == torch.float16:
        return True, {"note": "checkpoint đã là fp16 sẵn"}
    model.to(device)
    ref = model.encode(sample_texts, batch_size=32, convert_to_numpy=True,
                       normalize_embeddings=True, device=device)
    model.half()
    try:
        emb = model.encode(sample_texts, batch_size=32, convert_to_numpy=True,
                           normalize_embeddings=True, device=device)
    except Exception as e:
        model.float()
        return False, {"error": f"{type(e).__name__}: {e}"}
    finite = bool(np.isfinite(emb).all())
    cos = (ref.astype(np.float64) * emb.astype(np.float64)).sum(1) if finite else np.array([0.0])
    info = {"finite": finite, "min_cos": round(float(cos.min()), 5),
            "mean_cos": round(float(cos.mean()), 5), "n_sample": len(sample_texts)}
    ok = finite and float(cos.min()) >= FP16_MIN_COS
    if not ok:
        model.float()
    return ok, info


_rng = random.Random(SEED)
_by_len = sorted(range(len(texts_raw)), key=lambda i: -len(texts_raw[i]))
# Nửa mẫu là chunk DÀI nhất (tràn số hay xảy ra ở chuỗi dài), nửa ngẫu nhiên.
_guard_idx = _by_len[: FP16_GUARD_N // 2] + _rng.sample(range(len(texts_raw)), FP16_GUARD_N // 2)

for ch in DENSE_CHANNELS:
    t0 = time.time()
    texts = [ch["passage_prefix"] + t for t in texts_raw] if ch["passage_prefix"] else texts_raw
    model = ch["model"]
    dev0 = str(next(model.parameters()).device)
    use_fp16, ginfo = False, {"skipped": "ENCODE_FP16=False hoặc không có CUDA"}
    if ENCODE_FP16 and dev0.startswith("cuda"):
        use_fp16, ginfo = _fp16_guard(model, [texts[i] for i in _guard_idx], dev0)
    encode_info[ch["name"]] = {"fp16": use_fp16, **ginfo}
    print(f"  [{ch['name']}] cổng fp16: {'QUA' if use_fp16 else 'TRƯỢT -> fp32'} {ginfo}")
    print(f"  [{ch['name']}] encode {len(texts)} chunk ({'fp16' if use_fp16 else 'fp32'})"
          + (f' (tiền tố "{ch["passage_prefix"]}")' if ch["passage_prefix"] else "") + " ...")
    if len(DEVICES) > 1 and DEVICES[0].startswith("cuda"):
        # Pool spawn nhận bản pickle của model -> dtype hiện tại (fp16 nếu qua cổng) đi theo.
        pool = model.start_multi_process_pool(target_devices=DEVICES)
        try:
            emb = model.encode_multi_process(texts, pool, batch_size=ENCODE_BATCH_SIZE,
                                              normalize_embeddings=True)
        finally:
            model.stop_multi_process_pool(pool)
    else:
        device = DEVICES[0]
        model = model.to(device)
        if device.startswith("cuda") and use_fp16:
            model = model.half()
        batch_size = ENCODE_BATCH_SIZE
        while True:
            try:
                emb = model.encode(texts, batch_size=batch_size, convert_to_numpy=True,
                                    show_progress_bar=True, normalize_embeddings=True, device=device)
                break
            except RuntimeError as e:
                if "out of memory" in str(e).lower() and batch_size > 1:
                    print(f"    [CUDA OOM] batch_size={batch_size} -> thử {batch_size // 2}")
                    torch.cuda.empty_cache()
                    batch_size = max(1, batch_size // 2)
                    continue
                raise
        ch["model"] = model
    if not np.isfinite(emb).all():
        raise SystemExit(f"[{ch['name']}] embedding corpus có NaN/inf — dừng thay vì retrieval rác.")
    ch["embeddings"] = emb
    encode_info[ch["name"]]["encode_min"] = round((time.time() - t0) / 60, 1)
    print(f"    -> {emb.shape}, {time.time()-t0:.0f}s")

checkpoint("Xong encode corpus (2 encoder)")


In [ ]:
# Cell 10: Bước 5b — Fine-tune reranker (nếu USE_RERANKER_FINETUNE, tái dùng `rows_clean`
# CHỈ nhãn citation của Bước 4 — KHÔNG dùng nhãn Task 1, xem giải thích ở Cell 8) RỒI tải
# 1 bản MỖI GPU để rerank song song thật ở Bước 6/7 (xem Cell 11)
#
# LÝ DO: kết quả dual-encoder thật (0.5215/0.4829) chỉ nhích rất ít so với single-encoder
# (0.5199/0.4806) dù retrieval mạnh hơn nhiều -> retrieval không còn là nút thắt chính,
# reranker ZERO-SHOT (AITeamVN/Vietnamese_Reranker, train trên Legal Zalo 2021 — KHÔNG phải
# đúng format "Điều X" của bài này) nhiều khả năng đang là trần chặn điểm tiếp theo.
#
# Huấn luyện bằng vòng lặp PyTorch thuần (KHÔNG dùng CrossEncoderTrainer của
# sentence-transformers) — tránh phụ thuộc API cross-encoder mới có thể không có ở mọi
# phiên bản cài qua Cell 1; margin ranking loss (điểm(positive) phải lớn hơn điểm(negative)
# ít nhất RERANKER_FT_MARGIN) — không cần thang điểm chuẩn hoá, chỉ cần đúng THỨ TỰ, ổn
# định hơn BCE/MSE cho một đầu hồi quy logit thô như model này.
from transformers import AutoModelForSequenceClassification, AutoTokenizer


def load_reranker_on(device: str, source: str):
    for attempt in range(2):
        try:
            print(f"  Đang tải reranker {source} lên {device}"
                  f"{' — thử lại lần 2' if attempt else ''}...")
            tok = AutoTokenizer.from_pretrained(source)
            mdl = AutoModelForSequenceClassification.from_pretrained(source)
            mdl = mdl.to(device)
            if device.startswith("cuda"):
                mdl = mdl.half()
            mdl.eval()
            return mdl, tok
        except Exception as e:
            if attempt == 0:
                print(f"  [Lần 1 lỗi: {e}] thử lại sau 5s...")
                time.sleep(5)
                continue
            print(f"  [CẢNH BÁO] Không tải được reranker trên {device} ({e}) -> bỏ qua "
                  f"reranker trên thẻ này.")
            return None, None


# finetune_reranker [v6_2] — CÔNG THỨC TASK 1 (eval/train_reranker.py + eval/job_family.sh).
#
# Soi code v6_1 thấy vòng lặp cũ LỆCH công thức Task 1 ở bốn chỗ, độc lập với bug chốt chặn
# false-negative đã sửa:
#   · lr 3e-6 áp thẳng lên model GỐC — Task 1 dùng 3e-6 để train TIẾP từ checkpoint vòng 2,
#     còn train từ gốc là 1e-5;
#   · không warmup, không lịch lr;
#   · batch 8 câu/forward -> OOM -> tụt về 1 câu/bước (log v6: 7.494 bước ở batch 1), tức
#     batch hiệu dụng 1 thay vì 8;
#   · không gradient checkpointing, không đóng băng embedding, không clip gradient.
# Bản dưới đây chép đúng các lựa chọn đó: 1 nhóm/forward, tích luỹ 8 nhóm/bước cập nhật,
# warmup 10% + giảm tuyến tính, clip 1,0, weight decay 0,01, max_length 512, listwise CE.
# Chỉ PHƯƠNG PHÁP được chuyển giao; dữ liệu vẫn là nhãn citation của Task 2 (rows_clean).
def finetune_reranker(rows, base_model: str, device: str, time_budget_sec: float, lr: float,
                       seed: int, n_neg: int, accum: int = 8, epochs: int = 2,
                       warmup: float = 0.1, max_length: int = 512, use_8bit_optim: bool = False,
                       log_every: int = 25):
    import torch.nn.functional as F
    cuda = device.startswith("cuda")
    tok = AutoTokenizer.from_pretrained(base_model)
    model = AutoModelForSequenceClassification.from_pretrained(base_model, num_labels=1).to(device)
    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tok.pad_token_id
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except TypeError:
        model.gradient_checkpointing_enable()
    model.config.use_cache = False
    for p_ in model.get_input_embeddings().parameters():
        p_.requires_grad = False
    params = [p_ for p_ in model.parameters() if p_.requires_grad]
    opt = None
    if use_8bit_optim:
        try:
            import bitsandbytes as bnb
            opt = bnb.optim.AdamW8bit(params, lr=lr, weight_decay=0.01)
        except Exception as e:
            print(f"  [reranker-ft] AdamW 8-bit không dùng được ({e}) -> AdamW thường")
    if opt is None:
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    scaler = torch.cuda.amp.GradScaler(enabled=cuda)

    g = random.Random(seed)
    total_groups = len(rows) * epochs
    planned_updates = max(1, total_groups // accum)
    CALIB_GROUPS = 40

    def _lr_at(u):
        w = max(1, int(planned_updates * warmup))
        if u < w:
            return lr * (u + 1) / w
        return lr * max(0.0, (planned_updates - u) / max(1, planned_updates - w))

    model.train()
    t0, it, updates, run_loss, run_acc, seen = time.time(), 0, 0, 0.0, 0.0, 0
    stop_reason = "hết số epoch dự kiến"
    loss_log = []
    for ep in range(epochs):
        order = list(range(len(rows)))
        g.shuffle(order)
        for j in order:
            r = rows[j]
            texts = [r["positive"]] + [r[f"negative_{k+1}"] for k in range(n_neg)]
            enc = tok([[r["anchor"], t] for t in texts], padding=True, truncation=True,
                      max_length=max_length, return_tensors="pt").to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=cuda):
                logits = model(**enc).logits.view(1, -1)
            logits = logits.float()
            loss = F.cross_entropy(logits, torch.zeros(1, dtype=torch.long, device=device))
            scaler.scale(loss / accum).backward()
            run_loss += loss.item()
            run_acc += float(int(torch.argmax(logits, dim=1).item() == 0))
            seen += 1
            it += 1
            if it == CALIB_GROUPS:
                # Hiệu chỉnh số bước theo tốc độ ĐO THẬT (cùng ý với worker encoder ở Cell 8):
                # lịch lr phải kết thúc đúng lúc hết ngân sách, không phải bị cắt giữa chừng
                # khi lr còn cao.
                rate = (time.time() - t0) / it
                fit_groups = int(0.95 * time_budget_sec / max(rate, 1e-6))
                planned_updates = max(1, min(total_groups, fit_groups) // accum)
                print(f"    [reranker-ft] {rate:.2f} giây/nhóm -> {planned_updates} bước cập nhật "
                      f"(tối đa {total_groups // accum} nếu đủ {epochs} epoch)", flush=True)
            if it % accum == 0:
                for pg in opt.param_groups:
                    pg["lr"] = _lr_at(updates)
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                updates += 1
                if updates % log_every == 0:
                    loss_log.append({"update": updates, "loss": round(run_loss / seen, 4),
                                     "acc_in_group": round(run_acc / seen, 4),
                                     "min": round((time.time() - t0) / 60, 1)})
                    print(f"    reranker-ft ep {ep} bước {updates}/{planned_updates} · "
                          f"loss {run_loss/seen:.4f} · acc@1 trong nhóm {run_acc/seen:.3f} · "
                          f"lr {_lr_at(updates):.2e} · {(time.time()-t0)/60:.1f} phút", flush=True)
                    run_loss, run_acc, seen = 0.0, 0.0, 0
                if updates >= planned_updates:
                    stop_reason = "đủ số bước theo ngân sách"
                    break
                if time.time() - t0 > time_budget_sec:
                    stop_reason = "chạm trần thời gian"
                    break
        else:
            continue
        break
    opt.zero_grad(set_to_none=True)
    model.eval()
    meta = {"updates": updates, "groups_seen": it, "planned_updates": planned_updates,
            "elapsed_s": round(time.time() - t0, 1), "stop_reason": stop_reason,
            "lr": lr, "accum": accum, "max_length": max_length, "n_neg": n_neg,
            "n_rows": len(rows), "loss_log": loss_log}
    return model, tok, meta


# =============================================================================
# ĐÀO NEGATIVE "CÙNG HỌ" — chuyển giao PHƯƠNG PHÁP từ Task 1 (LegalIR)
# =============================================================================
# ⚖️ Vì sao được phép: rule.md §7c(b) cấm dùng **dữ liệu** (cặp câu hỏi–gold) của Task 1
#    để huấn luyện Task 2, và cấm cả checkpoint fine-tune bằng nhãn Task 1. Nó KHÔNG cấm
#    thuật toán. Ở đây ta chỉ mượn CÁCH ĐÀO NEGATIVE, còn nhãn thì lấy từ citation trong
#    answer của chính Task 2, model xuất phát là bản pretrain công khai. Không có một bit
#    dữ liệu Task 1 nào đi vào.
#
# Task 1 đo được (result.md §4): negative "cùng họ" 0,9428 · negative ngữ nghĩa 0,9360 ·
# negative hợp nhất 0,9357. Càng nhiều negative ngữ nghĩa điểm càng thấp. "Cùng họ" =
# văn bản cùng chủ đề/cùng series nhưng KHÔNG phải gold — negative khó nhất có thể có.
#
# Bản dịch sang Task 2: đơn vị ứng viên là ĐIỀU, nên "cùng họ" tự nhiên nhất là
# **các Điều ANH EM trong CÙNG văn bản gold**. Đó đúng là chỗ pipeline đang thua: đo trên
# 501 câu dev, reranker chọn đúng Điều tốt nhất trong 5 ứng viên chỉ 61,3% số lần, và
# khoảng cách điểm giữa hạng 1 và hạng 2 có trung vị 0,0023 — nó gần như không phân biệt
# nổi hai ứng viên đầu. Dạy đúng vào chỗ đó là dạy đúng chỗ đau.
#
# CHỐT CHẶN BẮT BUỘC (Task 1 gọi là "bỏ ứng viên CE chấm cao hơn gold"): văn bản/Điều cùng
# họ RẤT dễ là false negative — nó cũng trả lời được câu hỏi mà nhãn không ghi. Nhãn của
# Task 2 lại lấy citation ĐẦU TIÊN gặp trong answer, càng dễ trượt. Nếu không lọc, ta đang
# dạy model DÌM đáp án đúng xuống. Đây nhiều khả năng là nguyên nhân recall@1 sập từ 0,6503
# (không rerank) xuống 0,4266 (rerank fine-tune) trong log ver9.
def mine_family_negatives(rows, all_chunks, chunk_by_id, bm25, base_model, device,
                           n_neg=7, n_sibling=12, gate=True):
    """rows -> rows mới với negative_1..n_neg là Điều 'cùng họ' đã qua chốt chặn."""
    from collections import defaultdict
    import torch
    by_doc = defaultdict(list)
    for c in all_chunks:
        by_doc[str(c["id"]).split("_")[0]].append(c["id"])

    cand_of = {}
    for idx, r in enumerate(rows):
        pos_id = r.get("_pos_id")
        if not pos_id:
            continue
        doc = str(pos_id).split("_")[0]
        sib = [cid for cid in by_doc.get(doc, []) if cid != pos_id][:n_sibling]
        # Đổ thêm bằng BM25 để đủ số lượng khi văn bản gold chỉ có 1-2 Điều.
        if len(sib) < n_neg * 2:
            ranked = bm25.top_k(tokenize_simple(r["anchor"]), 60)
            extra = [all_chunks[i]["id"] for i in ranked
                      if all_chunks[i]["id"] != pos_id and all_chunks[i]["id"] not in sib]
            sib = sib + extra[: n_neg * 2 - len(sib)]
        if sib:
            cand_of[idx] = sib

    if not cand_of:
        print("  [mine] không đào được ứng viên nào -> giữ nguyên negative cũ")
        return rows

    kept = 0
    if gate:
        # Chấm bằng reranker ZERO-SHOT (chưa fine-tune) — đúng tinh thần Task 1: dùng một
        # bộ chấm ĐỘC LẬP với model sắp train để phát hiện positive chưa gán nhãn.
        # ⛔ BUG v6 ĐÃ SỬA Ở ĐÂY (v6_log.txt dòng 82-92). Bản v6 viết:
        #       load_reranker_on(device, "zero-shot")
        # tức truyền CHUỖI MÔ TẢ "zero-shot" vào tham số đợi TÊN MODEL. HuggingFace không có
        # repo nào tên "zero-shot" -> load_reranker_on trả (None, None) -> nhánh dưới in
        # "bỏ chốt chặn, RỦI RO CAO" -> toàn bộ negative đi thẳng vào train KHÔNG qua lọc
        # false-negative. Tham số `base_model` đã nằm sẵn trong chữ ký hàm và chính là thứ
        # cần truyền — lỗi chỉ là không dùng nó.
        #
        # Hậu quả đã đo được: reranker fine-tune trên negative bẩn thua zero-shot 3,3 điểm
        # METEOR (0,5382 vs 0,5709). Đúng cơ chế result.md §8.3 cảnh báo: không lọc tức là
        # dạy model dìm chính đáp án đúng xuống.
        model, tok = load_reranker_on(device, base_model)
        if model is None:
            print("  [mine] không tải được reranker để lọc -> BỎ HẲN fine-tune thay vì train "
                  "trên negative chưa lọc (bản v6 đi tiếp và mất 110 phút cho một checkpoint "
                  "thua zero-shot).")
            raise RuntimeError("không tải được reranker cho chốt chặn false-negative")
        else:
            pairs, index = [], []
            for idx, sib in cand_of.items():
                pairs.append([rows[idx]["anchor"], rows[idx]["positive"]])
                index.append((idx, None))
                for cid in sib:
                    pairs.append([rows[idx]["anchor"], chunk_by_id[cid]["text"]])
                    index.append((idx, cid))
            print(f"  [mine] chấm {len(pairs)} cặp để lọc false negative ...")
            scores = []
            with torch.no_grad():
                for i in range(0, len(pairs), RERANK_SUBBATCH):
                    enc = tok(pairs[i:i + RERANK_SUBBATCH], padding=True, truncation=True,
                              max_length=512, return_tensors="pt").to(device)
                    scores.extend(model(**enc).logits.view(-1).float().cpu().tolist())
            pos_score, cand_score = {}, defaultdict(list)
            for (idx, cid), sc in zip(index, scores):
                if cid is None:
                    pos_score[idx] = sc
                else:
                    cand_score[idx].append((sc, cid))
            for idx in list(cand_of):
                survive = [(sc, cid) for sc, cid in cand_score.get(idx, [])
                            if sc < pos_score.get(idx, float("inf"))]
                survive.sort(reverse=True)          # khó nhất trước
                cand_of[idx] = [cid for _sc, cid in survive]
                kept += len(survive)
            del model
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass

    out, n_drop = [], 0
    for idx, r in enumerate(rows):
        sib = cand_of.get(idx, [])
        if len(sib) < n_neg:
            # Không đủ negative sau khi lọc -> giữ row nhưng đệm bằng negative cũ nếu có.
            old = [r[k] for k in r if k.startswith("negative_")]
            texts = [chunk_by_id[c]["text"] for c in sib] + old
        else:
            texts = [chunk_by_id[c]["text"] for c in sib[:n_neg]]
        if len(texts) < n_neg:
            n_drop += 1
            continue
        r2 = {"anchor": r["anchor"], "positive": r["positive"]}
        for k, t in enumerate(texts[:n_neg]):
            r2[f"negative_{k+1}"] = t
        out.append(r2)
    print(f"  [mine] {len(out)}/{len(rows)} nhóm · {n_neg} negative/nhóm"
          + (f" · giữ lại {kept} ứng viên sau chốt chặn" if gate else " · KHÔNG lọc")
          + (f" · bỏ {n_drop} nhóm thiếu negative" if n_drop else ""))
    return out



# =============================================================================
# v6_2 — OPTION 1: FINE-TUNE RERANKER, NHƯNG GIỮ CẢ HAI BẢN ĐỂ SO TRONG CÙNG PHIÊN
# =============================================================================
# v6_1 ghi: "bản fine-tune THAY THẾ zero-shot, muốn so thì chạy hai lượt". Hai lượt Kaggle
# khác nhau không phải phép so paired — khác cả encoder fine-tune, khác cả máy. v6_2 nạp CẢ
# HAI bản lên mỗi GPU (568M fp16 ≈ 1,1GB/bản, T4 dư chỗ) rồi để Cell 12 chấm cả hai trên
# CÙNG câu dev, CÙNG ứng viên tầng 1, qua cổng split-half. Bản thắng mới đi vào bài nộp; bản
# thua bị giải phóng, nên pipeline nộp chỉ có một reranker (hoặc hai nếu nhánh "ft2" thắng —
# vẫn 2,3B < 4B, xem rule.md §2.1).
#
# Thứ tự trong cell: fine-tune TRƯỚC (cần gần trọn một thẻ T4), rồi mới nạp các bản suy luận.
# Encoder dense đang nằm trên thẻ fine-tune được dời tạm sang CPU để nhường VRAM.
print("=== Bước 5b [v6_2]: Fine-tune reranker + nạp zero-shot VÀ fine-tune ===")
reranker_finetune_info = {"used": False, "reason": None, "meta": None, "ckpt": None,
                          "mine_elapsed_s": None}
reranker_models_ft, reranker_tokenizers_ft = {}, {}
_ft_ckpt = os.path.join(CHECKPOINT_DIR, "reranker-ft")
# Ngân sách tối thiểu còn lại SAU fine-tune: chấm nhiều nhánh trên dev + harvest LTR + Bước 7.
_need_after_ft = PUBLIC_RESERVE_SEC + 90 * 60

if USE_RERANKER_FINETUNE and rows_clean and remaining() > RERANKER_FT_TIME_BUDGET_SEC + 15 * 60 + _need_after_ft:
    ft_dev = DEVICES[0]
    moved = []
    try:
        for ch in DENSE_CHANNELS:
            if str(next(ch["model"].parameters()).device) == ft_dev:
                ch["model"].to("cpu")
                moved.append(ch)
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        t_mine = time.time()
        rerank_rows = mine_family_negatives(
            rows_clean, all_chunks, chunk_by_id, bm25, RERANKER_BASE, ft_dev,
            n_neg=N_NEG_RERANK, gate=RERANK_FALSE_NEG_GATE)
        reranker_finetune_info["mine_elapsed_s"] = round(time.time() - t_mine, 1)
        budget = min(RERANKER_FT_TIME_BUDGET_SEC, remaining() - _need_after_ft - 5 * 60)
        print(f"  [reranker-ft] {len(rerank_rows)} nhóm · ngân sách {budget/60:.0f} phút")
        m_ft, t_ft, meta = finetune_reranker(
            rerank_rows, RERANKER_BASE, ft_dev, budget, RERANKER_FT_LR, SEED, N_NEG_RERANK,
            accum=RERANKER_FT_ACCUM, epochs=RERANKER_FT_EPOCHS, warmup=RERANKER_FT_WARMUP,
            max_length=RERANKER_FT_MAX_LEN, use_8bit_optim=USE_8BIT_OPTIM)
        m_ft.half().save_pretrained(_ft_ckpt)
        t_ft.save_pretrained(_ft_ckpt)
        del m_ft
        reranker_finetune_info.update({"used": True, "meta": meta, "ckpt": _ft_ckpt})
        print(f"  [reranker-ft] xong: {meta['updates']} bước, {meta['elapsed_s']/60:.1f} phút, "
              f"dừng vì {meta['stop_reason']} -> {_ft_ckpt}")
    except Exception as e:
        print(f"  [reranker-ft] BỎ QUA ({type(e).__name__}: {e}) -> chỉ còn nhánh zero-shot.")
        reranker_finetune_info["reason"] = f"{type(e).__name__}: {e}"
    finally:
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        for ch in moved:
            ch["model"].to(ft_dev)
else:
    reranker_finetune_info["reason"] = (
        "USE_RERANKER_FINETUNE=False" if not USE_RERANKER_FINETUNE
        else ("không có rows_clean" if not rows_clean else
              f"không đủ thời gian (còn {remaining()/60:.0f} phút)"))
    print(f"  {reranker_finetune_info['reason']} -> chỉ nhánh zero-shot.")

reranker_models, reranker_tokenizers = {}, {}          # ZERO-SHOT — luôn có
for dev in DEVICES:
    m, t = load_reranker_on(dev, RERANKER_BASE)
    if m is not None:
        reranker_models[dev] = m
        reranker_tokenizers[dev] = t

if reranker_finetune_info["used"]:
    for dev in reranker_models:
        m, t = load_reranker_on(dev, _ft_ckpt)
        if m is None:
            print(f"  [reranker-ft] không nạp được bản fine-tune lên {dev} -> bỏ nhánh fine-tune")
            reranker_models_ft, reranker_tokenizers_ft = {}, {}
            reranker_finetune_info["reason"] = f"không nạp lại được {_ft_ckpt} trên {dev}"
            break
        reranker_models_ft[dev] = m
        reranker_tokenizers_ft[dev] = t

HAS_RERANKER = len(reranker_models) > 0
HAS_FT = bool(reranker_models_ft) and set(reranker_models_ft) == set(reranker_models)
RERANK_DEVICES = list(reranker_models.keys())
RERANKER_SOURCE = "zeroshot"        # Cell 12 cập nhật sau cổng so nhánh
print(f"  Zero-shot trên: {RERANK_DEVICES or '(không tải được)'} · "
      f"fine-tune: {list(reranker_models_ft) if HAS_FT else '(không có)'}")
if not HAS_RERANKER:
    print("  ⛔ KHÔNG có reranker nào -> pipeline sẽ chạy bằng thứ hạng RRF thuần. "
          "Điểm sẽ thấp hơn hẳn (Recall@1 0,154 so với 0,580 — đo từ lượt trước).")
checkpoint("Xong fine-tune + nạp reranker")


In [ ]:
# Cell 11 [v6_2]: Hàm retrieval (RRF fusion N kênh — BM25 + N encoder) + rerank theo lô
# + hạ tầng chạy song song 2 GPU
#
# v6_2 tách rrf_retrieve thành hai nửa (xếp hạng từng kênh / trộn RRF) và thêm bộ nhớ đệm
# điểm CE theo id ứng viên. Mục đích: Cell 12 so NHIỀU cấu hình trên cùng câu dev (reranker
# zero-shot vs fine-tune, bỏ bớt kênh encoder) mà không phải chấm lại những cặp
# (câu hỏi, ứng viên) đã chấm. Khi không truyền mask/memo/rank_lists, kết quả TRÙNG v6_1:
# cùng thứ tự dựng tập, cùng công thức RRF, cùng thứ tự sort.
from concurrent.futures import ThreadPoolExecutor

_print_lock = __import__("threading").Lock()

CHANNEL_NAMES = ["bm25"] + [ch["name"] for ch in DENSE_CHANNELS]


def rrf_rank_lists(question: str, bm25, dense_channels, top_k: int = TOP_K_RETRIEVE):
    """-> [xếp hạng BM25, xếp hạng kênh dense 1, ...] — mỗi phần tử là list index chunk."""
    lists = [list(bm25.top_k(tokenize_simple(question), top_k))]
    for ch in dense_channels:
        q_text = ch["query_prefix"] + question if ch["query_prefix"] else question
        q_emb = ch["model"].encode([q_text], convert_to_numpy=True, normalize_embeddings=True)[0]
        scores = ch["embeddings"] @ q_emb
        lists.append(list(np.argsort(-scores)[:top_k]))
    return lists


def rrf_fuse(rank_lists, all_chunks, mask=None, top_k: int = TOP_K_RETRIEVE):
    """Trộn RRF các kênh có mask=True (None = mọi kênh)."""
    used = [rl for i, rl in enumerate(rank_lists) if mask is None or mask[i]]
    if not used:
        return []
    rank_maps = [{idx: r for r, idx in enumerate(rl)} for rl in used]
    all_idx = set(used[0])
    for rl in used[1:]:
        all_idx |= set(rl)
    rrf = {i: sum(1 / (60 + rm.get(i, top_k + 1)) for rm in rank_maps) for i in all_idx}
    ranked = sorted(rrf, key=rrf.get, reverse=True)
    return [all_chunks[i] for i in ranked]


def rrf_retrieve(question: str, bm25, dense_channels, all_chunks, top_k: int = TOP_K_RETRIEVE):
    return rrf_fuse(rrf_rank_lists(question, bm25, dense_channels, top_k), all_chunks, None, top_k)


def rerank(question: str, candidates: list, reranker_model, reranker_tokenizer,
           max_candidates: int = TOP_K_RETRIEVE, max_length: int = 1024,
           sub_batch: int = RERANK_SUBBATCH, memo: dict = None):
    """Chấm lại top `max_candidates` bằng cross-encoder theo lô nhỏ. `memo` = {id: điểm} của
    ĐÚNG model này cho ĐÚNG câu hỏi này — id đã có điểm thì không chấm lại."""
    if reranker_model is None or not candidates:
        return candidates, None
    subset = candidates[:max_candidates]
    scores = np.empty(len(subset), dtype=np.float32)
    todo = []
    for j, c in enumerate(subset):
        if memo is not None and c["id"] in memo:
            scores[j] = memo[c["id"]]
        else:
            todo.append(j)
    if todo:
        device = next(reranker_model.parameters()).device
        pairs = [[question, subset[j]["text"]] for j in todo]
        out_all = np.empty(len(pairs), dtype=np.float32)
        bs, i = max(1, sub_batch), 0
        while i < len(pairs):
            batch = pairs[i:i + bs]
            try:
                with torch.no_grad():
                    inputs = reranker_tokenizer(batch, padding=True, truncation=True,
                                                 return_tensors="pt", max_length=max_length).to(device)
                    out = reranker_model(**inputs, return_dict=True).logits.view(-1).float().cpu().numpy()
                out_all[i:i + len(batch)] = out
                i += bs
            except RuntimeError as e:
                if "out of memory" in str(e).lower() and bs > 1:
                    torch.cuda.empty_cache()
                    bs = max(1, bs // 2)
                    continue
                if "out of memory" in str(e).lower():
                    return candidates, None
                raise
        for j, s in zip(todo, out_all):
            scores[j] = s
            if memo is not None:
                memo[subset[j]["id"]] = float(s)
    order = np.argsort(-scores)
    reranked = [subset[i2] for i2 in order]
    sorted_scores = scores[order]
    return reranked + candidates[max_candidates:], sorted_scores


def adaptive_k_cutoff(scores, min_k: int = 1, max_k: int = TOP_K_RERANK, search_window: int = 15) -> int:
    """Adaptive-k (Taguchi et al. 2025, arXiv:2506.08479): tìm điểm "gãy" tự nhiên trong
    phân phối điểm reranker đã sort giảm dần thay vì luôn cắt ở top_n cố định."""
    if scores is None or len(scores) == 0:
        return min_k
    n = min(len(scores), search_window)
    if n <= 1:
        return min_k
    gaps = [scores[i] - scores[i + 1] for i in range(n - 1)]
    k_star = int(np.argmax(gaps)) + 1
    return max(min_k, min(k_star, max_k))


_TITLE_MAX_WORDS = 30


def _dieu_title(text: str) -> str:
    """Dòng tiêu đề của Điều ('Điều 17. Vi phạm quy định chung về ...' -> 'vi phạm quy định
    chung về ...'). Rỗng nếu dòng đầu quá dài (thân Điều dính liền, không có tiêu đề riêng)."""
    body = _DIEU_PREFIX_STRIP_RE.sub("", text, count=1)
    first = body.split("\n", 1)[0].strip().rstrip(".:;").strip()
    if not first or len(first.split()) > _TITLE_MAX_WORDS:
        return ""
    return first[0].lower() + first[1:]


def _dieu_lead(ref: str, text: str, template: str) -> str:
    """Câu dẫn cho đơn vị Điều theo biến thể template (Option 2, v6_2).
    Đo trước 0 GPU trên 3.427 Điều gold (METEOR thật, Δ so với T0, hai nửa cùng dấu):
      T1_theo_qd_tai +0,0007 · T2_title +0,0029 · T3_theo_qd_tai_title +0,0040.
    Cell 12 đo lại trên ứng viên pipeline THẬT chọn (không phải Điều gold) trước khi dùng."""
    if template == "T0_current":
        return f"Căn cứ {ref} quy định như sau:"
    title = _dieu_title(text) if template in ("T2_title", "T3_theo_qd_tai_title") else ""
    if template == "T1_theo_qd_tai":
        return f"Căn cứ theo quy định tại {ref} như sau:"
    if template == "T2_title":
        return f"Căn cứ {ref} quy định về {title} như sau:" if title else f"Căn cứ {ref} quy định như sau:"
    if template == "T3_theo_qd_tai_title":
        return (f"Căn cứ theo quy định tại {ref} quy định về {title} như sau:" if title
                else f"Căn cứ theo quy định tại {ref} như sau:")
    raise ValueError(f"template không hợp lệ: {template}")


def render_answer(selected_chunks: list, top_n: int, question: str = "",
                   concl: str = CONCL, template: str = None) -> str:
    """Câu dẫn "Căn cứ Điều X <loại VB> <số hiệu> quy định như sau:" — khuôn phổ biến nhất
    đo được trên answer thật (57.4% mở đầu "Căn cứ", 24.6% có "quy định như sau"). Cắt bỏ
    "Điều X." lặp lại ở đầu thân bài (98.8% answer thật không lặp).

    CÂU KẾT (`concl`) — thay đổi ĐÁNG GIÁ NHẤT và rẻ nhất trong cả pipeline. Đo trên
    501 câu dev, cùng retrieval, chỉ đổi một biến:

        concl=none    METEOR 0,5151
        concl=echo    METEOR 0,5499    Δ +0,0348 ± 0,0017 · 416 thắng / 85 thua · t = 20,4
        concl=echo2   METEOR 0,5630    Δ +0,0131 ± 0,0010 · 371 thắng / 130 thua · t = 12,8

    Split-half (chia đôi dev, chọn trên nửa này đo nửa kia): CẢ HAI nửa độc lập đều chọn
    echo2 (A 0,5675 · B 0,5589) — nên đây không phải ảo giác đỉnh-trên-toàn-dev.

    Vì sao ăn điểm: METEOR có alpha = 0,9 nên nặng recall, và 36,2% đáp án thật chứa
    "Như vậy", 27,5% chứa "Theo đó" — chúng nhắc lại nội dung câu hỏi ở phần kết. Lặp
    lại câu hỏi làm khớp đúng nhóm token đó.

    Đã dò tiếp số lần lặp: 1× 0,5499 · 2× 0,5630 · 3× 0,5674 · 4× 0,5682 · 6× 0,5661.
    Có đỉnh thật quanh 4, NHƯNG dừng ở 2: từ 2 lên 4 chỉ được +0,5 điểm (đúng vùng mà
    "chọn đỉnh trên toàn dev" đã lừa project này ba lần), còn đáp án lặp câu hỏi bốn lần
    thì nhìn bằng mắt là hỏng rõ ràng. Đây là tối ưu HÌNH DẠNG ĐỘ ĐO, hợp lệ theo luật
    nhưng không làm câu trả lời tốt hơn cho người đọc — biết để không đi xa hơn một cách
    mù quáng."""
    # BẢN SỬA v5 (Batch 1 A1): câu dẫn dựng theo `unit_type` thay vì chỉ nhánh "Điều/khác" —
    # unit_type mới ("muc"/"phu_luc"/"tiet" và biến thể "*_capped") đến từ chunk_passage()
    # (Cell 5) khi văn bản không có Điều nhưng có cấu trúc thay thế. "raw"/"raw450" (không
    # khớp tầng bậc nào) vẫn dùng câu dẫn cũ "Căn cứ {loai_vb} {so_hieu}".
    parts, seen = [], set()
    for c in selected_chunks:
        if c["id"] in seen or len(parts) >= top_n:
            continue
        seen.add(c["id"])
        loai_vb = c["loai_vb"] or "văn bản"
        so_hieu = c["so_hieu"] or ""
        dieu = c["dieu_so"]
        unit_type = c.get("unit_type", "dieu" if dieu != "0" else "raw")
        unit_no = c.get("unit_no", "")
        base_unit_type = unit_type[:-len("_capped")] if unit_type.endswith("_capped") else unit_type
        if base_unit_type == "dieu":
            lead = _dieu_lead(f"Điều {dieu} {loai_vb} {so_hieu}", c["text"],
                              template or ANSWER_TEMPLATE)
            body = _DIEU_PREFIX_STRIP_RE.sub("", c["text"], count=1)
        elif base_unit_type == "muc" and unit_no:
            lead = f"Căn cứ Mục {unit_no} {loai_vb} {so_hieu} quy định như sau:"
            body = _MUC_PREFIX_STRIP_RE.sub("", c["text"], count=1)
        elif base_unit_type == "phu_luc" and unit_no:
            lead = f"Căn cứ Phụ lục {unit_no} {loai_vb} {so_hieu} quy định như sau:"
            body = _PHU_LUC_PREFIX_STRIP_RE.sub("", c["text"], count=1)
        elif base_unit_type == "tiet" and unit_no:
            lead = f"Căn cứ tiết {unit_no} {loai_vb} {so_hieu} quy định như sau:"
            body = _TIET_PREFIX_STRIP_RE.sub("", c["text"], count=1)
        else:
            lead = f"Căn cứ {loai_vb} {so_hieu} quy định như sau:"
            body = c["text"]
        parts.append(f"{lead}\n{body}")
    ans = "\n\n".join(parts)
    if concl != "none" and question:
        q = question.strip().rstrip("?").strip()
        if q:
            ql = q[0].lower() + q[1:]
            if concl == "echo":
                ans += f"\nNhư vậy, theo quy định nêu trên thì {ql}."
            elif concl == "echo2":
                ans += f"\nTheo đó, {ql}.\nNhư vậy, theo quy định nêu trên thì {ql}."
    return ans



def _score_docs(candidates: list, scores, mode: str = "max", T: float = 1.0) -> dict:
    """{doc_id: điểm gộp}. Tách riêng khỏi aggregate_docs (BẢN SỬA v4) để dùng lại được ở
    TẦNG 1 (chọn top-K văn bản trong retrieve_two_tier) — trước v4 phép gộp này chỉ áp
    dụng được sau khi ĐÃ có danh sách Điều phẳng của một tầng duy nhất."""
    by_doc = {}
    for c, sc in zip(candidates, scores):
        by_doc.setdefault(str(c["id"]).split("_")[0], []).append(float(sc))
    if mode == "max":
        return {d: max(v) for d, v in by_doc.items()}
    doc_score = {}
    for d, v in by_doc.items():
        arr = np.array(v, dtype=np.float64) / max(T, 1e-6)
        doc_score[d] = float(T * (arr.max() + np.log(np.exp(arr - arr.max()).sum())))
    return doc_score


# v6_1 — ĐÃ XOÁ aggregate_docs(). Nó chỉ tồn tại để phục vụ AGG_MODE="lse", mà LSE đã đo âm
# hai lần độc lập và bị ép về "max" ở Cell 2 — tức hàm này chỉ còn là một no-op có 30 dòng
# docstring giải thích một hướng đã đóng. _score_docs() bên trên GIỮ LẠI vì retrieve_two_tier
# vẫn dùng nó để gộp điểm chọn top-K văn bản ở tầng 1 (luôn mode="max").

def retrieve_two_tier(question: str, bm25_t1, dense_channels, all_chunks_t1, dieu_by_doc,
                       reranker_model=None, reranker_tokenizer=None, doc_k: int = DOC_K,
                       t2_model=None, t2_tokenizer=None, mask=None, rank_lists=None,
                       memo_t1: dict = None, memo_t2: dict = None, diag: dict = None):
    """HAI TẦNG (result.md §3): tầng 1 truy xuất trên corpus 450 TỪ để chọn doc_k VĂN BẢN,
    tầng 2 gom Điều CHỈ TRONG các văn bản đó và rerank lại.

    v6_2 — tham số mới, đều tuỳ chọn (bỏ trống = hành vi v6_1):
      t2_model/t2_tokenizer  reranker riêng cho tầng 2 (nhánh "ft2": zero-shot chọn văn bản,
                             fine-tune chọn Điều — bản fine-tune chỉ học trên đơn vị Điều)
      mask                   kênh nào tham gia RRF tầng 1, theo CHANNEL_NAMES (Option 3)
      rank_lists             xếp hạng từng kênh đã tính sẵn (khỏi encode lại câu hỏi)
      memo_t1/memo_t2        {id: điểm CE} đã chấm — tránh chấm lại
      diag                   dict nhận top_docs để đo recall văn bản tầng 1
    """
    if rank_lists is None:
        rank_lists = rrf_rank_lists(question, bm25_t1, dense_channels)
    t1_ranked = rrf_fuse(rank_lists, all_chunks_t1, mask)
    if not t1_ranked:
        return [], None
    t1_scores = None
    if reranker_model is not None:
        t1_ranked, t1_scores = rerank(question, t1_ranked, reranker_model, reranker_tokenizer,
                                      memo=memo_t1)
    if t1_scores is None:
        top_docs = list(dict.fromkeys(str(c["id"]).split("_")[0] for c in t1_ranked))[:doc_k]
    else:
        doc_score = _score_docs(t1_ranked, t1_scores, "max", 1.0)
        top_docs = sorted(doc_score, key=doc_score.get, reverse=True)[:doc_k]
    if diag is not None:
        diag["top_docs"] = list(top_docs)

    cand = []
    for d in top_docs:
        cand.extend(dieu_by_doc.get(d, []))
    if len(cand) > MAX_DIEU_CANDIDATES:
        qt = set(tokenize_simple(question))
        cand = sorted(cand, key=lambda c: -len(qt & set(tokenize_simple(c["text"]))))[:MAX_DIEU_CANDIDATES]
    if not cand:
        return [], None
    m2, k2 = (t2_model, t2_tokenizer) if t2_model is not None else (reranker_model, reranker_tokenizer)
    if m2 is None:
        return cand, None
    return rerank(question, cand, m2, k2, max_candidates=len(cand), memo=memo_t2)


# v6_1 — ĐÃ XOÁ answer_question(). Bước 7 dùng answer_question_v61() ở Cell 14, vốn phải
# thêm bước LTR resort. Giữ cả hai là để hai đường dẫn sinh đáp án gần-giống-nhau cùng tồn
# tại trong một notebook — đúng loại bẫy đã làm result.md §4.3 đọc sai log một lần.

def split_evenly(lst, n):
    k, m = divmod(len(lst), n)
    return [lst[i * k + min(i, m):(i + 1) * k + min(i + 1, m)] for i in range(n)]


def parallel_process(ids, worker_fn, devices, label: str = "", progress_every: int = 50):
    """Chia `ids` đều cho từng thiết bị trong `devices`, chạy worker_fn(ids_chunk, device)
    ĐỒNG THỜI trên các luồng riêng. PyTorch giải phóng GIL trong lúc chờ CUDA hoàn thành nên
    2 luồng ghim vào 2 GPU vật lý khác nhau chạy song song THẬT (không phải giả song song do
    GIL) — đây là chỗ mang lại tốc độ x~2 cho phần rerank ở Bước 6/7.
    worker_fn(ids_chunk, device, worker_idx) -> dict {qid: kết quả}."""
    ids = list(ids)
    devices = list(devices) if devices else ["cpu"]
    chunks = split_evenly(ids, len(devices))
    results = {}
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=len(devices)) as ex:
        futures = [ex.submit(worker_fn, chunk, dev, i) for i, (chunk, dev) in enumerate(zip(chunks, devices))]
        for f in futures:
            results.update(f.result())
    print(f"    [{label}] {len(ids)} câu / {len(devices)} thiết bị song song -> {time.time()-t0:.0f}s")
    return results


def _progress_print(label, worker_idx, i, n):
    if (i + 1) % 50 == 0 or (i + 1) == n:
        with _print_lock:
            print(f"    [{label} · luồng {worker_idx}] {i+1}/{n}")

# =============================================================================
# v6_1 — ĐẶC TRƯNG CHO BỘ CHỌN ĐIỀU HỌC ĐƯỢC (LTR)
# =============================================================================
# Mục tiêu: bắt phần dư địa +7,7 điểm ở result.md §7 — Điều tốt nhất ĐÃ nằm trong 5 ứng viên
# reranker đưa ra, chỉ bị xếp sai thứ tự (reranker chọn đúng 61,3%, khoảng cách điểm hạng 1
# vs hạng 2 trung vị 0,0023).
#
# NGUYÊN TẮC CHỌN ĐẶC TRƯNG — không đặc trưng nào được nhìn thấy gold:
#   · mọi thứ tính từ (câu hỏi, văn bản ứng viên) hoặc từ chính phân phối điểm CE;
#   · KHÔNG dùng độ dài đáp án thật, KHÔNG dùng nhãn citation, KHÔNG dùng METEOR.
# METEOR chỉ xuất hiện ở phía NHÃN lúc train (Cell 13), không bao giờ ở phía đặc trưng.
#
# Vì sao các đặc trưng "hình dạng phân phối điểm" (z-score, gap) quan trọng hơn điểm thô:
# result.md §7 đo được điểm CE tuyệt đối gần như không phân biệt nổi hạng 1 với hạng 2. Cái
# còn mang tin là điểm đó nằm ở đâu TRONG NHÓM — một điểm 8,0 giữa nhóm toàn 7,9 nói lên
# điều khác hẳn một điểm 8,0 giữa nhóm toàn 2,0.
_Q_SO_HIEU_RE = re.compile(r"\d+\s*/\s*\d{4}\s*/\s*[A-ZĐ\-]+", re.IGNORECASE)
_Q_DIEU_RE = re.compile(r"[Đđ]iều\s+(\d+)")

LTR_FEATURE_NAMES = [
    "ce_score", "ce_rank", "ce_gap_to_top", "ce_gap_to_next", "ce_z_in_group",
    "n_words", "q_overlap", "q_overlap_ratio", "so_hieu_match", "dieu_no_match",
    "doc_rank", "is_first_in_doc", "unit_is_dieu",
]


def extract_ltr_features(question: str, ranked: list, scores, top_k: int = LTR_TOP_K_CANDIDATES):
    """-> (X, cands) với X là list vector đặc trưng song song với cands (top_k ứng viên đầu).

    Trả về list rỗng nếu không có điểm CE — LTR chỉ có nghĩa khi đã có một thứ hạng để sửa.
    """
    if scores is None or not ranked:
        return [], []
    cands = ranked[:top_k]
    sc = np.asarray(scores[:top_k], dtype=np.float64)
    if len(sc) == 0:
        return [], []
    top, mean, std = float(sc[0]), float(sc.mean()), float(sc.std())
    q_tokens = set(tokenize_simple(question))
    q_so_hieu = {norm_so_hieu(m.group(0)) for m in _Q_SO_HIEU_RE.finditer(question)}
    q_dieu = {m.group(1) for m in _Q_DIEU_RE.finditer(question)}
    # Thứ hạng VĂN BẢN theo thứ tự xuất hiện đầu tiên trong danh sách đã rerank.
    doc_order, seen_doc = {}, 0
    for c in cands:
        d = str(c["id"]).split("_")[0]
        if d not in doc_order:
            doc_order[d] = seen_doc
            seen_doc += 1
    X = []
    for i, c in enumerate(cands):
        c_tokens = set(tokenize_simple(c["text"]))
        overlap = len(q_tokens & c_tokens)
        n_words = len(c["text"].split())
        doc = str(c["id"]).split("_")[0]
        unit_type = c.get("unit_type", "dieu" if c.get("dieu_so", "0") != "0" else "raw")
        X.append([
            float(sc[i]),
            float(i),
            top - float(sc[i]),
            float(sc[i] - sc[i + 1]) if i + 1 < len(sc) else 0.0,
            (float(sc[i]) - mean) / std if std > 1e-9 else 0.0,
            float(n_words),
            float(overlap),
            overlap / max(len(q_tokens), 1),
            1.0 if (q_so_hieu and norm_so_hieu(c.get("so_hieu") or "") in q_so_hieu) else 0.0,
            1.0 if (q_dieu and str(c.get("dieu_so", "0")) in q_dieu) else 0.0,
            float(doc_order[doc]),
            1.0 if (i == 0 or str(cands[i - 1]["id"]).split("_")[0] != doc) else 0.0,
            1.0 if unit_type.startswith("dieu") else 0.0,
        ])
    return X, cands


def apply_ltr_resort(question: str, ranked: list, scores, ltr_model,
                      top_k: int = LTR_TOP_K_CANDIDATES):
    """Xếp lại top_k ứng viên đầu bằng LTR, GIỮ NGUYÊN phần đuôi.

    Chỉ đổi THỨ TỰ trong top_k — không thêm/bớt ứng viên, không đụng tầng 1. Nếu LTR lỗi
    (model None, đặc trưng rỗng, predict ném lỗi) thì trả về đúng đầu vào: hỏng ở đây phải
    im lặng lùi về hành vi đã proven, không được làm sập một phiên GPU nhiều giờ.
    """
    if ltr_model is None or scores is None or not ranked:
        return ranked, scores
    try:
        X, cands = extract_ltr_features(question, ranked, scores, top_k)
        if not X:
            return ranked, scores
        pred = ltr_model.predict(np.asarray(X, dtype=np.float64))
        order = list(np.argsort(-np.asarray(pred)))
        new_head = [cands[i] for i in order]
        head_scores = [float(scores[i]) for i in order]
        tail = list(ranked[len(cands):])
        tail_scores = [float(s) for s in np.asarray(scores[len(cands):], dtype=np.float64)]
        return new_head + tail, np.asarray(head_scores + tail_scores, dtype=np.float64)
    except Exception as e:
        print(f"    [LTR] lỗi lúc resort ({type(e).__name__}: {e}) -> giữ thứ hạng reranker")
        return ranked, scores


In [ ]:
# Cell 12 [v6_2]: Bước 6 — SO NHÁNH TRÊN DEV (3 option) + HARVEST PHA A + dev-eval
#
# v6_1 chạy MỘT cấu hình rồi harvest. v6_2 dùng đúng 300 câu dev đó để trả lời ba câu hỏi
# trong CÙNG một phiên, mỗi câu một cổng split-half (chọn trên nửa A, chỉ nhận khi CẢ HAI nửa
# cùng dương — quy trình đã bắt ảo giác "đỉnh trên toàn dev" nhiều lần trong repo):
#
#   Cổng 1 (Option 1)  reranker: zs (zero-shot) · ft2 (zs chọn văn bản, ft chọn Điều) · ft
#   Cổng 2 (Option 2)  câu dẫn: T0_current · T1 · T2 · T3 — trên nhánh thắng cổng 1
#   Cổng 3 (Option 3)  kênh encoder tầng 1: đủ kênh · bỏ A · bỏ B — trên nhánh + template thắng
#
# Ba cổng nối tiếp chứ không quét tổ hợp: 3×4×3 = 36 tổ hợp trên 150 câu/nửa là đúng kiểu dò
# tham số mà CLAUDE.md cấm. Mọi so sánh là PAIRED (cùng câu, cùng ứng viên tầng 1).
#
# Dev không nằm trong dữ liệu fine-tune encoder lẫn reranker (Cell 7 loại dev_ids khỏi
# train_positive trước khi dựng rows/rows_clean), nên cổng 1 không đo trí nhớ.
import nltk
if str(NLTK_CACHE_DIR) not in nltk.data.path:
    nltk.data.path.insert(0, str(NLTK_CACHE_DIR))
try:
    nltk.data.find("corpora/wordnet")
except LookupError:
    nltk.download("wordnet", quiet=True, download_dir=str(NLTK_CACHE_DIR))
    nltk.download("omw-1.4", quiet=True, download_dir=str(NLTK_CACHE_DIR))
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)


def _meteor(ref: str, hyp: str) -> float:
    # nltk meteor_score nhận list token, KHÔNG phải string.
    try:
        return float(meteor_score([str(ref).split()], str(hyp).split()))
    except Exception:
        return 0.0


# ---------------------------------------------------------------------------
# Cấu hình đang dùng — cổng nào thắng thì ghi đè vào đây; harvest/Bước 7/pha C đều đọc nó.
# ---------------------------------------------------------------------------
ACTIVE = {"arm": "zs", "mask": None}
ARM_LABEL = {"zs": "zeroshot", "ft2": "zeroshot_t1+finetuned_t2", "ft": "finetuned"}


def _models_for(arm: str, dev):
    """-> (model tầng 1, tokenizer tầng 1, model tầng 2 hoặc None, tokenizer tầng 2)."""
    zs_m, zs_t = reranker_models.get(dev), reranker_tokenizers.get(dev)
    if arm == "zs" or dev is None:
        return zs_m, zs_t, None, None
    ft_m, ft_t = reranker_models_ft.get(dev), reranker_tokenizers_ft.get(dev)
    if arm == "ft2":
        return zs_m, zs_t, ft_m, ft_t
    if arm == "ft":
        return ft_m, ft_t, None, None
    raise ValueError(arm)


def _pack(qid, question, gold, ranked, scores, template):
    """Bản ghi cache như v6_1: top-K ứng viên, điểm CE, đặc trưng LTR, METEOR từng ứng viên."""
    rec = {"qid": qid, "question": question, "n_cands": len(ranked) if ranked else 0,
           "cands": [], "feat": []}
    if not ranked:
        return rec
    X, cands = extract_ltr_features(question, ranked, scores, LTR_TOP_K_CANDIDATES)
    rec["feat"] = X
    for i, c in enumerate(cands):
        rec["cands"].append({
            "id": c["id"],
            "ce_score": float(scores[i]) if scores is not None else None,
            "meteor": _meteor(gold, render_answer([c], 1, question, template=template)),
            "n_words": len(c["text"].split()),
            "unit_type": c.get("unit_type", "dieu" if c.get("dieu_so", "0") != "0" else "raw"),
            "dieu_so": c.get("dieu_so", "0"),
            "so_hieu": c.get("so_hieu") or "",
        })
    return rec


def harvest_one(qid: str, question: str, gold: str, dev) -> dict:
    m1, k1, m2, k2 = _models_for(ACTIVE["arm"], dev)
    ranked, scores = retrieve_two_tier(question, bm25_t1, DENSE_CHANNELS, all_chunks_t1,
                                        dieu_by_doc, m1, k1, DOC_K, t2_model=m2,
                                        t2_tokenizer=k2, mask=ACTIVE["mask"])
    return _pack(qid, question, gold, ranked, scores, ANSWER_TEMPLATE)


def run_harvest(qids, label: str):
    """Harvest `qids` với cấu hình ACTIVE, song song trên các GPU có reranker."""
    if not qids:
        return {}
    if len(RERANK_DEVICES) > 1:
        def _worker(chunk, dev, widx):
            out = {}
            for i, qid in enumerate(chunk):
                out[qid] = harvest_one(qid, train_data[qid]["question"],
                                        train_data[qid].get("answer", ""), dev)
                _progress_print(label, widx, i, len(chunk))
            return out
        return parallel_process(qids, _worker, RERANK_DEVICES, label=label)
    dev = RERANK_DEVICES[0] if RERANK_DEVICES else None
    out, t0 = {}, time.time()
    for i, qid in enumerate(qids):
        out[qid] = harvest_one(qid, train_data[qid]["question"],
                                train_data[qid].get("answer", ""), dev)
        if (i + 1) % 50 == 0 or (i + 1) == len(qids):
            print(f"    [{label}] {i+1}/{len(qids)} ... {time.time()-t0:.0f}s")
    return out


def _run_on_devices(qids, fn, label):
    """fn(qid, dev) -> kết quả; chia đều cho các GPU có reranker."""
    devs = RERANK_DEVICES or [None]
    def _worker(chunk, dev, widx):
        out = {}
        for i, qid in enumerate(chunk):
            out[qid] = fn(qid, dev)
            _progress_print(label, widx, i, len(chunk))
        return out
    return parallel_process(qids, _worker, devs, label=label)


def _top1_meteor(rec):
    return rec["cands"][0]["meteor"] if rec["cands"] else 0.0


def _gold_doc(qid):
    cid = train_positive_all.get(qid)
    return str(cid).split("_")[0] if cid else None


def paired_gate(ids, base_fn, cand_fn):
    """Δ = cand - base theo từng câu, trên nửa A / nửa B (id sort, chẵn/lẻ) và toàn mẫu."""
    s = sorted(ids)
    def _stats(sub):
        d = [cand_fn(q) - base_fn(q) for q in sub]
        n = len(d)
        if n == 0:
            return {"n": 0, "delta": 0.0, "se": 0.0, "win": 0, "loss": 0}
        mean = sum(d) / n
        sd = (sum((x - mean) ** 2 for x in d) / (n - 1)) ** 0.5 if n > 1 else 0.0
        return {"n": n, "delta": round(mean, 5), "se": round(sd / n ** 0.5, 5),
                "win": sum(x > 1e-9 for x in d), "loss": sum(x < -1e-9 for x in d)}
    a, b = _stats(s[0::2]), _stats(s[1::2])
    return {"half_a": a, "half_b": b, "all": _stats(s),
            "pass": a["delta"] > 0 and b["delta"] > 0}


def choose_by_gate(label, ids, base_name, candidates: dict, base_fn):
    """candidates = {tên: cand_fn}. Chọn tên có Δ nửa A lớn nhất, NHẬN nếu hai nửa cùng dương."""
    report = {"base": base_name, "candidates": {}}
    for name, fn in candidates.items():
        report["candidates"][name] = paired_gate(ids, base_fn, fn)
    print(f"  --- Cổng {label} (n={len(ids)}, mốc = {base_name}) ---")
    for name, g_ in report["candidates"].items():
        a_, b_, al = g_["half_a"], g_["half_b"], g_["all"]
        weak = "  ⚠️ |Δ| < 2·SE" if abs(al["delta"]) < 2 * al["se"] else ""
        print(f"    {name:<22s} nửa A {a_['delta']:+.4f} · nửa B {b_['delta']:+.4f} · "
              f"toàn mẫu {al['delta']:+.4f} ± {al['se']:.4f} "
              f"({al['win']} thắng / {al['loss']} thua){weak}")
    winner = base_name
    if report["candidates"]:
        best = max(report["candidates"], key=lambda k: report["candidates"][k]["half_a"]["delta"])
        if report["candidates"][best]["pass"]:
            winner = best
    report["winner"] = winner
    print(f"    => chọn {winner}")
    return winner, report


print("=== Bước 6 [v6_2]: So nhánh trên dev + harvest pha A ===")
V62_DECISIONS = {"arms_evaluated": [], "gate_reranker": None, "gate_template": None,
                 "gate_channels": None, "doc_recall_at_doc_k": {}}
harvest_records = {}
dev_log = {q: {"qid": q, "gold_doc": _gold_doc(q)} for q in dev_ids}

# ---------------------------------------------------------------------------
# CỔNG 1 — reranker. Một lượt/câu cho mọi nhánh: câu hỏi encode một lần, điểm CE dùng chung
# giữa các nhánh qua memo (nhánh ft2 tái dùng nguyên tầng 1 của zs).
# ---------------------------------------------------------------------------
ARMS = ["zs"] + (["ft2", "ft"] if HAS_FT else [])
_est_dev_sec = 4.6 * len(dev_ids) * (1.0 + 0.75 * (len(ARMS) - 1))
if len(ARMS) > 1 and _est_dev_sec > remaining() - PUBLIC_RESERVE_SEC - 45 * 60:
    print(f"  [cổng 1] không đủ giờ cho {len(ARMS)} nhánh (ước {_est_dev_sec/60:.0f} phút) -> chỉ zs")
    ARMS = ["zs"]
V62_DECISIONS["arms_evaluated"] = ARMS
dev_memo = {}


def _dev_arms_one(qid, dev):
    q, gold = train_data[qid]["question"], train_data[qid].get("answer", "")
    rank_lists = rrf_rank_lists(q, bm25_t1, DENSE_CHANNELS)
    memo = {"zs": ({}, {}), "ft": ({}, {})}
    recs, diags = {}, {}
    for arm in ARMS:
        m1, k1, m2, k2 = _models_for(arm, dev)
        mt1 = memo["ft" if arm == "ft" else "zs"][0]
        mt2 = memo["zs" if arm == "zs" else "ft"][1]
        diag = {}
        ranked, scores = retrieve_two_tier(q, bm25_t1, DENSE_CHANNELS, all_chunks_t1,
                                            dieu_by_doc, m1, k1, DOC_K, t2_model=m2,
                                            t2_tokenizer=k2, rank_lists=rank_lists,
                                            memo_t1=mt1, memo_t2=mt2, diag=diag)
        recs[arm] = _pack(qid, q, gold, ranked, scores, ANSWER_TEMPLATE)
        diags[arm] = diag
    return {"rank_lists": rank_lists, "memo": memo, "recs": recs, "diags": diags}


t_dev = time.time()
dev_out = _run_on_devices(list(dev_ids), _dev_arms_one, f"dev {len(ARMS)} nhánh")
dev_arms_min = (time.time() - t_dev) / 60
print(f"  [cổng 1] {len(dev_ids)} câu × {len(ARMS)} nhánh trong {dev_arms_min:.1f} phút")
for qid, o in dev_out.items():
    dev_log[qid]["arms"] = {arm: {"top_docs": o["diags"][arm].get("top_docs", []),
                                  "cands": [{"id": c["id"], "ce": c["ce_score"],
                                             "meteor_T0": round(c["meteor"], 4)}
                                            for c in o["recs"][arm]["cands"]]}
                            for arm in ARMS}
for arm in ARMS:
    lab = [q for q in dev_ids if _gold_doc(q)]
    if lab:
        hit = sum(_gold_doc(q) in dev_out[q]["diags"][arm].get("top_docs", []) for q in lab)
        V62_DECISIONS["doc_recall_at_doc_k"][f"arm_{arm}"] = round(hit / len(lab), 4)

chosen_arm, V62_DECISIONS["gate_reranker"] = choose_by_gate(
    "1 · reranker", dev_ids, "zs",
    {arm: (lambda q, a=arm: _top1_meteor(dev_out[q]["recs"][a])) for arm in ARMS if arm != "zs"},
    lambda q: _top1_meteor(dev_out[q]["recs"]["zs"]))
ACTIVE["arm"] = chosen_arm
RERANKER_SOURCE = ARM_LABEL[chosen_arm]

# Giải phóng reranker không còn dùng — pipeline nộp chỉ giữ đúng thứ nó cần.
if chosen_arm == "zs" and reranker_models_ft:
    reranker_models_ft.clear()
    reranker_tokenizers_ft.clear()
elif chosen_arm == "ft":
    for dev in list(reranker_models):
        reranker_models[dev] = None
if chosen_arm != "ft2":
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
checkpoint(f"Xong cổng 1 (reranker = {RERANKER_SOURCE})")

# ---------------------------------------------------------------------------
# CỔNG 2 — câu dẫn. 0 GPU: ứng viên hạng 1 của nhánh thắng đã có sẵn, chỉ dựng lại chuỗi.
# ---------------------------------------------------------------------------
tmpl_meteor = {}
for qid in dev_ids:
    rec = dev_out[qid]["recs"][chosen_arm]
    gold = train_data[qid].get("answer", "")
    if not rec["cands"] or rec["cands"][0]["id"] not in chunk_by_id:
        tmpl_meteor[qid] = {t: 0.0 for t in ANSWER_TEMPLATES}
        continue
    c = chunk_by_id[rec["cands"][0]["id"]]
    tmpl_meteor[qid] = {t: (rec["cands"][0]["meteor"] if t == ANSWER_TEMPLATE else
                            _meteor(gold, render_answer([c], 1, rec["question"], template=t)))
                        for t in ANSWER_TEMPLATES}
    dev_log[qid]["template_meteor_top1"] = {t: round(v, 4) for t, v in tmpl_meteor[qid].items()}

if RUN_TEMPLATE_GATE:
    chosen_tmpl, V62_DECISIONS["gate_template"] = choose_by_gate(
        "2 · câu dẫn", dev_ids, ANSWER_TEMPLATE,
        {t: (lambda q, t_=t: tmpl_meteor[q][t_]) for t in ANSWER_TEMPLATES if t != ANSWER_TEMPLATE},
        lambda q: tmpl_meteor[q][ANSWER_TEMPLATE])
else:
    chosen_tmpl = ANSWER_TEMPLATE
if chosen_tmpl != ANSWER_TEMPLATE:
    ANSWER_TEMPLATE = chosen_tmpl
    # Nhãn LTR (METEOR từng ứng viên) phải khớp template sẽ nộp -> chấm lại 5 ứng viên/câu.
    for qid in dev_ids:
        rec = dev_out[qid]["recs"][chosen_arm]
        gold = train_data[qid].get("answer", "")
        for cand in rec["cands"]:
            if cand["id"] in chunk_by_id:
                cand["meteor"] = _meteor(gold, render_answer([chunk_by_id[cand["id"]]], 1,
                                                             rec["question"]))
checkpoint(f"Xong cổng 2 (câu dẫn = {ANSWER_TEMPLATE})")

# ---------------------------------------------------------------------------
# CỔNG 3 — kênh encoder tầng 1. Không encode lại gì: xếp hạng từng kênh đã có trong
# rank_lists, điểm CE đã chấm nằm trong memo -> chỉ chấm các ứng viên MỚI lọt vào.
# Trả lời câu hỏi "encoder nào đang thật sự kéo điểm" mà không tốn thêm một lượt encode corpus.
# ---------------------------------------------------------------------------
dev_final = {q: dev_out[q]["recs"][chosen_arm] for q in dev_ids}
if RUN_CHANNEL_ABLATION and len(DENSE_CHANNELS) >= 2 and HAS_RERANKER:
    masks = {}
    for i, name in enumerate(CHANNEL_NAMES):
        if name == "bm25":
            continue
        masks[f"no_{name}"] = tuple(j != i for j in range(len(CHANNEL_NAMES)))

    def _ablate_one(qid, dev):
        o = dev_out[qid]
        q, gold = train_data[qid]["question"], train_data[qid].get("answer", "")
        m1, k1, m2, k2 = _models_for(chosen_arm, dev)
        mt1 = o["memo"]["ft" if chosen_arm == "ft" else "zs"][0]
        mt2 = o["memo"]["zs" if chosen_arm == "zs" else "ft"][1]
        res = {}
        for mname, mask in masks.items():
            diag = {}
            ranked, scores = retrieve_two_tier(q, bm25_t1, DENSE_CHANNELS, all_chunks_t1,
                                                dieu_by_doc, m1, k1, DOC_K, t2_model=m2,
                                                t2_tokenizer=k2, mask=mask,
                                                rank_lists=o["rank_lists"],
                                                memo_t1=mt1, memo_t2=mt2, diag=diag)
            res[mname] = (_pack(qid, q, gold, ranked, scores, ANSWER_TEMPLATE), diag)
        return res

    _est_abl = 0.5 * dev_arms_min * 60 / max(len(ARMS), 1) * len(masks)
    if _est_abl < remaining() - PUBLIC_RESERVE_SEC - 30 * 60:
        abl = _run_on_devices(list(dev_ids), _ablate_one, "dev bỏ kênh")
        for qid, r in abl.items():
            dev_log[qid]["ablation"] = {m: {"top_docs": d.get("top_docs", []),
                                            "top1": (rec["cands"][0]["id"] if rec["cands"] else None),
                                            "meteor": round(_top1_meteor(rec), 4)}
                                        for m, (rec, d) in r.items()}
        lab = [q for q in dev_ids if _gold_doc(q)]
        for mname in masks:
            if lab:
                hit = sum(_gold_doc(q) in abl[q][mname][1].get("top_docs", []) for q in lab)
                V62_DECISIONS["doc_recall_at_doc_k"][mname] = round(hit / len(lab), 4)
        chosen_mask, V62_DECISIONS["gate_channels"] = choose_by_gate(
            "3 · kênh encoder", dev_ids, "all_channels",
            {m: (lambda q, m_=m: _top1_meteor(abl[q][m_][0])) for m in masks},
            lambda q: _top1_meteor(dev_final[q]))
        if chosen_mask != "all_channels":
            ACTIVE["mask"] = masks[chosen_mask]
            dev_final = {q: abl[q][chosen_mask][0] for q in dev_ids}
        V62_DECISIONS["gate_channels"]["active_mask"] = chosen_mask
    else:
        V62_DECISIONS["gate_channels"] = {"skipped": f"không đủ giờ (ước {_est_abl/60:.0f} phút)"}
        print(f"  [cổng 3] BỎ QUA — {V62_DECISIONS['gate_channels']['skipped']}")
checkpoint(f"Xong cổng 3 (mask = {ACTIVE['mask']})")

V62_DECISIONS["active"] = {"arm": ACTIVE["arm"], "reranker": RERANKER_SOURCE,
                           "template": ANSWER_TEMPLATE,
                           "channels": ([n for n, k in zip(CHANNEL_NAMES, ACTIVE["mask"]) if k]
                                        if ACTIVE["mask"] else CHANNEL_NAMES)}
print(f"  CẤU HÌNH CHỐT: {V62_DECISIONS['active']}")
harvest_records.update(dev_final)
del dev_out

# ---------------------------------------------------------------------------
# HARVEST PHA A (phần LTR) với cấu hình đã chốt. Ngân sách tự co như v6_1.
# ---------------------------------------------------------------------------
budget_a_sec = max(0.0, remaining() - PUBLIC_RESERVE_SEC)
phase_a_ids = [q for q in harvest_ids_all[:HARVEST_PHASE_A_N] if q not in harvest_records]
print(f"  Ngân sách pha A: {budget_a_sec/60:.1f} phút (giữ {PUBLIC_RESERVE_SEC/60:.0f} phút cho "
      f"Bước 7). Mục tiêu thêm {len(phase_a_ids)} câu LTR.")
done_ids, t_harvest = [], time.time()
for b0 in range(0, len(phase_a_ids), HARVEST_BATCH):
    batch = phase_a_ids[b0:b0 + HARVEST_BATCH]
    if b0 > 0:
        rate = (time.time() - t_harvest) / max(len(done_ids), 1)
        need = rate * len(batch)
        left = budget_a_sec - (time.time() - t_harvest)
        if need > left:
            print(f"  [pha A] DỪNG SỚM ở {len(done_ids)} câu — lô tiếp cần ~{need/60:.1f} phút, "
                  f"còn {left/60:.1f} phút trước ngân sách Bước 7.")
            break
    harvest_records.update(run_harvest(batch, f"harvest A {b0+len(batch)}/{len(phase_a_ids)}"))
    done_ids.extend(batch)
harvest_rate_s = ((time.time() - t_harvest) / len(done_ids)) if done_ids else 8.0
print(f"  [pha A] xong {len(done_ids)} câu trong {(time.time()-t_harvest)/60:.1f} phút "
      f"({harvest_rate_s:.2f} giây/câu).")

# ---------------------------------------------------------------------------
# Dev-eval: chọn TOP_N_ANSWER trên cấu hình chốt — như v6_1, chỉ đọc cache.
# ---------------------------------------------------------------------------
dev_done = [q for q in dev_ids if q in harvest_records]
print(f"  --- dev-eval trên {len(dev_done)} câu dev ---")


def meteor_at_top_n(qids, records, top_n: int, order_key=None):
    ms, rs = [], []
    for qid in qids:
        rec = records[qid]
        gold = str(train_data[qid].get("answer", ""))
        if not rec["cands"]:
            ms.append(0.0)
            rs.append(0.0)
            continue
        idx = order_key(rec) if order_key else list(range(len(rec["cands"])))
        if top_n == 1:
            ms.append(rec["cands"][idx[0]]["meteor"])
            rs.append(0.0)
            continue
        chunks = [chunk_by_id[rec["cands"][i]["id"]] for i in idx[:top_n]
                   if rec["cands"][i]["id"] in chunk_by_id]
        pred = render_answer(chunks, top_n, rec["question"]) if chunks else ""
        ms.append(_meteor(gold, pred))
        rs.append(rouge.score(gold, pred)["rougeL"].fmeasure)
    return (sum(ms) / max(len(ms), 1)), (sum(rs) / max(len(rs), 1))


best_n, best_m = 1, -1.0
for top_n in range(1, TOP_K_RERANK + 1):
    m, r = meteor_at_top_n(dev_done, harvest_records, top_n)
    print(f"    top_n={top_n}  METEOR={m:.4f}" + (f"  ROUGE-L={r:.4f}" if top_n > 1 else "")
          + f"  (n={len(dev_done)})")
    if m > best_m:
        best_m, best_n = m, top_n

best_r = 0.0
if dev_done:
    _rs = []
    for qid in dev_done:
        rec = harvest_records[qid]
        gold = str(train_data[qid].get("answer", ""))
        chunks = [chunk_by_id[c["id"]] for c in rec["cands"][:best_n] if c["id"] in chunk_by_id]
        pred = render_answer(chunks, best_n, rec["question"]) if chunks else ""
        _rs.append(rouge.score(gold, pred)["rougeL"].fmeasure)
    best_r = sum(_rs) / max(len(_rs), 1)

top_n_answer = best_n
use_reranker = HAS_RERANKER
use_adaptive = False
print(f"  => TOP_N_ANSWER={top_n_answer}  (METEOR={best_m:.4f} · ROUGE-L={best_r:.4f})")

recall_ids = [q for q in dev_done if q in train_positive_all]
recall_at_k = {}
if recall_ids:
    ks = [1, 3, 5]
    hits = {k: 0 for k in ks}
    for qid in recall_ids:
        ids = [c["id"] for c in harvest_records[qid]["cands"]]
        pos = train_positive_all[qid]
        for k in ks:
            if pos in ids[:k]:
                hits[k] += 1
    recall_at_k = {str(k): round(hits[k] / len(recall_ids), 4) for k in ks}
    print(f"  Recall@k trên {len(recall_ids)} câu dev có citation (THAM KHẢO): "
          + " · ".join(f"@{k}={100*hits[k]/len(recall_ids):.1f}%" for k in ks))

eval_info = {"meteor": round(best_m, 4), "rouge_l": round(best_r, 4),
             "n_dev": len(dev_done), "recall_at_k": recall_at_k,
             "harvest_rate_s": round(harvest_rate_s, 2),
             "n_harvest_phase_a": len(dev_done) + len(done_ids),
             "dev_arms_min": round(dev_arms_min, 1)}

with open(os.path.join(OUT_DIR, "v6_2_decisions.json"), "w", encoding="utf-8") as f:
    json.dump(V62_DECISIONS, f, ensure_ascii=False, indent=2, default=str)
with open(os.path.join(OUT_DIR, "v6_2_dev_arms.jsonl"), "w", encoding="utf-8") as f:
    for qid in dev_ids:
        f.write(json.dumps(dev_log[qid], ensure_ascii=False, default=str) + "\n")
print(f"  Đã ghi v6_2_decisions.json + v6_2_dev_arms.jsonl ({len(dev_ids)} câu, mọi nhánh)")
checkpoint(f"Xong so nhánh + harvest pha A ({len(harvest_records)} câu) + dev-eval")


In [ ]:
# Cell 13 [v6_1]: BỘ CHỌN ĐIỀU HỌC ĐƯỢC (LTR) — train trên cache của Cell 12, 0 giây GPU
#
# Đây là hướng ưu tiên #1 của cả result.md §7 lẫn RESEARCH_PLAN_BREAKTHROUGH.md §3.1: Điều
# tốt nhất ĐÃ nằm trong 5 ứng viên (oracle 0,6401 vs thực tế 0,5630) — cần một bộ chọn HỌC
# ĐƯỢC, vì cả bốn quy tắc heuristic đã thử (dài nhất/ngắn nhất/gần trung vị) đều TỆ HƠN giữ
# nguyên hạng 1, và tương quan (độ dài, METEOR) chỉ −0,078.
#
# TÁCH DỮ LIỆU — ba tập rời nhau, không tập nào chồng lên tập nào:
#   fine-tune encoder :  câu CÓ citation, trừ dev_ids            (Cell 7/8)
#   train LTR         :  ltr_pool = câu KHÔNG citation, trừ dev_ids   <- ở đây
#   gate LTR          :  dev_ids                                  (Cell 7)
# Nhờ vậy LTR không bao giờ được chấm trên câu nó đã thấy, VÀ không câu nào trong cả ba tập
# từng đi vào fine-tune encoder cùng lúc với việc được dùng để chấm.
#
# CỔNG SPLIT-HALF (chuyển giao từ Task 1, ../result.md §14): chọn trên nửa A, ĐỌC ĐIỂM trên
# nửa B, chỉ nhận khi CẢ HAI nửa cùng dương. Quy trình này đã ba lần bắt được "đỉnh trên toàn
# dev" trong repo (ensemble RRF, hàm gộp, quét 1.050 tổ hợp) — mỗi lần dev tăng rồi public
# giảm. Ở lượt Kaggle gần nhất chính cổng này đã tự loại LSE khi nửa B ra âm.
print("=== Bước 6b [v6_1]: Bộ chọn Điều học được (LTR) ===")
ltr_model = None
ltr_report = {"trained": False, "reason": None, "n_groups": 0,
              "delta_half_a": None, "delta_half_b": None, "accepted": False}

ltr_train_ids = [q for q in harvest_records if q not in dev_ids_set and harvest_records[q]["feat"]]
print(f"  Nhóm khả dụng để train: {len(ltr_train_ids)} (ngưỡng tối thiểu {LTR_MIN_GROUPS})")

if not USE_LTR:
    ltr_report["reason"] = "USE_LTR=False"
elif len(ltr_train_ids) < LTR_MIN_GROUPS:
    ltr_report["reason"] = f"chỉ {len(ltr_train_ids)} nhóm < LTR_MIN_GROUPS={LTR_MIN_GROUPS}"
else:
    try:
        import lightgbm as lgb

        # -------------------------------------------------------------------
        # NHÃN: METEOR của từng ứng viên, rời rạc hoá TRONG TỪNG NHÓM về thang 0..4.
        # Phải rời rạc hoá theo nhóm chứ không theo thang tuyệt đối: LambdaRank chỉ quan tâm
        # THỨ TỰ trong nhóm, và METEOR tuyệt đối giữa các câu hỏi không so sánh được với nhau
        # (câu có đáp án dài dễ được điểm cao hơn câu đáp án ngắn, không liên quan tới việc
        # ứng viên nào đúng).
        #
        # Bỏ nhóm có spread quá nhỏ: nếu cả 5 ứng viên cho METEOR gần như nhau thì nhóm đó
        # không dạy được gì về thứ tự, chỉ thêm nhiễu vào gradient.
        # -------------------------------------------------------------------
        X_rows, y_rows, groups, n_flat = [], [], [], 0
        for qid in ltr_train_ids:
            rec = harvest_records[qid]
            ms = [c["meteor"] for c in rec["cands"]]
            feat = rec["feat"]
            if len(ms) < 2 or len(feat) != len(ms):
                continue
            lo, hi = min(ms), max(ms)
            if hi - lo < 0.02:
                n_flat += 1
                continue
            for f, m in zip(feat, ms):
                X_rows.append(f)
                y_rows.append(int(round(4 * (m - lo) / (hi - lo))))
            groups.append(len(ms))
        print(f"  {len(groups)} nhóm sau lọc ({n_flat} nhóm bỏ vì 5 ứng viên gần như đồng "
              f"điểm — không có thứ tự nào để học), {len(X_rows)} dòng đặc trưng.")

        if len(groups) < LTR_MIN_GROUPS:
            raise RuntimeError(f"còn {len(groups)} nhóm sau lọc, dưới ngưỡng {LTR_MIN_GROUPS}")

        ltr_model = lgb.LGBMRanker(
            # eval_at=[1] chứ không phải ndcg_eval_at: cùng ý nghĩa nhưng ndcg_eval_at là
            # alias mức thấp, LGBMRanker bắn UserWarning mỗi lần fit. Và [1] chứ không phải
            # [1,3,5] — ta chỉ dùng ỨNG VIÊN HẠNG 1 để dựng đáp án (TOP_N_ANSWER=1), nên tối
            # ưu NDCG@3 là tối ưu một thứ không ai đọc.
            objective="lambdarank", metric="ndcg", eval_at=[1],
            num_leaves=LTR_NUM_LEAVES, n_estimators=LTR_N_ESTIMATORS,
            learning_rate=LTR_LEARNING_RATE, min_child_samples=20,
            random_state=SEED, n_jobs=2, verbose=-1)
        ltr_model.fit(np.asarray(X_rows, dtype=np.float64), np.asarray(y_rows),
                       group=groups, feature_name=LTR_FEATURE_NAMES)
        ltr_report.update({"trained": True, "n_groups": len(groups)})

        imp = sorted(zip(LTR_FEATURE_NAMES, ltr_model.feature_importances_),
                      key=lambda kv: -kv[1])
        print("  Đặc trưng quan trọng nhất: "
              + " · ".join(f"{k}={v}" for k, v in imp[:5]))
        ltr_report["feature_importance"] = {k: int(v) for k, v in imp}

        # -------------------------------------------------------------------
        # CỔNG SPLIT-HALF trên dev_ids
        # -------------------------------------------------------------------
        def _ltr_order(rec):
            pred = ltr_model.predict(np.asarray(rec["feat"], dtype=np.float64))
            return list(np.argsort(-np.asarray(pred)))

        def _meteor_of(qids, order_key):
            vals = []
            for qid in qids:
                rec = harvest_records[qid]
                if not rec["cands"]:
                    vals.append(0.0)
                    continue
                idx = order_key(rec) if order_key else list(range(len(rec["cands"])))
                vals.append(rec["cands"][idx[0]]["meteor"])
            return sum(vals) / max(len(vals), 1)

        ids_sorted = sorted(dev_done)
        half_a, half_b = ids_sorted[0::2], ids_sorted[1::2]
        base_a, base_b = _meteor_of(half_a, None), _meteor_of(half_b, None)
        ltr_a, ltr_b = _meteor_of(half_a, _ltr_order), _meteor_of(half_b, _ltr_order)
        d_a, d_b = ltr_a - base_a, ltr_b - base_b
        ltr_report.update({"delta_half_a": round(d_a, 4), "delta_half_b": round(d_b, 4),
                            "base_half_a": round(base_a, 4), "base_half_b": round(base_b, 4)})
        print(f"    reranker (baseline)  nửa A {base_a:.4f} · nửa B {base_b:.4f}")
        print(f"    + LTR resort         nửa A {ltr_a:.4f} ({d_a:+.4f}) · "
              f"nửa B {ltr_b:.4f} ({d_b:+.4f})")

        # Trần lý thuyết trên chính mẫu này — để biết LTR ăn được bao nhiêu phần của dư địa.
        orc = _meteor_of(ids_sorted, lambda r: list(np.argsort(
            -np.asarray([c["meteor"] for c in r["cands"]]))))
        base_all = _meteor_of(ids_sorted, None)
        print(f"    oracle (trần lý thuyết, chọn tốt nhất trong {LTR_TOP_K_CANDIDATES}) "
              f"{orc:.4f} — dư địa {orc-base_all:+.4f} trên mẫu này")
        ltr_report.update({"oracle_meteor": round(orc, 4), "base_meteor": round(base_all, 4)})

        if d_a > 0 and d_b > 0:
            ltr_report["accepted"] = True
            captured = (min(d_a, d_b)) / max(orc - base_all, 1e-9)
            print(f"    => NHẬN LTR (cả hai nửa cùng dương; nửa yếu hơn ăn được "
                  f"{100*captured:.0f}% dư địa oracle)")
        else:
            print(f"    => BỎ LTR, giữ thứ hạng reranker. Một trong hai nửa không xác nhận "
                  f"({d_a:+.4f} / {d_b:+.4f}) — đúng kiểu ảo giác đỉnh-trên-toàn-dev mà cổng "
                  f"này sinh ra để bắt. Bản thân việc BỎ cũng là kết quả: nó nói dư địa +7,7 "
                  f"điểm không nằm trong 13 đặc trưng hiện có.")
    except Exception as e:
        print(f"  [LTR] KHÔNG train được ({type(e).__name__}: {e}) -> giữ thứ hạng reranker.")
        ltr_report["reason"] = f"{type(e).__name__}: {e}"
        ltr_model = None

use_ltr = bool(ltr_report["accepted"]) and ltr_model is not None
if not use_ltr:
    ltr_model = None          # ép None để apply_ltr_resort() thành no-op ở Bước 7
    if ltr_report["reason"]:
        print(f"  LTR không dùng: {ltr_report['reason']}")

with open(os.path.join(OUT_DIR, "ltr_report.json"), "w", encoding="utf-8") as f:
    json.dump(ltr_report, f, ensure_ascii=False, indent=2, default=str)
checkpoint(f"Xong LTR (dùng={use_ltr})")


In [ ]:
# Cell 14 [v6_2]: Bước 7 — Sinh câu trả lời cho public-official.json RỒI ĐÓNG GÓI NGAY
#
# Giữ nguyên thứ tự v6_1 (nộp trước, chẩn đoán sau). Khác duy nhất: đọc cấu hình ĐÃ QUA CỔNG
# ở Cell 12 (ACTIVE + ANSWER_TEMPLATE) thay vì một reranker cố định.
print("=== Bước 7 [v6_2]: Sinh câu trả lời cho public-official.json ===")
print(f"  Cấu hình: reranker={RERANKER_SOURCE} · câu dẫn={ANSWER_TEMPLATE} · "
      f"kênh={V62_DECISIONS['active']['channels']} · top_n={top_n_answer} · LTR={use_ltr}")


def answer_question_v62(question: str, dev=None) -> str:
    m1, k1, m2, k2 = _models_for(ACTIVE["arm"], dev) if use_reranker else (None, None, None, None)
    ranked, scores = retrieve_two_tier(question, bm25_t1, DENSE_CHANNELS, all_chunks_t1,
                                        dieu_by_doc, m1, k1, DOC_K, t2_model=m2,
                                        t2_tokenizer=k2, mask=ACTIVE["mask"])
    if not ranked:
        return "Không tìm thấy thông tin pháp lý cho câu hỏi này."
    if use_ltr:
        ranked, scores = apply_ltr_resort(question, ranked, scores, ltr_model)
    return render_answer(ranked, top_n_answer, question)


with open(PUBLIC_PATH, encoding="utf-8") as f:
    questions = json.load(f)
qids = list(questions.keys())
print(f"  {len(qids)} câu hỏi public.")

if use_reranker and len(RERANK_DEVICES) > 1:
    def _worker(chunk, dev, widx):
        out = {}
        for i, qid in enumerate(chunk):
            out[qid] = answer_question_v62(questions[qid]["question"], dev)
            _progress_print("Bước 7", widx, i, len(chunk))
        return out
    answers = parallel_process(qids, _worker, RERANK_DEVICES, label="Bước 7")
else:
    rr_dev = RERANK_DEVICES[0] if (use_reranker and RERANK_DEVICES) else None
    answers = {}
    for i, qid in enumerate(qids):
        answers[qid] = answer_question_v62(questions[qid]["question"], rr_dev)
        if (i + 1) % 200 == 0:
            print(f"  ... {i+1}/{len(qids)}  ({elapsed()/60:.1f} phút)")

n_empty = sum(1 for a in answers.values() if not a.strip())
print(f"  Đã sinh {len(answers)} câu trả lời, {n_empty} câu rỗng")

# ---------------------------------------------------------------------------
# Đóng gói. Ba ràng buộc chết người của Task 2 (xem CLAUDE.md §6 + scoring/SCORING_LegalQA.md):
#   · tập key phải TRÙNG KHÍT public-official.json — thiếu/thừa là scorer raise, 0 điểm cả bài
#   · answer phải là string không rỗng
#   · zip phải chứa ĐÚNG MỘT file tên submission.json (đã từng sai ở Task 1 — nộp zip có
#     thêm thư mục con và validate_submission.py mù với lỗi đó)
# Kiểm lại bằng cách ĐỌC NGƯỢC từ đĩa, không tin biến trong RAM.
# ---------------------------------------------------------------------------
def build_submission(answers: dict, expected_ids: set, out_zip: Path) -> None:
    errors = []
    got = set(answers.keys())
    if got != expected_ids:
        errors.append(f"Key lệch: thiếu {len(expected_ids-got)}, thừa {len(got-expected_ids)}")
    for qid, ans in answers.items():
        if not isinstance(ans, str) or not ans.strip():
            errors.append(f"[{qid}] answer rỗng hoặc không phải string")
    if errors:
        raise ValueError("Submission KHÔNG hợp lệ:\n  - " + "\n  - ".join(errors[:20]))

    normalized = {qid: {"answer": str(ans)} for qid, ans in answers.items()}
    json_path = out_zip.with_suffix(".json")
    with json_path.open("w", encoding="utf-8") as f:
        json.dump(normalized, f, ensure_ascii=False)
    with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(json_path, arcname="submission.json")
    with zipfile.ZipFile(out_zip) as zf:
        assert zf.namelist() == ["submission.json"], f"zip chứa {zf.namelist()}"
        reloaded = json.loads(zf.read("submission.json").decode("utf-8"))
        assert reloaded == normalized
    print(f"  OK — {out_zip} ({len(normalized)} câu trả lời, đã kiểm tra lại từ đĩa)")


print("=== Bước 8 [v6_2]: Đóng gói submission.zip (TRƯỚC phần chẩn đoán) ===")
build_submission(answers, set(questions.keys()), Path(OUT_DIR) / "submission.zip")
checkpoint("✅ BÀI NỘP ĐÃ AN TOÀN TRÊN ĐĨA — mọi thứ sau đây là phần ăn thêm")
if elapsed() > SUBMISSION_DEADLINE_SEC:
    print(f"  [GHI CHÚ] Về đích ở phút {elapsed()/60:.1f}, muộn hơn mốc tự đặt "
          f"{SUBMISSION_DEADLINE_SEC/60:.0f} phút. Lần sau hạ HARVEST_PHASE_A_N.")


In [ ]:
# Cell 15 [v6_1]: HARVEST PHA C (mở rộng) + PHÂN TÍCH LỖI
#
# Chạy SAU khi submission.zip đã nằm trên đĩa. Mọi thứ ở đây là phần ăn thêm: nếu Kaggle ngắt
# phiên giữa chừng, bài nộp không suy suyển.
#
# ⚠️ VÌ SAO KHÔNG PHẢI 7.000 CÂU — đọc kỹ trước khi tăng HARVEST_TARGET_TOTAL. Đo từ lượt
# thật: ~4,9 giây/câu với 2 GPU song song. 7.000 câu = 572 phút, tức nhiều hơn TOÀN BỘ phần
# fine-tune + encode corpus cộng lại (307 phút). Con số 7.000 không nằm trong bất kỳ phiên
# nào, dù có bỏ hết mọi thứ khác. Cỡ mẫu ở đây do THỜI GIAN CÒN LẠI quyết định.
#
# Và cỡ mẫu lớn hơn KHÔNG mua được nhiều như ta tưởng: với ~3.000 câu, sai số chuẩn của một
# tỉ lệ quanh 40% đã là √(0,4·0,6/3000) = 0,9 điểm phần trăm. Đủ để nói "lỗi xếp hạng chiếm
# 38-42%", mà đó chính là độ phân giải cần cho việc ra quyết định hướng đi. Gấp đôi mẫu chỉ
# thu sai số về 0,63 điểm — không đổi kết luận nào.
print("=== Bước 9 [v6_2]: Harvest mở rộng + phân tích lỗi ===")

already = set(harvest_records)
rest_ids = [q for q in harvest_ids_all if q not in already][
    : max(0, HARVEST_TARGET_TOTAL - len(already))]
print(f"  Đã có {len(already)} câu · muốn thêm tối đa {len(rest_ids)} câu "
      f"· còn {remaining()/60:.1f} phút.")

t_c = time.time()
n_added = 0
for b0 in range(0, len(rest_ids), HARVEST_BATCH):
    batch = rest_ids[b0:b0 + HARVEST_BATCH]
    # Dự phòng 5 phút để kịp ghi file phân tích — dừng cụt giữa lô thì mất cả lô.
    need = harvest_rate_s * len(batch)
    if need > remaining() - 5 * 60:
        print(f"  [pha C] DỪNG ở {len(already)+n_added} câu — lô tiếp cần ~{need/60:.1f} phút, "
              f"còn {remaining()/60:.1f} phút.")
        break
    harvest_records.update(run_harvest(batch, f"harvest C +{n_added+len(batch)}"))
    n_added += len(batch)
print(f"  [pha C] thêm {n_added} câu trong {(time.time()-t_c)/60:.1f} phút. "
      f"TỔNG harvest = {len(harvest_records)} câu.")


# ---------------------------------------------------------------------------
# PHÂN LOẠI LỖI. Bốn nhóm, phân biệt được bằng hai con số đã có sẵn trong cache:
#   chosen  = METEOR của ứng viên pipeline THỰC SỰ chọn (hạng 1 sau LTR nếu có)
#   oracle  = METEOR của ứng viên TỐT NHẤT trong top-K
#
#   ok              chosen đã đủ tốt -> không phải lỗi
#   retrieval_fail  oracle thấp -> KHÔNG ứng viên nào cứu được câu này. Bộ chọn bó tay;
#                   muốn chữa phải sửa tầng 1 (chọn văn bản) hoặc tầng cắt Điều.
#   ranking_fail    oracle cao hơn chosen rõ rệt -> đáp án tốt ĐÃ nằm trong tay mà không
#                   chọn. Đây đúng là loại lỗi LTR nhắm tới, và là chỗ +7,7 điểm của §7.
#   extraction_weak chosen ≈ oracle nhưng cả hai đều thấp -> chọn đúng Điều rồi mà điểm vẫn
#                   kém: lỗi ở khâu dựng câu trả lời (template/độ dài/ranh giới Điều), không
#                   phải ở khâu chọn.
#
# Ranh giới giữa bốn nhóm là NGƯỠNG TỰ ĐẶT (ERR_* ở Cell 2), không phải sự thật khách quan.
# Tỉ lệ tuyệt đối vì thế đọc kèm ngưỡng; cái đáng tin là so sánh giữa các lượt CÙNG ngưỡng.
# ---------------------------------------------------------------------------
def classify_error(chosen_m: float, oracle_m: float) -> str:
    if chosen_m >= ERR_OK_METEOR:
        return "ok"
    if oracle_m < ERR_RETRIEVAL_CEIL:
        return "retrieval_fail"
    if oracle_m - chosen_m >= ERR_RANKING_GAP:
        return "ranking_fail"
    return "extraction_weak"


def _chosen_index(rec) -> int:
    """Ứng viên pipeline thật sự chọn — phải khớp ĐÚNG cấu hình đã nộp, kể cả LTR."""
    if use_ltr and ltr_model is not None and rec["feat"]:
        try:
            pred = ltr_model.predict(np.asarray(rec["feat"], dtype=np.float64))
            return int(np.argmax(pred))
        except Exception:
            return 0
    return 0


rows_out, summary_counts = [], {}
m_chosen_all, m_oracle_all, ce_gap_01 = [], [], []
rank_of_best, n_ok_rank1 = {}, 0
by_unit = {}

for qid, rec in harvest_records.items():
    gold = str(train_data[qid].get("answer", ""))
    if not rec["cands"]:
        rows_out.append({"qid": qid, "question": rec["question"], "error_type": "no_candidate",
                          "meteor": 0.0, "oracle_best_meteor": 0.0, "oracle_gap": 0.0})
        summary_counts["no_candidate"] = summary_counts.get("no_candidate", 0) + 1
        continue
    ms = [c["meteor"] for c in rec["cands"]]
    ci = _chosen_index(rec)
    oi = int(np.argmax(ms))
    chosen_m, oracle_m = ms[ci], ms[oi]
    etype = classify_error(chosen_m, oracle_m)

    summary_counts[etype] = summary_counts.get(etype, 0) + 1
    m_chosen_all.append(chosen_m)
    m_oracle_all.append(oracle_m)
    rank_of_best[oi] = rank_of_best.get(oi, 0) + 1
    if ci == oi:
        n_ok_rank1 += 1
    if len(rec["cands"]) > 1 and rec["cands"][0]["ce_score"] is not None \
            and rec["cands"][1]["ce_score"] is not None:
        ce_gap_01.append(rec["cands"][0]["ce_score"] - rec["cands"][1]["ce_score"])
    u = rec["cands"][ci]["unit_type"]
    by_unit.setdefault(u, []).append(chosen_m)

    rows_out.append({
        "qid": qid,
        "question": rec["question"],
        "error_type": etype,
        "meteor": round(chosen_m, 4),
        "oracle_best_meteor": round(oracle_m, 4),
        "oracle_gap": round(oracle_m - chosen_m, 4),
        "chosen_rank": ci,
        "oracle_rank": oi,
        "n_candidates": rec["n_cands"],
        "gold_len_words": len(gold.split()),
        # Đủ để tra ngược một câu cụ thể mà không phải chạy lại GPU: id Điều tra được trong
        # corpus, điểm CE cho biết reranker "tự tin" tới đâu, METEOR cho biết lẽ ra được bao
        # nhiêu nếu chọn ứng viên đó.
        "candidates": [{
            "rank": i, "id": c["id"], "dieu_so": c["dieu_so"], "so_hieu": c["so_hieu"],
            "unit_type": c["unit_type"], "n_words": c["n_words"],
            "ce_score": (round(c["ce_score"], 4) if c["ce_score"] is not None else None),
            "meteor_if_chosen": round(c["meteor"], 4),
        } for i, c in enumerate(rec["cands"])],
    })

n_tot = max(len(m_chosen_all), 1)
summary = {
    "meta": {
        "ts": time.strftime("%Y-%m-%d %H:%M:%S"),
        "n_harvested": len(harvest_records),
        "n_scored": len(m_chosen_all),
        "harvest_rate_s_per_q": round(harvest_rate_s, 2),
        "config": {"top_n": top_n_answer, "agg_mode": AGG_MODE, "use_ltr": bool(use_ltr),
                    "reranker": RERANKER_SOURCE, "top_k_candidates": LTR_TOP_K_CANDIDATES,
                    "answer_template": ANSWER_TEMPLATE,
                    "channels": V62_DECISIONS["active"]["channels"]},
        "thresholds": {"ok": ERR_OK_METEOR, "ranking_gap": ERR_RANKING_GAP,
                        "retrieval_ceil": ERR_RETRIEVAL_CEIL},
        "sample_note": (
            "Mẫu gồm câu dev (đã loại khỏi fine-tune) + câu KHÔNG có nhãn citation (chưa bao "
            "giờ vào fine-tune). Không câu nào bị contamination bởi fine-tune encoder. Điểm "
            "vẫn cao hơn public vì template render_answer khớp văn phong train.json "
            "(result.md §5) — đó là lệch phân phối, không phải rò rỉ."),
    },
    "scores": {
        "meteor_mean": round(sum(m_chosen_all) / n_tot, 4),
        "oracle_meteor_mean": round(sum(m_oracle_all) / n_tot, 4),
        "gap_to_oracle": round((sum(m_oracle_all) - sum(m_chosen_all)) / n_tot, 4),
        "pick_best_rate": round(n_ok_rank1 / n_tot, 4),
        "ce_gap_rank0_rank1_median": (round(float(np.median(ce_gap_01)), 4) if ce_gap_01 else None),
    },
    "error_distribution": {k: {"count": v, "pct": round(100 * v / max(len(rows_out), 1), 1)}
                            for k, v in sorted(summary_counts.items(), key=lambda kv: -kv[1])},
    "where_best_candidate_sits": {str(k): v for k, v in sorted(rank_of_best.items())},
    "meteor_by_unit_type": {k: {"n": len(v), "meteor": round(sum(v) / len(v), 4)}
                             for k, v in sorted(by_unit.items(), key=lambda kv: -len(kv[1]))},
    "ltr": ltr_report,
}

full_path = os.path.join(OUT_DIR, "eval_harvest_full.json")
sum_path = os.path.join(OUT_DIR, "eval_harvest_summary.json")
with open(full_path, "w", encoding="utf-8") as f:
    json.dump(rows_out, f, ensure_ascii=False, indent=1)
with open(sum_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"\n  --- TỔNG HỢP trên {len(rows_out)} câu ---")
print(f"    METEOR thực tế   {summary['scores']['meteor_mean']:.4f}")
print(f"    METEOR oracle    {summary['scores']['oracle_meteor_mean']:.4f}  "
      f"(dư địa còn {summary['scores']['gap_to_oracle']:+.4f})")
print(f"    chọn đúng ứng viên tốt nhất: {100*summary['scores']['pick_best_rate']:.1f}%")
if summary["scores"]["ce_gap_rank0_rank1_median"] is not None:
    print(f"    khoảng cách điểm CE hạng 1 vs hạng 2 (trung vị): "
          f"{summary['scores']['ce_gap_rank0_rank1_median']:.4f}")
print("    phân bố lỗi:")
for k, v in summary["error_distribution"].items():
    print(f"      {k:<16s} {v['count']:>5d}  ({v['pct']:>4.1f}%)")
print(f"\n  Đã ghi:\n    {full_path}  (chi tiết từng câu — dùng để soi ca lỗi cụ thể)"
      f"\n    {sum_path}  (tổng hợp — dùng để quyết định hướng đi)")
checkpoint("Xong phân tích lỗi")


In [ ]:
# Cell 16 [v6_1]: Sổ thí nghiệm — 1 dòng JSON/lượt, APPEND (không ghi đè)
#
# BẢN SỬA v6_1 — bản v6 crash Ở ĐÂY. Cell log của v6 đọc `USE_TASK1_LABELS` và
# `n_task1_added`, nhưng v6 đã xoá sạch code Task 1 khỏi Cell 7 (đúng rule.md §7c(b)) mà
# quên cập nhật cell này -> NameError ở cell CUỐI CÙNG, sau khi đã tiêu gần hết phiên GPU.
# Bài học lặp lại đúng lỗi §9 của result.md: xoá một hướng thì phải xoá cả những chỗ ĐỌC nó.
# v6_1 chỉ ghi những biến thực sự tồn tại trong chính notebook này.
n_empty = sum(1 for a in answers.values() if not a.strip())
record = {
    "ts": time.strftime("%Y-%m-%d %H:%M:%S"),
    "version": "v6_2",
    "seed": SEED, "use_warmup": USE_WARMUP, "n_warmup_used": n_warmup_used,
    "hardware": (f"kaggle_t4x{N_GPU}" if IS_KAGGLE else f"local_{N_GPU}gpu"),
    "concl": CONCL,
    # --- cấu hình đã chạy thật ---
    "dense_model_a": BASE_DENSE_MODEL_A, "dense_model_b": BASE_DENSE_MODEL_B,
    "use_new_encoders_requested": USE_NEW_ENCODERS,
    "encoder_fallback_fired": (USE_NEW_ENCODERS and BASE_DENSE_MODEL_A == "BAAI/bge-m3"),
    "agg_mode": AGG_MODE,
    "reranker_source": RERANKER_SOURCE,
    "reranker_finetuned": reranker_finetune_info["used"],
    "reranker_finetune_reason": reranker_finetune_info["reason"],
    "doc_k": DOC_K, "max_dieu_candidates": MAX_DIEU_CANDIDATES,
    "top_n_answer": top_n_answer, "use_reranker": use_reranker,
    # --- fine-tune encoder ---
    "n_dev_excluded_from_train": n_dev_had_label,
    "n_train_pairs_available": finetune_info["n_pairs_available"],
    "n_train_pairs_used": finetune_info["n_pairs_used"],
    "used_finetune": finetune_info["used_finetune"], "finetune_reason": finetune_info["reason"],
    "finetune_models": finetune_info["models"],
    # --- v6_1: harvest + LTR ---
    "n_harvest_total": len(harvest_records),
    "n_harvest_phase_a": eval_info["n_harvest_phase_a"],
    "harvest_rate_s_per_q": eval_info["harvest_rate_s"],
    "ltr_trained": ltr_report["trained"], "ltr_accepted": ltr_report["accepted"],
    "ltr_delta_half_a": ltr_report["delta_half_a"], "ltr_delta_half_b": ltr_report["delta_half_b"],
    "ltr_n_groups": ltr_report["n_groups"], "use_ltr": bool(use_ltr),
    "harvest_meteor": summary["scores"]["meteor_mean"],
    "harvest_oracle_meteor": summary["scores"]["oracle_meteor_mean"],
    "harvest_pick_best_rate": summary["scores"]["pick_best_rate"],
    "error_distribution": {k: v["pct"] for k, v in summary["error_distribution"].items()},
    # --- dev-eval ---
    "dev_meteor": eval_info["meteor"], "dev_rouge_l": eval_info["rouge_l"],
    "dev_n": eval_info["n_dev"], "dev_recall_at_k": eval_info["recall_at_k"],
    "n_empty_answers": n_empty, "elapsed_min": round(elapsed() / 60, 1),
    # --- v6_2: ba cổng + fp16 + reranker-ft ---
    "answer_template": ANSWER_TEMPLATE,
    "v62_decisions": V62_DECISIONS,
    "encode_info": encode_info,
    "reranker_ft_meta": reranker_finetune_info.get("meta"),
    "reranker_ft_mine_s": reranker_finetune_info.get("mine_elapsed_s"),
    # Mốc thời gian + VRAM/RAM đỉnh tại mỗi checkpoint. Đây là thứ lượt trước KHÔNG có, nên
    # không truy được vì sao encoder B tụt mini-batch 32->4 (99 phút cho 149 step).
    "resource_log": _RESOURCE_LOG,
}
try:
    with open(EXPERIMENT_LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"  Đã ghi thêm 1 dòng vào {EXPERIMENT_LOG_PATH}")
except OSError as e:
    print(f"  [CẢNH BÁO] Không ghi được sổ thí nghiệm ({e}) — không ảnh hưởng submission.zip.")

# LUÔN in ra console: /kaggle/working có thể mất nếu không commit đúng cách, nhưng output của
# cell được giữ trong tab Logs của version khi dùng "Save & Run All (Commit)".
print("  [SỔ THÍ NGHIỆM — copy dòng dưới đây nếu cần đối chiếu sau này]")
print("  " + json.dumps(record, ensure_ascii=False))

print(f"\n{'='*78}\nXONG v6_2 — {elapsed()/60:.1f} phút")
print(f"  submission.zip        bài nộp ({len(answers)} câu)")
print(f"  eval_harvest_full.json     chi tiết {len(rows_out)} câu — soi ca lỗi")
print(f"  eval_harvest_summary.json  tổng hợp — quyết định hướng đi")
print(f"  ltr_report.json            bộ chọn học được: nhận hay bị cổng split-half loại")
print(f"  v6_2_decisions.json        số đo ba cổng (reranker / câu dẫn / kênh encoder)")
print(f"  v6_2_dev_arms.jsonl        từng câu dev × từng nhánh — phân tích ngược")
print(f"  experiment_log.jsonl       sổ thí nghiệm")
print('='*78)
